# S01 · Fundamentos, adquisición y procesamiento por fragmentos
**Aula alternativa · 150 minutos propuestos.** No sustituye el curso oficial.
Abre [la presentación de revisión](https://raw.githack.com/jazaineam1/BigData2026/mejora-pedagogica-integral/revision/s01.html). No necesitas internet para la ruta principal una vez abierto este cuaderno: incluye una copia del corpus de 200 procesos del repositorio, con su procedencia. No requiere secretos, instalar motores ni descargar archivos adicionales. Los datos son registros SECOP del snapshot, no una muestra representativa del universo.

**Objetivos observables.** Distinguir unidad de observación y formato; diagnosticar restricciones de volumen y velocidad; comprobar calidad e identidad; adquirir por páginas de forma reproducible; comparar agregaciones completas y por fragmentos; producir y releer un paquete con RAW, derivados y manifiesto. La consulta viva es opcional y se etiqueta separadamente.

**Cómo trabajar:** predice antes de ejecutar, observa las salidas, cambia un parámetro por vez y explica el límite de la evidencia. Las celdas del núcleo están en el orden de ejecución; el apéndice retoma A0–A8 por tema. Ejecutar todas con configuración inicial produce un paquete local sin acceso externo; las variantes del reto son formativas. El resultado no certifica aprendizaje hasta comprobar tu propia modificación.

### Ruta de aula: de la fuente al producto
Ejecuta en orden las celdas hasta el ticket de salida. Los capítulos extensos, el servidor HTTP real y la matriz de pruebas están después, en el apéndice de profundización. La ruta principal funciona sin red y sin abrir un puerto. Las celdas anteriores al ticket de salida forman el núcleo; la última actividad autónoma se activa después de escribir la hipótesis. Para QA del autor puede ejecutarse todo: los experimentos del apéndice están en orden de dependencias. Si Colab no abre, trabaja el concepto en la presentación, pero la práctica Python queda pendiente. No se acepta una captura de pantalla como sustituto del paquete.

En Colab, ejecuta las celdas anteriores al ticket de salida (o una por una hasta ese punto). A6.2 ejecuta 100.000 filas y deja un millón como opción manual. A8.2–A8.4 distinguen construcción, comparación y entrega. No es necesario usar Ejecutar todo para la clase: esa opción también recorre el apéndice y la demostración HTTP local. El núcleo no depende de dichas celdas.

La presentación incluye una portada con competencias, ilustraciones, modos de proyección/exploración/lectura y libreto de apoyo docente. Usa «Ruta de aula» para localizar el identificador de cada celda. A6.3 en el apéndice contrasta representación y complejidad, sin aumentar la carga obligatoria.


In [ ]:
import sys, json, csv, hashlib, math, tempfile, time, re, unicodedata
from collections import Counter
from pathlib import Path
import pandas as pd
from IPython.display import display
print("Python",sys.version.split()[0],"pandas",pd.__version__)
def sha(b): return hashlib.sha256(b).hexdigest()
def canon(x): return json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(",",":"),allow_nan=False).encode("utf-8")

# Función de huella disponible sin ejecutar el servidor HTTP.
def sha_archivo(ruta):
    h=hashlib.sha256()
    with Path(ruta).open("rb") as f:
        for bloque in iter(lambda:f.read(65536),b""):
            h.update(bloque)
    return h.hexdigest()


## Mapa de trabajo
| Estación | Qué ejecutas | Qué debes demostrar |
|---|---|---|
| 1 · Preparación y fuente | A0, A3 | 200 IDs, esquema, conteos y selección |
| 2 · Adquisición | A4 básica, A5 | El salto incorrecto pierde identidades; el correcto conserva el universo |
| 3 · Reducción | A6 | Conteos iguales con tamaños de bloque distintos, memoria lógica y tiempo |
| 4 · Producto | A7 | RAW, conteos y manifiesto verificados por una función independiente |
| 5 · Transferencia | A8 | Predicciones propias, dos funciones construidas, cuatro escenarios aislados y defensa |

Las ideas de escala, formatos, calidad y texto se exploran primero en la presentación. Sus extensiones Python están al final. Cada estación produce una salida concreta: no avances si falla un control. El instructor puede demostrar HTTP real en el apéndice después de paginación; no es una dependencia del producto principal.

### Salidas de referencia para identificar si tu ejecución es coherente
Estas cifras se calcularon sobre los 200 objetos exactos del snapshot incorporado mediante JavaScript, y la suma sintética se obtuvo por fórmula y ejecución Python de referencia. No son salidas guardadas de una ejecución integral del cuaderno. Los tiempos dependen del ambiente.

| Control | Qué deberías observar |
|---|---|
| A3 universo | 200 procesos únicos y 51 nombres de entidad |
| A3 prefijos | 40 procesos: 26 nombres; 80: 39; 120: 46 |
| A3 descripción, término salud | 6 ocurrencias en 4 procesos con la regla de tokens definida |
| A5 salto 40 | 200 filas recibidas, 170 IDs, 30 repeticiones y 30 ausentes |
| A5 límite 37 correcto | 6 páginas: 37, 37, 37, 37, 37 y 15; 200 IDs únicos |
| A6 referencia de 10.000 filas | suma 499.950, igual con los tres tamaños de lectura |
| A7 paquete completo | 200 filas, 51 nombres de entidad, 6 páginas y 8 bloques de 25 |
| A8 | depende de tus parámetros; no hay una cifra única para todos |

Si tu resultado difiere, revisa primero el universo, los parámetros, el estado del kernel y el nombre de la columna. No cambies las aserciones para llegar a una cifra esperada. En A6 la memoria lógica puede variar con la versión de pandas y el tipo inferido; la suma y el número de filas sí deben coincidir. En A8 usa las predicciones propias y explica las diferencias si alguna falla.


## A3 · Fuente real, calidad, pandas y texto
El snapshot se incorpora directamente para que el laboratorio funcione sin red. Fuente original indicada por el repositorio: Datos/bandeja_revision_2026.json; versión del archivo tc2_corpus_2026.json: blob 4b1ac0dd22137984b7e81864a70e04075af2af39. Los registros contienen ID de proceso, nombre, descripción, entidad, enlace, fuente y tipo. La copia preserva los objetos del snapshot; no es una descarga de SECOP en vivo. La celda de datos es extensa: se puede plegar; no es contenido explicativo.

Después de cargar, comprobamos esquema, unicidad y faltantes. La tabla agrupada cambia grano de proceso a entidad; size cuenta filas. Si se modifica una copia para introducir duplicados o faltantes, RAW no debe cambiar. La descripción se tokeniza por letras y números normalizados: un término puede aparecer varias veces en un proceso, por lo que ocurrencias y frecuencia documental son diferentes.

**Cómo se lee:** total de grupos, suma de conteos y primeras filas. **Qué nos dice:** cuántos procesos de esta fuente corresponden a cada entidad. **Qué no permite concluir:** concentración del gasto ni representatividad nacional. **Error común:** eliminar registros por tener textos iguales aunque sus IDs sean diferentes.


In [ ]:
# Copia de los 200 registros SECOP del snapshot del repositorio. No editar.
CORPUS = json.loads("[{\"id_proceso\":\"CO1.REQ.10017413\",\"nombre_proceso\":\"CONTRATO INTERADMINITRATIVO PARA EL FORTALECIMIENTO DEL NIVEL PRIMARIO; BASADO EN LA ATENCIÓN PRIMARIA EN SALUD A TRAVÉS DE LA CONFORMACIÓN \",\"descripcion\":\"CONTRATO INTERADMINITRATIVO PARA EL FORTALECIMIENTO DEL NIVEL PRIMARIO; BASADO EN LA ATENCIÓN PRIMARIA EN SALUD A TRAVÉS DE LA CONFORMACIÓN Y OPERACIÓN DE EQUIPOS BÁSICOS DE SALUD \",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9868453\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.10021334\",\"nombre_proceso\":\"CONTRATO DE PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA FORTALECER LOS PROCESOS COMUNICACIONALES DE LA SECRETARÍA DE COMUNICACIONES; \",\"descripcion\":\"CONTRATO DE PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA FORTALECER LOS PROCESOS COMUNICACIONALES DE LA SECRETARÍA DE COMUNICACIONES; MEDIANTE EL DISEÑO; PRODUCCIÓN Y DIFUSIÓ\",\"entidad\":\"MUNICIPIO DE ITAGUI\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9877717\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.10027889\",\"nombre_proceso\":\"CI-0709-2026\",\"descripcion\":\"CI-0709-2026\",\"entidad\":\"GOBERNACION DEL MAGDALENA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9879068\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.10048613\",\"nombre_proceso\":\"PRESTACIÓN DEL SERVICIO PARA ADMINISTRAR LA CANASTA PARCIAL DE LA POBLACIÓN ÉTNICA DE LA INSTITUCIÓN ETNOEDUCATIVA INTEGRAL RURAL KUISA; LA \",\"descripcion\":\"PRESTACIÓN DEL SERVICIO PARA ADMINISTRAR LA CANASTA PARCIAL DE LA POBLACIÓN ÉTNICA DE LA INSTITUCIÓN ETNOEDUCATIVA INTEGRAL RURAL KUISA; LA INSTITUCION ETNOEDUCATIVA INTEGRAL RURAL\",\"entidad\":\"MUNICIPIO DE URIBIA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9900290\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.10056281\",\"nombre_proceso\":\"PRESTACIÓN DEL SERVICIO PARA ADMINISTRAR LA CANASTA PARCIAL DE LA POBLACIÓN ÉTNICA DE LA INSTITUCION ETNOEDUCATIVA INTEGRAL RURAL JACOBO ARE\",\"descripcion\":\"PRESTACIÓN DEL SERVICIO PARA ADMINISTRAR LA CANASTA PARCIAL DE LA POBLACIÓN ÉTNICA DE LA INSTITUCION ETNOEDUCATIVA INTEGRAL RURAL JACOBO ARENDS Y LA INSTITUCION ETNOEDUCATIVA INTEG\",\"entidad\":\"MUNICIPIO DE URIBIA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9908043\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.10083057\",\"nombre_proceso\":\"CONVENIO SOLIDARIO\",\"descripcion\":\"CONVENIO SOLIDARIO\",\"entidad\":\"GOBERNACION DE NARIÑO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9937124\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.10246305\",\"nombre_proceso\":\"Prestar los servicios tecnológicos necesarios para la puesta en operación del software utilizado para la comisión de escrutinio nacional; as\",\"descripcion\":\"Prestar los servicios tecnológicos necesarios para la puesta en operación del software utilizado para la comisión de escrutinio nacional; asegurando los componentes requeridos; Inf\",\"entidad\":\"Consejo Nacional Electoral\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.10104026\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1109601\",\"nombre_proceso\":\"SECRETARIA DISTRITAL DE MOVILIDAD\",\"descripcion\":\"SECRETARIA DISTRITAL DE MOVILIDAD\",\"entidad\":\"SECRETARIA DE EDUCACION DEL DISTRITO\",\"url_secop\":\"https://community.secop.gov.co/STS/Users/Login/Index\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1144334\",\"nombre_proceso\":\"CONTRATO TEVEANDINA- PRENSA\",\"descripcion\":\"CONTRATO TEVEANDINA- PRENSA\",\"entidad\":\"FONDO UNICO DE TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1106024\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1227580\",\"nombre_proceso\":\"CONTRATO IVA\",\"descripcion\":\"CONTRATO IVA\",\"entidad\":\"PROSPERIDAD SOCIAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1188491\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1229269\",\"nombre_proceso\":\"IMPRENTA NACIONAL DE COLOMBIA\",\"descripcion\":\"IMPRENTA NACIONAL DE COLOMBIA\",\"entidad\":\"Ministerio de las Culturas, las Artes y los Saberes\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1189868\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1235779\",\"nombre_proceso\":\"CONTRATO DE ARRENDAMIENTO\",\"descripcion\":\"CONTRATO DE ARRENDAMIENTO\",\"entidad\":\"(Secretaría Distrital de Integración Social)\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1206076\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1253135\",\"nombre_proceso\":\"PONER A DISPOSICIÓN DE SUSCRIPTORES LÍNEAS MÓVILES PREPAGO SERVICIOS TIC - EMERGENCIA SANITARIA; ECONÓMICA Y ECOLÓGICA COVID-19 - COMUNICACI\",\"descripcion\":\"PONER A DISPOSICIÓN DE SUSCRIPTORES LÍNEAS MÓVILES PREPAGO SERVICIOS TIC - EMERGENCIA SANITARIA; ECONÓMICA Y ECOLÓGICA COVID-19 - COMUNICACIÓN CELULAR S.A. COMCEL S.A.\",\"entidad\":\"FONDO UNICO DE TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1213079\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1257693\",\"nombre_proceso\":\"SUMINISTRO E INSTALACIÓN DE EQUIPAMENTO BIOMEDICO PARA EL HOSPITAL DEPARTAMENTAL DE VILLAVICENCIO E.S.E. COVID 19\",\"descripcion\":\"SUMINISTRO E INSTALACIÓN DE EQUIPAMENTO BIOMEDICO PARA EL HOSPITAL DEPARTAMENTAL DE VILLAVICENCIO E.S.E. COVID 19\",\"entidad\":\"DEPARTAMENTO DEL META\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1218081\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1309251\",\"nombre_proceso\":\"CONTRATACIÓN DIRECTA - URGENCIA MANIFIESTA\",\"descripcion\":\"CONTRATACIÓN DIRECTA - URGENCIA MANIFIESTA\",\"entidad\":\"INSTITUTO DE DESARROLLO URBANO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1268026\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1423544\",\"nombre_proceso\":\"FORTALECIMIENTO A LA PRODUCTIVIDAD DEL SECTOR AGROPECUARIO EN SITUACIÓN DE  EMERGENCIA SANITARIA DEL DEPARTAMENTO DE CORDOBA; IDENTIFICADO C\",\"descripcion\":\"FORTALECIMIENTO A LA PRODUCTIVIDAD DEL SECTOR AGROPECUARIO EN SITUACIÓN DE  EMERGENCIA SANITARIA DEL DEPARTAMENTO DE CORDOBA; IDENTIFICADO CON EL CÓDIGO BPIN  2020000020008\",\"entidad\":\"DEPARTAMENTO DE CORDOBA\",\"url_secop\":\"https://community.secop.gov.co/STS/Users/Login/Index\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1561617\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO CONECTIVIDAD\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO CONECTIVIDAD\",\"entidad\":\"GOBERNACION DE SUCRE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1513764\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1591012\",\"nombre_proceso\":\"PROVEER SOLUCIONES DE VIVIENDA EN EL MUNICIPIO DE MARIA LA BAJA; DEPARTAMENTO DE BOLÍVAR - CONSORCIO BOLÍVAR\",\"descripcion\":\"PROVEER SOLUCIONES DE VIVIENDA EN EL MUNICIPIO DE MARIA LA BAJA; DEPARTAMENTO DE BOLÍVAR - CONSORCIO BOLÍVAR\",\"entidad\":\"FONDO ADAPTACIÓN\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1540718\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1676905\",\"nombre_proceso\":\"Aunar esfuerzos técnicos; administrativos y financieros entre el Ministerio del Deporte y el Municipio de Natagaima; para la ejecución del p\",\"descripcion\":\"Aunar esfuerzos técnicos; administrativos y financieros entre el Ministerio del Deporte y el Municipio de Natagaima; para la ejecución del proyecto denominado MEJORAMIENTO Y ADECU\",\"entidad\":\"MINISTERIO DEL DEPORTE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1625113\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1681646\",\"nombre_proceso\":\"CONTRATO DE ORDEN DE SERVICIO\",\"descripcion\":\"CONTRATO DE ORDEN DE SERVICIO\",\"entidad\":\"GOBERNACIÓN DE LA GUAJIRA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1629766\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1766696\",\"nombre_proceso\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTIÓN\",\"descripcion\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTIÓN\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1713178\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1791463\",\"nombre_proceso\":\"PRESTACION DE SERVICIOS - BANCO DE LA REPUBLICA\",\"descripcion\":\"PRESTACION DE SERVICIOS - BANCO DE LA REPUBLICA\",\"entidad\":\"MINISTERIO DE HACIENDA Y CREDITO PUBLICO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1738021\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1828218\",\"nombre_proceso\":\"PRESTACIÓN DEL SERVICIO PARA ADMINISTRAR LA CANASTA PARCIAL DE LA POBLACIÓN ÉTNICA\",\"descripcion\":\"PRESTACIÓN DEL SERVICIO PARA ADMINISTRAR LA CANASTA PARCIAL DE LA POBLACIÓN ÉTNICA\",\"entidad\":\"MINISTERIO DE EDUCACION NACIONAL (MEN)\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1779662\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1842719\",\"nombre_proceso\":\"LA SECRETARÍA DISTRITAL DE MOVILIDAD Y LA POLICÍA NACIONAL; AÚNAN ESFUERZOS PARA COORDINAR Y COOPERAR MUTUAMENTE PARA EJERCER EL CONTROL Y R\",\"descripcion\":\"LA SECRETARÍA DISTRITAL DE MOVILIDAD Y LA POLICÍA NACIONAL; AÚNAN ESFUERZOS PARA COORDINAR Y COOPERAR MUTUAMENTE PARA EJERCER EL CONTROL Y REGULACIÓN DEL TRÁNSITO Y EL TRANSPORTE E\",\"entidad\":\"SECRETARÍA DISTRITAL DE MOVILIDAD\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1801312\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1859670\",\"nombre_proceso\":\"FAMOC DEPANEL S.A.\",\"descripcion\":\"FAMOC DEPANEL S.A.\",\"entidad\":\"SECRETARIA DISTRITAL DE SEGURIDAD, CONVIVENCIA Y JUSTICIA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1808987\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1862814\",\"nombre_proceso\":\"OICH\",\"descripcion\":\"OICH\",\"entidad\":\"GOBERNACION DEL CHOCÓ.\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1807608\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1903338\",\"nombre_proceso\":\"Convenio interadministrativo\",\"descripcion\":\"Convenio interadministrativo\",\"entidad\":\"GOBERNACION DE NARIÑO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1847239\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1928260\",\"nombre_proceso\":\"Caja Colombiana de Subsidio Familiar  COLSUBSIDIO\",\"descripcion\":\"Caja Colombiana de Subsidio Familiar  COLSUBSIDIO\",\"entidad\":\"SECRETARIA DE EDUCACION DEL DISTRITO\",\"url_secop\":\"https://community.secop.gov.co/STS/Users/Login/Index\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1941822\",\"nombre_proceso\":\"AUNAR ESFUERZOS PARA ASEGURAR LA PRESTACIÓN DEL SERVICIO PÚBLICO ESENCIAL DEATENCIÓN Y CONTROL DE INCENDIOS Y DEMÁS ACTIVIDADES CONEXAS\",\"descripcion\":\"AUNAR ESFUERZOS PARA ASEGURAR LA PRESTACIÓN DEL SERVICIO PÚBLICO ESENCIAL DEATENCIÓN Y CONTROL DE INCENDIOS Y DEMÁS ACTIVIDADES CONEXAS\",\"entidad\":\"ALCALDÍA DE POPAYÁN\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1885416\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1942046\",\"nombre_proceso\":\"CONTRATO INTERADMINISTRATIVO DE ADMINISTRACIÓN DELEGADA ENTRE EL MUNICIPIO DE ITAGÜÍ Y LA AGENCIA DE DESARROLLO LOCAL DE ITAGÜÍ ADELI PARA \",\"descripcion\":\"CONTRATO INTERADMINISTRATIVO DE ADMINISTRACIÓN DELEGADA ENTRE EL MUNICIPIO DE ITAGÜÍ Y LA AGENCIA DE DESARROLLO LOCAL DE ITAGÜÍ ADELI PARA LA ATENCIÓN PREVENTIVA Y CORRECTIVA Y AD\",\"entidad\":\"MUNICIPIO DE ITAGUI\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1885828\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1949451\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA LA ATENCIÓN EN EDUCACIÓN INICIAL A LA PRIMERA INFANCIA EN EL MARCO DE LA POLÍTICA DE EST\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA LA ATENCIÓN EN EDUCACIÓN INICIAL A LA PRIMERA INFANCIA EN EL MARCO DE LA POLÍTICA DE ESTADO PARA EL DESARROLLO INTEGRAL DE LA PR\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1893361\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1949570\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA LA ATENCIÓN EN EDUCACIÓN INICIAL A LA PRIMERA INFANCIA EN EL MARCO DE LA POLÍTICA DE EST\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA LA ATENCIÓN EN EDUCACIÓN INICIAL A LA PRIMERA INFANCIA EN EL MARCO DE LA POLÍTICA DE ESTADO PARA EL DESARROLLO INTEGRAL DE LA PR\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1893802\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1958141\",\"nombre_proceso\":\"APOYO A LA GESTIÓN PRIMERA INFANCIA\",\"descripcion\":\"APOYO A LA GESTIÓN PRIMERA INFANCIA\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1901948\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.1958230\",\"nombre_proceso\":\"REALIZAR LA GERENCIA INTEGRAL Y COORDINACION PARA REALIZAR EL MEJORAMIENTO DE LOS SISTEMAS DE SONIDO E ILUMINACIÓN DEL ESTADIO METROPOLITANO\",\"descripcion\":\"REALIZAR LA GERENCIA INTEGRAL Y COORDINACION PARA REALIZAR EL MEJORAMIENTO DE LOS SISTEMAS DE SONIDO E ILUMINACIÓN DEL ESTADIO METROPOLITANO ROBERTO MELÉNDEZ BARRANQUILLA\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1902364\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2052103\",\"nombre_proceso\":\"CONVENIO CONV-SED-016-2021\",\"descripcion\":\"CONVENIO CONV-SED-016-2021\",\"entidad\":\"GOBERNACION DE SUCRE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.1998706\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2094652\",\"nombre_proceso\":\"Universidad Distrital Francisco José de Caldas\",\"descripcion\":\"Universidad Distrital Francisco José de Caldas\",\"entidad\":\"SECRETARIA DE EDUCACION DEL DISTRITO\",\"url_secop\":\"https://community.secop.gov.co/STS/Users/Login/Index\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2136149\",\"nombre_proceso\":\"AUNAR ESFUERZOS TÉCNICOS; ADMINISTRATIVOS Y FINANCIEROS ENTRE EL MINISTERIO DEL DEPORTE Y EL MUNICIPIO DE VALENCIA; CÓRDOBA\",\"descripcion\":\"AUNAR ESFUERZOS TÉCNICOS; ADMINISTRATIVOS Y FINANCIEROS ENTRE EL MINISTERIO DEL DEPORTE Y EL MUNICIPIO DE VALENCIA; CÓRDOBA\",\"entidad\":\"MINISTERIO DEL DEPORTE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2078925\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2197504\",\"nombre_proceso\":\"CONTRATO INTERADMINISTRATIVO PARA LA INTERVENTORÍA DE OBRAS DE INFRAESTRUCTURA DEL MUNICIPIO DE RIONEGRO.\",\"descripcion\":\"CONTRATO INTERADMINISTRATIVO PARA LA INTERVENTORÍA DE OBRAS DE INFRAESTRUCTURA DEL MUNICIPIO DE RIONEGRO.\",\"entidad\":\"MUNICIPIO DE RIONEGRO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2138606\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2256136\",\"nombre_proceso\":\"CONVENIO ESPECÍFICO DE COOPERACIÓN\",\"descripcion\":\"CONVENIO ESPECÍFICO DE COOPERACIÓN\",\"entidad\":\"ALCALDIA DE PASTO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2210538\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2267818\",\"nombre_proceso\":\"CI-014-2021\",\"descripcion\":\"CI-014-2021\",\"entidad\":\"GOBERNACION DEL MAGDALENA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2216573\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2273783\",\"nombre_proceso\":\"CONTRATACION DIRECTA FIDUCIARIA CORFICOLOMBIANA SA\",\"descripcion\":\"CONTRATACION DIRECTA FIDUCIARIA CORFICOLOMBIANA SA\",\"entidad\":\"TRANSMILENIO S.A.\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2231919\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2282243\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO\",\"entidad\":\"Ministerio de las Culturas, las Artes y los Saberes\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2223381\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2306622\",\"nombre_proceso\":\"CONVENIO BECAS POLICIA NACIONAL\",\"descripcion\":\"CONVENIO BECAS POLICIA NACIONAL\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2247066\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2422511\",\"nombre_proceso\":\"ADQUISICIÓN; OPTIMIZACIÓN Y ACTUALIZACIÓN DEL LICENCIAMIENTO PARA LA PLATAFORMA DE RECEPCIÓN Y ANÁLISIS DE COMUNICACIONES INTERCEPTADAS CON \",\"descripcion\":\"ADQUISICIÓN; OPTIMIZACIÓN Y ACTUALIZACIÓN DEL LICENCIAMIENTO PARA LA PLATAFORMA DE RECEPCIÓN Y ANÁLISIS DE COMUNICACIONES INTERCEPTADAS CON EL PROTOCOLO DE COMUNICACIONES ETSI - II\",\"entidad\":\"DIRECCIÓN DE INVESTIGACIÓN CRIMINAL E INTERPOL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2362029\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2427611\",\"nombre_proceso\":\"AUNAR ESFUERZOS TÉCNICOS; ADMINISTRATIVOS Y FINANCIEROS ENTRE EL MINISTERIO DEL DEPORTE Y EL MUNICIPIO DE PUERTO LOPEZ PARA LA EJECUCIÓN DEL\",\"descripcion\":\"AUNAR ESFUERZOS TÉCNICOS; ADMINISTRATIVOS Y FINANCIEROS ENTRE EL MINISTERIO DEL DEPORTE Y EL MUNICIPIO DE PUERTO LOPEZ PARA LA EJECUCIÓN DEL PROYECTO DENOMINADO CONSTRUCCIÓN DE PLA\",\"entidad\":\"MINISTERIO DEL DEPORTE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2367035\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2435456\",\"nombre_proceso\":\"Aunar esfuerzos técnicos; administrativos y financieros para ejecutar el plan integral de ocupantes de espacio público (OEP) de la Primera L\",\"descripcion\":\"Aunar esfuerzos técnicos; administrativos y financieros para ejecutar el plan integral de ocupantes de espacio público (OEP) de la Primera Línea Del Metro De Bogotá Tramo 1; en su \",\"entidad\":\"METRO DE BOGOTA SA\",\"url_secop\":\"https://community.secop.gov.co/STS/Users/Login/Index\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2435627\",\"nombre_proceso\":\"construcción de escenario polideportivo cancha sintetica en pie de pato cabecera municial del municipio de Alto Baudó Departamento del Choco\",\"descripcion\":\"construcción de escenario polideportivo cancha sintetica en pie de pato cabecera municial del municipio de Alto Baudó Departamento del Choco\",\"entidad\":\"MINISTERIO DEL DEPORTE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2378302\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2437231\",\"nombre_proceso\":\"SECRETARIA DISTRITAL DE INTEGRACION SOCIAL\",\"descripcion\":\"SECRETARIA DISTRITAL DE INTEGRACION SOCIAL\",\"entidad\":\"SECRETARIA DE EDUCACION DEL DISTRITO\",\"url_secop\":\"https://community.secop.gov.co/STS/Users/Login/Index\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2442640\",\"nombre_proceso\":\"CD-660-2021\",\"descripcion\":\"CD-660-2021\",\"entidad\":\"MINISTERIO DE AGRICULTURA Y DESARROLLO RURAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2380558\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2443494\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO\",\"entidad\":\"GOBERNACIÓN DE BOYACÁ\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2384919\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2443796\",\"nombre_proceso\":\"AUNAR ESFUERZOS ENTRE EL MINISTERIO DEL DEPORTE Y EL MUNICIPIO DE CLEMENCIA; BOLÍVAR; PARA LA CONSTRUCCIÓN DEL PROYECTO DENOMINADO  CONSTRUC\",\"descripcion\":\"AUNAR ESFUERZOS ENTRE EL MINISTERIO DEL DEPORTE Y EL MUNICIPIO DE CLEMENCIA; BOLÍVAR; PARA LA CONSTRUCCIÓN DEL PROYECTO DENOMINADO  CONSTRUCCIÓN DE ESTADIO DE DE SOFTBOL EN EL MUNI\",\"entidad\":\"MINISTERIO DEL DEPORTE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2382614\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2454035\",\"nombre_proceso\":\"ADMINISTRACIÓN DELEGADA DE RECURSOS PARA LA OPTIMIZACIÓN Y CONSTRUCCIÓN DE LOS SISTEMAS DE ALCANTARILLADO EN LOS CENTROS POBLADOS TABLAZO; P\",\"descripcion\":\"ADMINISTRACIÓN DELEGADA DE RECURSOS PARA LA OPTIMIZACIÓN Y CONSTRUCCIÓN DE LOS SISTEMAS DE ALCANTARILLADO EN LOS CENTROS POBLADOS TABLAZO; PONTEZUELA; ABREO Y LA PLAYA DEL MUNICIPI\",\"entidad\":\"MUNICIPIO DE RIONEGRO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2390852\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2454516\",\"nombre_proceso\":\"CI-031-2021\",\"descripcion\":\"CI-031-2021\",\"entidad\":\"GOBERNACION DEL MAGDALENA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2395823\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2747342\",\"nombre_proceso\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTIÓN A LA SECRETARIA DISTRITAL DE RECREACION Y DEPORTES PARA EL DESARROLLO DE ACTIVIDADES ENCAMINAD\",\"descripcion\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTIÓN A LA SECRETARIA DISTRITAL DE RECREACION Y DEPORTES PARA EL DESARROLLO DE ACTIVIDADES ENCAMINADAS A LA MASIFICACION Y PROMOCIÓN DEL DEP\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2673037\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2790843\",\"nombre_proceso\":\"CD-436-2022\",\"descripcion\":\"CD-436-2022\",\"entidad\":\"MINISTERIO DE AGRICULTURA Y DESARROLLO RURAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2716096\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.280655\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO AGENCIA NACIONAL INMOBILIARIA\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO AGENCIA NACIONAL INMOBILIARIA\",\"entidad\":\"SUPERINTENDENCIA DEL SUBSIDIO FAMILIAR\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.272852\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2812254\",\"nombre_proceso\":\"CANAL REGIONAL DE TELEVISIÓN TEVEANDINA LTDA\",\"descripcion\":\"CANAL REGIONAL DE TELEVISIÓN TEVEANDINA LTDA\",\"entidad\":\"FONDO UNICO DE TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2737344\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.2899950\",\"nombre_proceso\":\"OICH\",\"descripcion\":\"OICH\",\"entidad\":\"GOBERNACION DEL CHOCÓ.\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2829538\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3076649\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO\",\"entidad\":\"ALCALDIA DE PASTO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2997588\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3100263\",\"nombre_proceso\":\"TECNOLOGIAS DE LA INFORMACION Y PAGO INTEGRADO S.A.S.\",\"descripcion\":\"TECNOLOGIAS DE LA INFORMACION Y PAGO INTEGRADO S.A.S.\",\"entidad\":\"SECRETARIA DISTRITAL DE SEGURIDAD, CONVIVENCIA Y JUSTICIA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3020366\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3279289\",\"nombre_proceso\":\"CONTRATACION DIRECTA 836 DE 2022\",\"descripcion\":\"CONTRATACION DIRECTA 836 DE 2022\",\"entidad\":\"SUPERINTENDENCIA DE NOTARIADO Y REGISTRO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3195558\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.339790\",\"nombre_proceso\":\"PROGRAMA DE LAS NACIONES UNIDAS PARA EL DESARROLLO\",\"descripcion\":\"PROGRAMA DE LAS NACIONES UNIDAS PARA EL DESARROLLO\",\"entidad\":\"MINISTERIO DEL INTERIOR\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.330265\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3482395\",\"nombre_proceso\":\"CONTRATO DE EMPRÉSTITO\",\"descripcion\":\"CONTRATO DE EMPRÉSTITO\",\"entidad\":\"ALCALDIA DE PASTO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3403614\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3506421\",\"nombre_proceso\":\"CONVENIO DE COPERACIÓN INTERNACIONAL - PNUD - DAR\",\"descripcion\":\"CONVENIO DE COPERACIÓN INTERNACIONAL - PNUD - DAR\",\"entidad\":\"MINISTERIO DEL INTERIOR\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3420839\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3533907\",\"nombre_proceso\":\"PLAZA MAYOR MEDELLÍN CONVENCIONES Y EXPOSICIONES S.A.\",\"descripcion\":\"PLAZA MAYOR MEDELLÍN CONVENCIONES Y EXPOSICIONES S.A.\",\"entidad\":\"Ministerio de las Culturas, las Artes y los Saberes\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3450053\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3549233\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA ACOMPAÑAR A LA SECRETARÍA DE GOBIERNO DEL DISTRITO BARRANQUILLA EN LA ATENCIÓN DE COMUNID\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA ACOMPAÑAR A LA SECRETARÍA DE GOBIERNO DEL DISTRITO BARRANQUILLA EN LA ATENCIÓN DE COMUNIDADES NEGRAS AFROCOLOMBIANAS; RAIZALES Y \",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3463192\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3583869\",\"nombre_proceso\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTION\",\"descripcion\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTION\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3498545\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3590455\",\"nombre_proceso\":\"Contrato de Emprestito Banco de Bogotá\",\"descripcion\":\"Contrato de Emprestito Banco de Bogotá\",\"entidad\":\"GOBERNACION DEL MAGDALENA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3506491\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3615563\",\"nombre_proceso\":\"Prestar a la Federación Colombiana de Municipios en cumplimiento de la función pública asignada; por sus propios medios; con plena autonomía\",\"descripcion\":\"Prestar a la Federación Colombiana de Municipios en cumplimiento de la función pública asignada; por sus propios medios; con plena autonomía técnica; administrativa y operacional; \",\"entidad\":\"FEDERACION COLOMBIANA DE MUNICIPIOS\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3530681\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3684277\",\"nombre_proceso\":\"Entrega en comodatos espacio de lote ubicado en la calle 27  entre autopista sur y la carrera 41 Itagui para ser utilizado  Patios  de vehíc\",\"descripcion\":\"Entrega en comodatos espacio de lote ubicado en la calle 27  entre autopista sur y la carrera 41 Itagui para ser utilizado  Patios  de vehículos\",\"entidad\":\"AREA METROPOLITANA DEL VALLE DE ABURRA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3687992\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3714823\",\"nombre_proceso\":\"EL SERVICIO DE MANTENIMIENTO INSPECCIÓN MAYOR ESTRUCTURAL PDM; CUMPLIMIENTO DE BOLETINES DE SERVICIO MANDATORIOS Y TRABAJOS ESPECIALES DE LA\",\"descripcion\":\"EL SERVICIO DE MANTENIMIENTO INSPECCIÓN MAYOR ESTRUCTURAL PDM; CUMPLIMIENTO DE BOLETINES DE SERVICIO MANDATORIOS Y TRABAJOS ESPECIALES DE LA AERONAVE C-130 FAC1018; DE ACUERDO CON \",\"entidad\":\"FUERZA AEROESPACIAL COLOMBIANA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3636264\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3730074\",\"nombre_proceso\":\"CONSTRUCCION DE LA FASE II DEL CENTRO DE INSTRUCCIÓN EN CAÍDA LIBRE E INVESTIGACIÓN AERONÁUTICA DE LA FAC EN LAS INSTALACIONES DEL MUSEO AER\",\"descripcion\":\"CONSTRUCCION DE LA FASE II DEL CENTRO DE INSTRUCCIÓN EN CAÍDA LIBRE E INVESTIGACIÓN AERONÁUTICA DE LA FAC EN LAS INSTALACIONES DEL MUSEO AEROESPACIAL EN TOCANCIPÁ - CUNDINAMARCA; C\",\"entidad\":\"FUERZA AEROESPACIAL COLOMBIANA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3644540\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3730428\",\"nombre_proceso\":\"NORMALIZACIÓN DE REDES ELÉCTRICAS BARRIO SOLIMAR\",\"descripcion\":\"NORMALIZACIÓN DE REDES ELÉCTRICAS BARRIO SOLIMAR\",\"entidad\":\"MINISTERIO DE MINAS Y ENERGIA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3644468\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3755271\",\"nombre_proceso\":\"Contratación Directa\",\"descripcion\":\"Contratación Directa\",\"entidad\":\"POLITECNICO COLOMBIANO JAIME ISAZA CADAVID\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3668146\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3755377\",\"nombre_proceso\":\"SERVICIOS DE MANTENIMIENTO PROGRAMADO (RUTINA) Y MANTENIMIENTO NO PROGRAMADO (NO RUTINA) DE LAS AERONAVES KFIR DE LA FUERZA AÉREA COLOMBIANA\",\"descripcion\":\"SERVICIOS DE MANTENIMIENTO PROGRAMADO (RUTINA) Y MANTENIMIENTO NO PROGRAMADO (NO RUTINA) DE LAS AERONAVES KFIR DE LA FUERZA AÉREA COLOMBIANA; DE ACUERDO A ANEXO TÉCNICO\",\"entidad\":\"FUERZA AEROESPACIAL COLOMBIANA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3675471\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3763772\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA REFORZAR Y FORTALECER LAS ACTIVIDADES DE ASISTENCIA OPERATIVA Y APOYO LOGÍSTICO REQUERIDA\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA REFORZAR Y FORTALECER LAS ACTIVIDADES DE ASISTENCIA OPERATIVA Y APOYO LOGÍSTICO REQUERIDAS POR LA SECRETARÍA DE GOBIERNO MUNICIPA\",\"entidad\":\"MUNICIPIO DE ITAGUI\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3683039\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3766622\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO - ARAUQUITA -ARAUCA\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO - ARAUQUITA -ARAUCA\",\"entidad\":\"MINISTERIO DEL INTERIOR\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3678710\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3766925\",\"nombre_proceso\":\"CONVENIO DE COFINANCIACION - SACUDETE TIPO 1 - TOCA; BOYACA\",\"descripcion\":\"CONVENIO DE COFINANCIACION - SACUDETE TIPO 1 - TOCA; BOYACA\",\"entidad\":\"MINISTERIO DEL INTERIOR\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3678790\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3920667\",\"nombre_proceso\":\"Aunar esfuerzos técnicos; administrativos y financieros para ejecutar el plan integral de ocupantes de espacio público (OEP) de la primera l\",\"descripcion\":\"Aunar esfuerzos técnicos; administrativos y financieros para ejecutar el plan integral de ocupantes de espacio público (OEP) de la primera línea del metro de Bogotá Tramo 1; en su \",\"entidad\":\"METRO DE BOGOTA SA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3827148\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.3982234\",\"nombre_proceso\":\"FORTALECIMIENTO TÉCNICO DEL EJERCICIO DE AUTORIDAD AMBIENTAL  URBANO EN EL ÁREA METROPOLITANA DEL VALLE DE ABURRÁ\",\"descripcion\":\"FORTALECIMIENTO TÉCNICO DEL EJERCICIO DE AUTORIDAD AMBIENTAL  URBANO EN EL ÁREA METROPOLITANA DEL VALLE DE ABURRÁ\",\"entidad\":\"AREA METROPOLITANA DEL VALLE DE ABURRA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3927703\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4038392\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO PARA EL APOYO A LA GESTIÓN DE SERVICIOS ESPECIALIZADOS PARA LAS ETAPAS DE EJECUCIÓN Y RECAUDO DE LA CONTRIBUCIÓ\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO PARA EL APOYO A LA GESTIÓN DE SERVICIOS ESPECIALIZADOS PARA LAS ETAPAS DE EJECUCIÓN Y RECAUDO DE LA CONTRIBUCIÓN DE VALORIZACIÓN PARA EL PROYECTO DE VA\",\"entidad\":\"MUNICIPIO DE RIONEGRO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3944722\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4072320\",\"nombre_proceso\":\"PRESTACION DE SERVICIOS PROFESIONALES\",\"descripcion\":\"PRESTACION DE SERVICIOS PROFESIONALES\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3977655\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4115029\",\"nombre_proceso\":\"ARRENDAMIENTO DE BIENES INMUEBLES PARA EL FUNCIONAMIENTO ADMINISTRATIVO; TECNICO Y OPERATIVO DE LAS DIFERENTES SECRETARIAS Y/O DEPENDENCIAS \",\"descripcion\":\"ARRENDAMIENTO DE BIENES INMUEBLES PARA EL FUNCIONAMIENTO ADMINISTRATIVO; TECNICO Y OPERATIVO DE LAS DIFERENTES SECRETARIAS Y/O DEPENDENCIAS EN EL DISTRITO DE BARRANQUILLA.\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4020262\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4134446\",\"nombre_proceso\":\"CONTRATO INTERADMINISTRATIVO POR ADMINISTRACIÓN DELEGADA PARA APOYAR EL DESARROLLO DE ESTRATEGIAS ENCAMINADAS A PROMOVER EL FORTALECIMIENTO \",\"descripcion\":\"CONTRATO INTERADMINISTRATIVO POR ADMINISTRACIÓN DELEGADA PARA APOYAR EL DESARROLLO DE ESTRATEGIAS ENCAMINADAS A PROMOVER EL FORTALECIMIENTO EMPRESARIAL; CRECIMIENTO DEL TURISMO; PR\",\"entidad\":\"MUNICIPIO DE RIONEGRO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4039795\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4154601\",\"nombre_proceso\":\"CDPS CANASTA EDUCATIVA 012 DE 2023\",\"descripcion\":\"CDPS CANASTA EDUCATIVA 012 DE 2023\",\"entidad\":\"GOBERNACIÓN DE LA GUAJIRA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4059614\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4157866\",\"nombre_proceso\":\"DESARROLLO DE LAS ACTIVIDADES DEL PLAN DE INTERVENCIONES COLECTIVAS (PIC); PARA LA PROMOCION DE LA SALUD Y PREVENCION DE LA ENFERMEDAD EN EL\",\"descripcion\":\"DESARROLLO DE LAS ACTIVIDADES DEL PLAN DE INTERVENCIONES COLECTIVAS (PIC); PARA LA PROMOCION DE LA SALUD Y PREVENCION DE LA ENFERMEDAD EN EL MARCO DE ATENCION PRIMARIA EN SALUD (AP\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4063175\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4292664\",\"nombre_proceso\":\"Prestar los servicios integrales como operador de comunicaciones para la preproducción; producción; postproducción y emisión de piezas; prog\",\"descripcion\":\"Prestar los servicios integrales como operador de comunicaciones para la preproducción; producción; postproducción y emisión de piezas; programas y campañas audiovisuales para la p\",\"entidad\":\"DEFENSORÍA DEL PUEBLO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4209126\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4328089\",\"nombre_proceso\":\"ADQUISICIÓN DE LICENCIAMIENTO; NETAPP Y OPTIMIZACIÓN PARA LA PLATAFORMA DE RECEPCIÓN Y ANÁLISIS DE COMUNICACIONES INTERCEPTADAS\",\"descripcion\":\"ADQUISICIÓN DE LICENCIAMIENTO; NETAPP Y OPTIMIZACIÓN PARA LA PLATAFORMA DE RECEPCIÓN Y ANÁLISIS DE COMUNICACIONES INTERCEPTADAS\",\"entidad\":\"DIRECCIÓN DE INVESTIGACIÓN CRIMINAL E INTERPOL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4235876\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4374101\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS - COMPENSAR\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS - COMPENSAR\",\"entidad\":\"TRANSMILENIO S.A.\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4279522\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4398956\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS - COMPENSAR\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS - COMPENSAR\",\"entidad\":\"TRANSMILENIO S.A.\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4312142\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4442088\",\"nombre_proceso\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTIÓN A LA GERENCIA DE LAS TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES DEL DISTRITO DE BARRAN\",\"descripcion\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTIÓN A LA GERENCIA DE LAS TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES DEL DISTRITO DE BARRANQUILLA EN EL SOPORTE; MANTENIMIENTO Y AC\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4345290\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4452101\",\"nombre_proceso\":\"GERENCIA INTEGRAL Y COORDINACION PARA LA ADMINISTRACION; MANTENIMIENTO Y REPARACION DE LA INSFRAESTRUCTURA FISICA DE LOS ESCENARIOS DEPORTIV\",\"descripcion\":\"GERENCIA INTEGRAL Y COORDINACION PARA LA ADMINISTRACION; MANTENIMIENTO Y REPARACION DE LA INSFRAESTRUCTURA FISICA DE LOS ESCENARIOS DEPORTIVOS DEL DISTRITO DE BARRANQUILLA\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4356428\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4680510\",\"nombre_proceso\":\"Servicios en ejecución de las campañas de comunicación pública; en el marco del programa de información y comunicación pública\",\"descripcion\":\"Servicios en ejecución de las campañas de comunicación pública; en el marco del programa de información y comunicación pública\",\"entidad\":\"METRO DE BOGOTA SA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4582575\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4714843\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS PARA LA REALIZACIÓN DE ACTIVIDADES DE BIENESTAR PARA EL PERSONAL QUE INTEGRA LA SECCIONAL DE TRÁNSITO Y TRANSPORTE D\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS PARA LA REALIZACIÓN DE ACTIVIDADES DE BIENESTAR PARA EL PERSONAL QUE INTEGRA LA SECCIONAL DE TRÁNSITO Y TRANSPORTE DE BOGOTÁ\",\"entidad\":\"FONDO ROTATORIO DE LA POLICIA NACIONAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4618181\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4760867\",\"nombre_proceso\":\"CONVENIO DE ASOCIACION\",\"descripcion\":\"CONVENIO DE ASOCIACION\",\"entidad\":\"GOBERNACIÓN DE BOYACÁ\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4663504\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4885898\",\"nombre_proceso\":\"BOLSA MERCANTIL DE COLOMBIA S.A.\",\"descripcion\":\"BOLSA MERCANTIL DE COLOMBIA S.A.\",\"entidad\":\"MINISTERIO DE AGRICULTURA Y DESARROLLO RURAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4787508\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4888521\",\"nombre_proceso\":\"Aunar esfuerzos técnicos; administrativos y financieros para apoyar el desarrollo de las actividades de los programas de la Dirección de Fom\",\"descripcion\":\"Aunar esfuerzos técnicos; administrativos y financieros para apoyar el desarrollo de las actividades de los programas de la Dirección de Fomento y Desarrollo; enfocado a la promoci\",\"entidad\":\"MINISTERIO DEL DEPORTE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4791313\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4905303\",\"nombre_proceso\":\"Aunar esfuerzos técnicos; administrativos y financieros para apoyar el desarrollo de las actividades de los programas de la Dirección de Fom\",\"descripcion\":\"Aunar esfuerzos técnicos; administrativos y financieros para apoyar el desarrollo de las actividades de los programas de la Dirección de Fomento y Desarrollo; enfocado a la promoci\",\"entidad\":\"MINISTERIO DEL DEPORTE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4812141\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.4932500\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO\",\"entidad\":\"AGENCIA DE DESARROLLO RURAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4833910\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5009790\",\"nombre_proceso\":\"Contrato Interadministrativo - ENTerritorio\",\"descripcion\":\"Contrato Interadministrativo - ENTerritorio\",\"entidad\":\"MINISTERIO DEL DEPORTE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4909374\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5077712\",\"nombre_proceso\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTION PARA BRINDAR ACOMPAÑAMIENTO A LA SECRETARIA DISTRITAL DE RECREACION Y DEPORTE PARA EL DESARROL\",\"descripcion\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTION PARA BRINDAR ACOMPAÑAMIENTO A LA SECRETARIA DISTRITAL DE RECREACION Y DEPORTE PARA EL DESARROLLO DE LA ESTRATEGIA DE FOMENTO DEL DEPOR\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4976590\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5079345\",\"nombre_proceso\":\"SERVICIOS ESPECIALIZADOS DE GESTIÓN DOCUMENTAL\",\"descripcion\":\"SERVICIOS ESPECIALIZADOS DE GESTIÓN DOCUMENTAL\",\"entidad\":\"PROCURADURIA GENERAL DE LA NACION\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.4978414\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5116390\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO\",\"entidad\":\"MINISTERIO DE AMBIENTE Y DESARROLLO SOSTENIBLE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.5015617\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5128143\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO\",\"entidad\":\"MINISTERIO DE AMBIENTE Y DESARROLLO SOSTENIBLE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.5026468\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5134421\",\"nombre_proceso\":\"DESARROLLAR EL PROCESO DE EVALUACIÓN INTEGRAL DE LOS PROYECTOS QUE PARTICIPEN EN LAS CONVOCATORIAS DEL PROGRAMA NACIONAL DE CONCERTACIÓN CUL\",\"descripcion\":\"DESARROLLAR EL PROCESO DE EVALUACIÓN INTEGRAL DE LOS PROYECTOS QUE PARTICIPEN EN LAS CONVOCATORIAS DEL PROGRAMA NACIONAL DE CONCERTACIÓN CULTURAL 2024\",\"entidad\":\"Ministerio de las Culturas, las Artes y los Saberes\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.5032868\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5161265\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN DE LA SECRETARÍA DISTRITAL DE RECREACIÓN Y DEPORTE PARA EL DESARROLLO DEL PROGRAMA BARRANQUILL\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN DE LA SECRETARÍA DISTRITAL DE RECREACIÓN Y DEPORTE PARA EL DESARROLLO DEL PROGRAMA BARRANQUILLA ACTIVA Y RECREATIVA A TRAVES DE LA EST\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.5059181\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5269290\",\"nombre_proceso\":\"CONTRATO INTERADMINISTRATIVO\",\"descripcion\":\"CONTRATO INTERADMINISTRATIVO\",\"entidad\":\"GOBERNACION DE NARIÑO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.5166271\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5291411\",\"nombre_proceso\":\"Prestación de servicios profesionales de abogado para la representación judicial; extrajudicial y las demás que disponga la ley en favor de \",\"descripcion\":\"Prestación de servicios profesionales de abogado para la representación judicial; extrajudicial y las demás que disponga la ley en favor de los usuarios del servicio de defensoría \",\"entidad\":\"DEFENSORÍA DEL PUEBLO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.5248108\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5319517\",\"nombre_proceso\":\"Aunar esfuerzos técnicos y recursos físicos para la  implementación de la ruta o metodología que permita a través  del diálogo e intercambio\",\"descripcion\":\"Aunar esfuerzos técnicos y recursos físicos para la  implementación de la ruta o metodología que permita a través  del diálogo e intercambio de saberes; adelantar las acciones  nec\",\"entidad\":\"MINISTERIO DE MINAS Y ENERGIA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.5215643\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5327151\",\"nombre_proceso\":\"UNION TEMPORAL RH\",\"descripcion\":\"UNION TEMPORAL RH\",\"entidad\":\"DEPARTAMENTO ADMINISTRATIVO DE LA PRESIDENCIA DE LA REPÚBLICA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.5223079\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5408655\",\"nombre_proceso\":\"ARRENDAMIENTO INMUEBLE SEDE PRINCIPAL BOGOTÁ\",\"descripcion\":\"ARRENDAMIENTO INMUEBLE SEDE PRINCIPAL BOGOTÁ\",\"entidad\":\"SUPERINTENDENCIA NACIONAL DE SALUD\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.5311225\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5463625\",\"nombre_proceso\":\"PROCOMERCIO S.A\",\"descripcion\":\"PROCOMERCIO S.A\",\"entidad\":\"MINISTERIO DE AGRICULTURA Y DESARROLLO RURAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.5360131\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5935970\",\"nombre_proceso\":\"Prestación de servicios profesionales para brindar asesoría y seguimiento de la estrategia  de fortalecimiento de capacidad técnica para el \",\"descripcion\":\"Prestación de servicios profesionales para brindar asesoría y seguimiento de la estrategia  de fortalecimiento de capacidad técnica para el ejercicio misional preventivo en la  imp\",\"entidad\":\"DEFENSORÍA DEL PUEBLO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.5929808\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5943339\",\"nombre_proceso\":\"CONTRATAR EL SERVICIO Y SOSTENIBILIDAD DE LA SOLUCIÓN TECNOLÓGICA INTEGRADA  TIPO SaaS DENOMINADA PARA LA DEFENSORÍA DEL PUEBLO APP CONTIGO;\",\"descripcion\":\"CONTRATAR EL SERVICIO Y SOSTENIBILIDAD DE LA SOLUCIÓN TECNOLÓGICA INTEGRADA  TIPO SaaS DENOMINADA PARA LA DEFENSORÍA DEL PUEBLO APP CONTIGO; PARA LA  TOMA DE DECISIONES OPORTUNAS E\",\"entidad\":\"DEFENSORÍA DEL PUEBLO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.5854184\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.5987887\",\"nombre_proceso\":\"Fondo Mixto de Promoción de la Cultura y las Artes de Boyacá\",\"descripcion\":\"Fondo Mixto de Promoción de la Cultura y las Artes de Boyacá\",\"entidad\":\"Ministerio de las Culturas, las Artes y los Saberes\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.5876910\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6116403\",\"nombre_proceso\":\"ALIANZA PUBLICA PARA EL DESDARROLLO INTEGRAL (ALDESARROLLO)\",\"descripcion\":\"ALIANZA PUBLICA PARA EL DESDARROLLO INTEGRAL (ALDESARROLLO)\",\"entidad\":\"GOBERNACION DEL CHOCÓ.\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6005101\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6237418\",\"nombre_proceso\":\"Universidad Nacional de Colombia\",\"descripcion\":\"Universidad Nacional de Colombia\",\"entidad\":\"SECRETARIA DE EDUCACION DEL DISTRITO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6128253\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6477624\",\"nombre_proceso\":\"INTERNEXA S.A.- ACUERDO ESPECÍFICO No. 4\",\"descripcion\":\"INTERNEXA S.A.- ACUERDO ESPECÍFICO No. 4\",\"entidad\":\"FONDO UNICO DE TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6363642\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6487930\",\"nombre_proceso\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTION PARA CUSTODIAR; ALMACENAR Y ADMINISTRAR ARCHIVOS FISICOS DE LA ALCALDIA DISTRITAL DE BARRANQUI\",\"descripcion\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTION PARA CUSTODIAR; ALMACENAR Y ADMINISTRAR ARCHIVOS FISICOS DE LA ALCALDIA DISTRITAL DE BARRANQUILLA; ASI COMO TAMBIEN REGISTRAR; ATENDER\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6373487\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6509936\",\"nombre_proceso\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTION PARA CUSTODIAR; ALMACENAR Y ADMINISTRAR ARCHIVOS FISICOS DE LA ALCALDIA DISTRITAL DE BARRANQUI\",\"descripcion\":\"PRESTACION DE SERVICIOS DE APOYO A LA GESTION PARA CUSTODIAR; ALMACENAR Y ADMINISTRAR ARCHIVOS FISICOS DE LA ALCALDIA DISTRITAL DE BARRANQUILLA; ASI COMO TAMBIEN REGISTRAR; ATENDER\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6395315\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6527293\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO FDL FONTIBON\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO FDL FONTIBON\",\"entidad\":\"(Secretaría Distrital de Integración Social)\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6413656\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6594810\",\"nombre_proceso\":\"CONVENIO UNIVERSIDAD DE CALDAS\",\"descripcion\":\"CONVENIO UNIVERSIDAD DE CALDAS\",\"entidad\":\"FONDO UNICO DE TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6482253\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6601074\",\"nombre_proceso\":\"CI-011-2024\",\"descripcion\":\"CI-011-2024\",\"entidad\":\"GOBERNACION DEL MAGDALENA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6484864\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6635424\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA ACOMPAÑAR A LA ENTIDAD EN EL DESARROLLO DE LAS PRESENTACIONES ARTÍSTICAS; CULTURALES; INC\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA ACOMPAÑAR A LA ENTIDAD EN EL DESARROLLO DE LAS PRESENTACIONES ARTÍSTICAS; CULTURALES; INCLUIDO EL CONCIERTO DE MÚSICA GOSPEL; ENM\",\"entidad\":\"MUNICIPIO DE ITAGUI\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6520474\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.666327\",\"nombre_proceso\":\"SERVICIOS POSTALES Y GESTIÓN DOCUMENTAL\",\"descripcion\":\"SERVICIOS POSTALES Y GESTIÓN DOCUMENTAL\",\"entidad\":\"DEPARTAMENTO NACIONAL DE PLANEACIÓN\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.638702\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6735811\",\"nombre_proceso\":\"CONVENIO CENSO HABITANTE DE CALLE SDIS-SDP\",\"descripcion\":\"CONVENIO CENSO HABITANTE DE CALLE SDIS-SDP\",\"entidad\":\"(Secretaría Distrital de Integración Social)\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6619354\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6808401\",\"nombre_proceso\":\"FIDUCOLDEX-FONDO MUJER LIBRE Y REPRODUCTIVA\",\"descripcion\":\"FIDUCOLDEX-FONDO MUJER LIBRE Y REPRODUCTIVA\",\"entidad\":\"MINISTERIO DE IGUALDAD Y EQUIDAD\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6706476\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6824136\",\"nombre_proceso\":\"DEPARTAMENTO DE VAUPÉS\",\"descripcion\":\"DEPARTAMENTO DE VAUPÉS\",\"entidad\":\"FONDO UNICO DE TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6706088\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6827516\",\"nombre_proceso\":\"FIDUCOLDEX-FONDO MUJER LIBRE Y PRODUCTIVA\",\"descripcion\":\"FIDUCOLDEX-FONDO MUJER LIBRE Y PRODUCTIVA\",\"entidad\":\"MINISTERIO DE IGUALDAD Y EQUIDAD\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6709881\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6835303\",\"nombre_proceso\":\"CONTRATO UNAL CONCERTACIONES\",\"descripcion\":\"CONTRATO UNAL CONCERTACIONES\",\"entidad\":\"Ministerio de las Culturas, las Artes y los Saberes\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6717498\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.6995311\",\"nombre_proceso\":\"AUNAR ESFUERZOS TÉCNICOS; ADMINISTRATIVOS Y FINANCIEROS PARA FORTALECIMIENTO DE LA INFRAESTRUCTURA TECNOLÓGICA Y DINAMIZACIÓN DE LA IMPLEMEN\",\"descripcion\":\"AUNAR ESFUERZOS TÉCNICOS; ADMINISTRATIVOS Y FINANCIEROS PARA FORTALECIMIENTO DE LA INFRAESTRUCTURA TECNOLÓGICA Y DINAMIZACIÓN DE LA IMPLEMENTACIÓN DE LA POLÍTICA DE GOBIERNO DIGITA\",\"entidad\":\"GOBERNACION DE SUCRE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6878334\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7278319\",\"nombre_proceso\":\"CONTRATO INTERADMINISTRATIVO CON LA EMPRESA GENERADORA DE ENERGIA DEL TOLIMA EGETSA SA ESP\",\"descripcion\":\"CONTRATO INTERADMINISTRATIVO CON LA EMPRESA GENERADORA DE ENERGIA DEL TOLIMA EGETSA SA ESP\",\"entidad\":\"GOBIERNO DEPARTAMENTAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7166709\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7280635\",\"nombre_proceso\":\"CORPORACION FINANCIERA INTERNACIONAL\",\"descripcion\":\"CORPORACION FINANCIERA INTERNACIONAL\",\"entidad\":\"SECRETARIA DE EDUCACION DEL DISTRITO\",\"url_secop\":\"https://community.secop.gov.co/STS/Users/Login/Index\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7338894\",\"nombre_proceso\":\"Contratar bajo la modalidad de arrendamiento el inmueble; para el funcionamiento de la sede Archivo de Gestión; para la administración; cons\",\"descripcion\":\"Contratar bajo la modalidad de arrendamiento el inmueble; para el funcionamiento de la sede Archivo de Gestión; para la administración; conservación y custodia de la información de\",\"entidad\":\"PROSPERIDAD SOCIAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7222199\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7431031\",\"nombre_proceso\":\"EJECUTAR ACTIVIDADES ADMINISTRATIVAS; OPERATIVAS; TÉCNICAS; ASISTENCIALES Y LOGÍSTICAS QUE FORTALEZCAN LA GESTIÓN Y EL FUNCIONAMIENTO DE LAS\",\"descripcion\":\"EJECUTAR ACTIVIDADES ADMINISTRATIVAS; OPERATIVAS; TÉCNICAS; ASISTENCIALES Y LOGÍSTICAS QUE FORTALEZCAN LA GESTIÓN Y EL FUNCIONAMIENTO DE LAS 24 INSTITUCIONES EDUCATIVAS OFICIALES Y\",\"entidad\":\"MUNICIPIO DE ITAGUI\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7321180\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7449230\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA SOPORTAR AL MUNICIPIO DE ITAGÜÍ EN LA EJECUCIÓN DE ACTIVIDADES OPERATIVAS; LOGÍSTICAS; TÉ\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA SOPORTAR AL MUNICIPIO DE ITAGÜÍ EN LA EJECUCIÓN DE ACTIVIDADES OPERATIVAS; LOGÍSTICAS; TÉCNICAS; ASISTENCIALES Y ADMINISTRATIVAS \",\"entidad\":\"MUNICIPIO DE ITAGUI\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7330359\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7466190\",\"nombre_proceso\":\"BRINDAR ATENCION BASICA INTEGRAL Y PROTECCION AL ADULTO MAYOR; EN LA MODALIDAD INTERNADO; PARA GARANTIZAR EL MEJORAMIENTO DE LA CALIDAD DE V\",\"descripcion\":\"BRINDAR ATENCION BASICA INTEGRAL Y PROTECCION AL ADULTO MAYOR; EN LA MODALIDAD INTERNADO; PARA GARANTIZAR EL MEJORAMIENTO DE LA CALIDAD DE VIDA DE LOS ADULTOS MAYORES VULNERABLES Y\",\"entidad\":\"MUNICIPIO DE RIONEGRO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7346489\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.755348\",\"nombre_proceso\":\"Prestación de servicios profesionales especializados y de apoyo a la gestión para realizar la evaluación en fase de precalificación del proc\",\"descripcion\":\"Prestación de servicios profesionales especializados y de apoyo a la gestión para realizar la evaluación en fase de precalificación del proceso GT-LPI-001-2018\",\"entidad\":\"METRO DE BOGOTA SA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.727638\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7561573\",\"nombre_proceso\":\"CONTRATO INTERADMINISTRATIVO PARA EL ARRENDAMIENTO DE VEHÍCULOS PARA APOYAR A LA FUERZA PÚBLICA; ORGANISMOS DE SEGURIDAD Y OTRAS ENTIDADES D\",\"descripcion\":\"CONTRATO INTERADMINISTRATIVO PARA EL ARRENDAMIENTO DE VEHÍCULOS PARA APOYAR A LA FUERZA PÚBLICA; ORGANISMOS DE SEGURIDAD Y OTRAS ENTIDADES DEL MUNICIPIO DE RIONEGRO - ANTIOQUIA\",\"entidad\":\"MUNICIPIO DE RIONEGRO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7441197\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7578958\",\"nombre_proceso\":\"IMPLEMENTACIÓN DE ACCIONES PARA EL FORTALECIMIENTO DE LA GESTIÓN CATASTRAL DEL ÁREA METROPOLITANA DEL VALLE DE ABURRÁ FASE 2025\",\"descripcion\":\"IMPLEMENTACIÓN DE ACCIONES PARA EL FORTALECIMIENTO DE LA GESTIÓN CATASTRAL DEL ÁREA METROPOLITANA DEL VALLE DE ABURRÁ FASE 2025\",\"entidad\":\"AREA METROPOLITANA DEL VALLE DE ABURRA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7489575\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7603157\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO - UNP\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO - UNP\",\"entidad\":\"MINISTERIO DEL INTERIOR\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7483281\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.769107\",\"nombre_proceso\":\"FENASCOL\",\"descripcion\":\"FENASCOL\",\"entidad\":\"FONDO UNICO DE TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.737357\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7700801\",\"nombre_proceso\":\"ORGANIZACIÓN WIWA YUGUMAIUN BUNKUANARRUA TAIRONA\",\"descripcion\":\"ORGANIZACIÓN WIWA YUGUMAIUN BUNKUANARRUA TAIRONA\",\"entidad\":\"GOBERNACIÓN DE LA GUAJIRA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7579096\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7710982\",\"nombre_proceso\":\"PRESTACION DEL SERVICIO PARA ADMINISTRAR LA CANASTA PARCIAL DE LA POBLACION ETNICA DE LA INSTITUCION ETNOEDUCATIVA INTEGRAL RURAL JACOBO ARE\",\"descripcion\":\"PRESTACION DEL SERVICIO PARA ADMINISTRAR LA CANASTA PARCIAL DE LA POBLACION ETNICA DE LA INSTITUCION ETNOEDUCATIVA INTEGRAL RURAL JACOBO ARENDS GOURIYU; INSTITUCION ETNOEDUCATIVA I\",\"entidad\":\"MUNICIPIO DE URIBIA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7589624\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7769056\",\"nombre_proceso\":\"CONSEJO DE AUTORIDADES  TRADICIONALES INDÍGENAS DEL CHOCÓ_ CATICH\",\"descripcion\":\"CONSEJO DE AUTORIDADES  TRADICIONALES INDÍGENAS DEL CHOCÓ_ CATICH\",\"entidad\":\"GOBERNACION DEL CHOCÓ.\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7651491\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7844264\",\"nombre_proceso\":\"EMPRESA DE TELECOMUNICACIONES DE BOGOTÁ ETB S.A. ESP\",\"descripcion\":\"EMPRESA DE TELECOMUNICACIONES DE BOGOTÁ ETB S.A. ESP\",\"entidad\":\"SECRETARIA DE EDUCACION DEL DISTRITO\",\"url_secop\":\"https://community.secop.gov.co/STS/Users/Login/Index\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7865778\",\"nombre_proceso\":\"Prestación de servicios profesionales de abogado para la representación judicial; extrajudicial y las demás que disponga la ley en favor de \",\"descripcion\":\"Prestación de servicios profesionales de abogado para la representación judicial; extrajudicial y las demás que disponga la ley en favor de los usuarios del servicio de defensoría \",\"entidad\":\"DEFENSORÍA DEL PUEBLO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7744188\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7961767\",\"nombre_proceso\":\"ESTUDIOS PARA EL ACOTAMIENTO DE LA RONDA HÍDRICA DEL RÍO MEDELLÍN - ABURRÁ EN EL VALLE DE ABURRÁ\",\"descripcion\":\"ESTUDIOS PARA EL ACOTAMIENTO DE LA RONDA HÍDRICA DEL RÍO MEDELLÍN - ABURRÁ EN EL VALLE DE ABURRÁ\",\"entidad\":\"AREA METROPOLITANA DEL VALLE DE ABURRA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7977449\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.7966756\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA LA ATENCIÓN EN EDUCACIÓN INICIAL A LA PRIMERA INFANCIA EN EL MARCO DE LA  POLÍTICA DE EST\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA LA ATENCIÓN EN EDUCACIÓN INICIAL A LA PRIMERA INFANCIA EN EL MARCO DE LA  POLÍTICA DE ESTADO PARA EL DESARROLLO INTEGRAL DE LA PR\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7843892\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8005315\",\"nombre_proceso\":\"CONTRATAR EL ARRENDAMIENTO DEL ESPACIO FÍSICO PARA LA CONSERVACIÓN DE LOS ARCHIVOS DE LA GESTIÓN DOCUMENTAL DE LA SECRETARÍA DISTRITAL DE MO\",\"descripcion\":\"CONTRATAR EL ARRENDAMIENTO DEL ESPACIO FÍSICO PARA LA CONSERVACIÓN DE LOS ARCHIVOS DE LA GESTIÓN DOCUMENTAL DE LA SECRETARÍA DISTRITAL DE MOVILIDAD.\",\"entidad\":\"SECRETARÍA DISTRITAL DE MOVILIDAD\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7889991\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8018431\",\"nombre_proceso\":\"LA ADQUISICIÓN BLINDAJES AERONÁUTICOS; SEGÚN ANEXO TÉCNICO; COMO SOPORTE PARA LA FUNCIONALIDAD Y OPERATIVIDAD DE LOS SISTEMAS DE ARMAS DE LA\",\"descripcion\":\"LA ADQUISICIÓN BLINDAJES AERONÁUTICOS; SEGÚN ANEXO TÉCNICO; COMO SOPORTE PARA LA FUNCIONALIDAD Y OPERATIVIDAD DE LOS SISTEMAS DE ARMAS DE LA FUERZA AEROESPACIAL COLOMBIANA.\",\"entidad\":\"FUERZA AEROESPACIAL COLOMBIANA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7912562\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8020511\",\"nombre_proceso\":\"CONTRATO INTERADMINISTRATIVO DE MANDATO SIN REPRESENTACION PARA LA ADMNISTRACION DE LOS SERVICIOS DE VIGILANCIA Y SEGURIDAD PRIVADA DEL PERS\",\"descripcion\":\"CONTRATO INTERADMINISTRATIVO DE MANDATO SIN REPRESENTACION PARA LA ADMNISTRACION DE LOS SERVICIOS DE VIGILANCIA Y SEGURIDAD PRIVADA DEL PERSONAL LOS BIENES INMUEBLES Y SU CONTENIDO\",\"entidad\":\"AREA METROPOLITANA DEL VALLE DE ABURRA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7912558\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8035689\",\"nombre_proceso\":\"ASESOFTWARE S.A.S\",\"descripcion\":\"ASESOFTWARE S.A.S\",\"entidad\":\"FONDO UNICO DE TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7921422\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8037736\",\"nombre_proceso\":\"Prestación de servicios profesionales de abogado para la representación judicial; extrajudicial y las demás que disponga la ley en favor de \",\"descripcion\":\"Prestación de servicios profesionales de abogado para la representación judicial; extrajudicial y las demás que disponga la ley en favor de los usuarios del servicio de defensoría \",\"entidad\":\"DEFENSORÍA DEL PUEBLO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7918350\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8057757\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO\",\"entidad\":\"GOBERNACIÓN DE BOYACÁ\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7960849\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8073472\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN DE  LA OFICINA PARA LA SEGURIDAD Y CONVIVENCIA CIUDADANA DEL DISTRITO DE BARRANQUILLA\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN DE  LA OFICINA PARA LA SEGURIDAD Y CONVIVENCIA CIUDADANA DEL DISTRITO DE BARRANQUILLA\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7951692\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8112173\",\"nombre_proceso\":\"CONTRATO EMPRESTITO DE CREDITO DE TESORERÍA A CORTO PLAZO CELEBRADO ENTRE EL DISTRITO ESPECIAL; INDUSTRIAL Y PORTUARIO DE BARRANQUILLA Y ALI\",\"descripcion\":\"CONTRATO EMPRESTITO DE CREDITO DE TESORERÍA A CORTO PLAZO CELEBRADO ENTRE EL DISTRITO ESPECIAL; INDUSTRIAL Y PORTUARIO DE BARRANQUILLA Y ALIANZA FIDUCIARIA S.A.\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7990225\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8160797\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA  GESTIÓN PARA LA IMPLEMENTACIÓN DE LA ESTRATEGIA SISTEMAS LOCALES DE JUSTICIA  POR PARTE DE LA SECRETA\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA  GESTIÓN PARA LA IMPLEMENTACIÓN DE LA ESTRATEGIA SISTEMAS LOCALES DE JUSTICIA  POR PARTE DE LA SECRETARÍA DE GOBIERNO DEL DISTRITO DE BARRANQU\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8036751\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8184970\",\"nombre_proceso\":\"CI-0866-2025\",\"descripcion\":\"CI-0866-2025\",\"entidad\":\"GOBERNACION DEL MAGDALENA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8060041\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8257854\",\"nombre_proceso\":\"CONVENIO ALCANTARILLADO ISTMINA\",\"descripcion\":\"CONVENIO ALCANTARILLADO ISTMINA\",\"entidad\":\"GOBERNACION DEL CHOCÓ.\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8132711\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8326782\",\"nombre_proceso\":\"Prestación de servicios profesionales de abogado para la representación judicial; extrajudicial y las demás que disponga la ley en favor de \",\"descripcion\":\"Prestación de servicios profesionales de abogado para la representación judicial; extrajudicial y las demás que disponga la ley en favor de los usuarios del servicio de defensoría \",\"entidad\":\"DEFENSORÍA DEL PUEBLO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8200590\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8326892\",\"nombre_proceso\":\"CONTRATO INTERADMINISTRATIVO PARA LA GERENCIA Y OPERACIÓN INTEGRAL DEL COMPONENTE RIOBUS INTEGRADO AL ESPACIO PUBLICO DENOMINADO GRAN MALECO\",\"descripcion\":\"CONTRATO INTERADMINISTRATIVO PARA LA GERENCIA Y OPERACIÓN INTEGRAL DEL COMPONENTE RIOBUS INTEGRADO AL ESPACIO PUBLICO DENOMINADO GRAN MALECON DEL RIO\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8200728\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8376652\",\"nombre_proceso\":\"CONTRATO INTERADMINISTRATIVO PARA LOS ESTUDIOS Y DISEÑOS TÉCNICOS INTEGRALES DE LA NUEVA PLAZA DE MERCADO DEL MUNICIPIO DE RIONEGRO ANTIOQUI\",\"descripcion\":\"CONTRATO INTERADMINISTRATIVO PARA LOS ESTUDIOS Y DISEÑOS TÉCNICOS INTEGRALES DE LA NUEVA PLAZA DE MERCADO DEL MUNICIPIO DE RIONEGRO ANTIOQUIA; ASÍ COMO LA FORMULACIÓN Y ESTRUCTURAC\",\"entidad\":\"MUNICIPIO DE RIONEGRO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8250335\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8686546\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS PROFESIONALES PARA  BRINDAR ACOMPAÑAMIENTO A LA SECRETARÍA DISTRITAL DE GESTIÓN HUMANA  EN EL PLAN INSTITUCIONAL DE \",\"descripcion\":\"PRESTACIÓN DE SERVICIOS PROFESIONALES PARA  BRINDAR ACOMPAÑAMIENTO A LA SECRETARÍA DISTRITAL DE GESTIÓN HUMANA  EN EL PLAN INSTITUCIONAL DE CAPACITACIÓN DIRIGIDO A LOS FUNCIONARIOS\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8554272\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8691802\",\"nombre_proceso\":\"LA ASOCIACIÓN DE AUTORIDADES TRADICIONALES AWA - ORGANIZACIÓN UNIDAD INDÍGENA DEL PUEBLO AWA- UNIPA\",\"descripcion\":\"LA ASOCIACIÓN DE AUTORIDADES TRADICIONALES AWA - ORGANIZACIÓN UNIDAD INDÍGENA DEL PUEBLO AWA- UNIPA\",\"entidad\":\"MINISTERIO DE AGRICULTURA Y DESARROLLO RURAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8559767\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8801252\",\"nombre_proceso\":\"CONVENIO SAN ANDRES\",\"descripcion\":\"CONVENIO SAN ANDRES\",\"entidad\":\"AGENCIA DE DESARROLLO RURAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8672321\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8855449\",\"nombre_proceso\":\"DEPARTAMENTO DE PUTUMAYO\",\"descripcion\":\"DEPARTAMENTO DE PUTUMAYO\",\"entidad\":\"FONDO UNICO DE TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8725864\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8934765\",\"nombre_proceso\":\"Prestación de servicios profesionales de abogado para la representación judicial; extrajudicial y las demás que disponga la ley en favor de \",\"descripcion\":\"Prestación de servicios profesionales de abogado para la representación judicial; extrajudicial y las demás que disponga la ley en favor de los usuarios del servicio de defensoría \",\"entidad\":\"DEFENSORÍA DEL PUEBLO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8800669\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8985249\",\"nombre_proceso\":\"Prestar los servicios de organización de fondo documental acumulado; digitalización de documentos y administración; almacenamiento y custodi\",\"descripcion\":\"Prestar los servicios de organización de fondo documental acumulado; digitalización de documentos y administración; almacenamiento y custodia del archivo del CNE; en cumplimiento d\",\"entidad\":\"Consejo Nacional Electoral\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8851305\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.8999605\",\"nombre_proceso\":\"AGENCIA NACIONAL INMOBILIARIA VIRGILIO BARCO VARGAS\",\"descripcion\":\"AGENCIA NACIONAL INMOBILIARIA VIRGILIO BARCO VARGAS\",\"entidad\":\"MINISTERIO DE HACIENDA Y CREDITO PUBLICO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8864269\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9018983\",\"nombre_proceso\":\"CONECTIVIDAD ESCOLAR 2025\",\"descripcion\":\"CONECTIVIDAD ESCOLAR 2025\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8884094\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9021155\",\"nombre_proceso\":\"CONTRATO INTERADMINISTRATIVO POR ADMINISTRACIÓN DELEGADA DE RECURSOS PARA LA CONSTRUCCIÓN E INTERVENTORIA DEL PROYECTO CIUDADELA EDUCATIVA 4\",\"descripcion\":\"CONTRATO INTERADMINISTRATIVO POR ADMINISTRACIÓN DELEGADA DE RECURSOS PARA LA CONSTRUCCIÓN E INTERVENTORIA DEL PROYECTO CIUDADELA EDUCATIVA 4.0 ETAPA 1 EN EL MUNICIPIO DE RIONEGRO.\",\"entidad\":\"MUNICIPIO DE RIONEGRO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8886200\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9100088\",\"nombre_proceso\":\"CONVENIO GUAJIRA\",\"descripcion\":\"CONVENIO GUAJIRA\",\"entidad\":\"AGENCIA DE DESARROLLO RURAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8967705\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9104218\",\"nombre_proceso\":\"CONVENIO INTERADMINISTRATIVO - CABILDO KOGUI - DAI\",\"descripcion\":\"CONVENIO INTERADMINISTRATIVO - CABILDO KOGUI - DAI\",\"entidad\":\"MINISTERIO DEL INTERIOR\",\"url_secop\":\"https://community.secop.gov.co/STS/Users/Login/Index\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9120928\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN A LA SECRETARÍA DISTRITAL DE RECREACIÓN Y DEPORTES\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN A LA SECRETARÍA DISTRITAL DE RECREACIÓN Y DEPORTES\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8987221\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9130747\",\"nombre_proceso\":\"CONVENIO ARIT IM-169 Y IM-170\",\"descripcion\":\"CONVENIO ARIT IM-169 Y IM-170\",\"entidad\":\"MINISTERIO DE IGUALDAD Y EQUIDAD\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.8998068\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9145065\",\"nombre_proceso\":\"CONTRATACION DIRECTA - CCTV YOPAL CASANARE - SPS\",\"descripcion\":\"CONTRATACION DIRECTA - CCTV YOPAL CASANARE - SPS\",\"entidad\":\"MINISTERIO DEL INTERIOR\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9006742\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9171487\",\"nombre_proceso\":\"FNV-0025-2025\",\"descripcion\":\"FNV-0025-2025\",\"entidad\":\"FONDO NACIONAL DE VIVIENDA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9033424\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9217348\",\"nombre_proceso\":\"CONVENIO ASOMURA\",\"descripcion\":\"CONVENIO ASOMURA\",\"entidad\":\"AGENCIA DE DESARROLLO RURAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9086653\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9224040\",\"nombre_proceso\":\"MEJORAMIENTO DE LA INFRAESTRUCTURA EDUCATIVA\",\"descripcion\":\"MEJORAMIENTO DE LA INFRAESTRUCTURA EDUCATIVA\",\"entidad\":\"GOBERNACION DEL CHOCÓ.\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9084618\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9224503\",\"nombre_proceso\":\"GOBERNACIÓN HUILA\",\"descripcion\":\"GOBERNACIÓN HUILA\",\"entidad\":\"COMPUTADORES PARA EDUCAR\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9084859\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9229871\",\"nombre_proceso\":\"AUNAR ESFUERZOS TECNICOS; ADMINISTRATIVOS Y FINANCIEROS PARA LA EJECUCION DEL PROYECTO DENOMINADO CONSTRUCCIÓN DE UN CENTRO DE ALTO RENDIMIE\",\"descripcion\":\"AUNAR ESFUERZOS TECNICOS; ADMINISTRATIVOS Y FINANCIEROS PARA LA EJECUCION DEL PROYECTO DENOMINADO CONSTRUCCIÓN DE UN CENTRO DE ALTO RENDIMIENTO DEPORTIVO EN LA CIUDAD DE MONTERÍA; \",\"entidad\":\"MINISTERIO DEL DEPORTE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9092192\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9231569\",\"nombre_proceso\":\"AUNAR ESFUERZOS TECNICOS; ADMINISTRATIVOS Y FINANCIEROS PARA LA EJECUCION DEL PROYECTO DENOMINADO  CONSTRUCCIÓN DE LA VILLA OLIMPICA FASE II\",\"descripcion\":\"AUNAR ESFUERZOS TECNICOS; ADMINISTRATIVOS Y FINANCIEROS PARA LA EJECUCION DEL PROYECTO DENOMINADO  CONSTRUCCIÓN DE LA VILLA OLIMPICA FASE II; COLISEO DE PELOTAS MUNICIPIO DE TOLÚ; \",\"entidad\":\"MINISTERIO DEL DEPORTE\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9092190\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9241289\",\"nombre_proceso\":\"CONVENIO DE ASOCIACIÓN\",\"descripcion\":\"CONVENIO DE ASOCIACIÓN\",\"entidad\":\"GOBIERNO DEPARTAMENTAL\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9101774\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9311175\",\"nombre_proceso\":\"Prestación de servicios profesionales de abogado para la representación judicial; extrajudicial y las demás que disponga la ley en favor de \",\"descripcion\":\"Prestación de servicios profesionales de abogado para la representación judicial; extrajudicial y las demás que disponga la ley en favor de los usuarios del servicio de defensoría \",\"entidad\":\"DEFENSORÍA DEL PUEBLO\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9185835\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9449488\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS PROFESIONALES PARA EL ACOMPAÑAMIENTO; ASESORÍA; APOYO Y SOPORTE A LA GESTIÓN; ASÍ COMO A LA ADMINISTRACIÓN DEL DATA \",\"descripcion\":\"PRESTACIÓN DE SERVICIOS PROFESIONALES PARA EL ACOMPAÑAMIENTO; ASESORÍA; APOYO Y SOPORTE A LA GESTIÓN; ASÍ COMO A LA ADMINISTRACIÓN DEL DATA CENTER; SERVICIOS DIGITALES; SISTEMAS DE\",\"entidad\":\"MUNICIPIO DE ITAGUI\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9317035\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9467668\",\"nombre_proceso\":\"DE CERO A SIEMPRE\",\"descripcion\":\"DE CERO A SIEMPRE\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9323821\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9495453\",\"nombre_proceso\":\"CONVENIO DE COOPERACION INTERNACIONAL - COMITATO INTERNAZIONALE\",\"descripcion\":\"CONVENIO DE COOPERACION INTERNACIONAL - COMITATO INTERNAZIONALE\",\"entidad\":\"MINISTERIO DEL INTERIOR\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9347839\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9516203\",\"nombre_proceso\":\"APOYAR A LA SECRETARÍA DE HACIENDA DEL MUNICIPIO DE ITAGÜÍ EN EL PROCESO CONTABLE; DE TESORERÍA; PENSIONAL; DE SEGURIDAD SOCIAL Y ADMINISTRA\",\"descripcion\":\"APOYAR A LA SECRETARÍA DE HACIENDA DEL MUNICIPIO DE ITAGÜÍ EN EL PROCESO CONTABLE; DE TESORERÍA; PENSIONAL; DE SEGURIDAD SOCIAL Y ADMINISTRATIVOS; ORIENTADOS AL SANEAMIENTO FINANCI\",\"entidad\":\"MUNICIPIO DE ITAGUI\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9366959\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9517307\",\"nombre_proceso\":\"PRESTACIÒN DE SERVICIOS PROFESIONALES\",\"descripcion\":\"PRESTACIÒN DE SERVICIOS PROFESIONALES\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9367621\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9890719\",\"nombre_proceso\":\"ARRENDAMIENTO DE BIEN INMUEBLE PARA EL FUNCIONAMIENTO DE LAS DEPENDENCIAS DE LA CONTRALORIA DISTRITAL DE BARRANQUILLA EN EL MARCO DEL CONVEN\",\"descripcion\":\"ARRENDAMIENTO DE BIEN INMUEBLE PARA EL FUNCIONAMIENTO DE LAS DEPENDENCIAS DE LA CONTRALORIA DISTRITAL DE BARRANQUILLA EN EL MARCO DEL CONVENIO INTERADMINISTRATIVO No. CD-48-2025-84\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9739742\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9893329\",\"nombre_proceso\":\"ASOCAITADO- AYATALII\",\"descripcion\":\"ASOCAITADO- AYATALII\",\"entidad\":\"GOBERNACION DEL CHOCÓ.\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9819678\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9897571\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA DESARROLLAR LA ESTRATEGIA DE PROMOCIÓN DE LA SALUD MENTAL MIS EMOCIONES; MI MAYOR RIQUEZA\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA DESARROLLAR LA ESTRATEGIA DE PROMOCIÓN DE LA SALUD MENTAL MIS EMOCIONES; MI MAYOR RIQUEZA EN EL DISTRITO DE BARRANQUILLA\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9757193\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9912169\",\"nombre_proceso\":\"Centros de Vida promoción de la salud\",\"descripcion\":\"Centros de Vida promoción de la salud\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9762336\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9917266\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN A LA SECRETARIA DISTRITAL DE RECREACIÓN Y DEPORTES PARA EL DESARROLLO DE RETOS ORIENTADOS AL A\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN A LA SECRETARIA DISTRITAL DE RECREACIÓN Y DEPORTES PARA EL DESARROLLO DE RETOS ORIENTADOS AL APROVECHAMIENTO DEL TIEMPO LIBRE MEDIANTE\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9767946\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9927170\",\"nombre_proceso\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA EL FORTALECIMIENTO DE ESTRATEGIAS DE PEDAGOGÍA CIUDADANA ORIENTADAS A LA DEMOCRACIA PARTI\",\"descripcion\":\"PRESTACIÓN DE SERVICIOS DE APOYO A LA GESTIÓN PARA EL FORTALECIMIENTO DE ESTRATEGIAS DE PEDAGOGÍA CIUDADANA ORIENTADAS A LA DEMOCRACIA PARTICIPATIVA Y PLURALISTA\",\"entidad\":\"DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9777475\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9958123\",\"nombre_proceso\":\"CONTRATO INTERADMINISTRATIVO\",\"descripcion\":\"CONTRATO INTERADMINISTRATIVO\",\"entidad\":\"FONDO UNICO DE TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9811133\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9963832\",\"nombre_proceso\":\"PRESTACIÓN DEL SERVICIO PARA ADMINISTRAR LA CANASTA EDUCATIVA DE LA POBLACIÓN ÉTNICA DE LOS CENTROS ETNOEDUCATIVOS UTPURAI- UBICADO EN LA ZO\",\"descripcion\":\"PRESTACIÓN DEL SERVICIO PARA ADMINISTRAR LA CANASTA EDUCATIVA DE LA POBLACIÓN ÉTNICA DE LOS CENTROS ETNOEDUCATIVOS UTPURAI- UBICADO EN LA ZONA RURAL ÉTNICA DEL MUNICIPIO DE ALBANIA\",\"entidad\":\"GOBERNACIÓN DE LA GUAJIRA\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9824442\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9991057\",\"nombre_proceso\":\"CANAL REGIONAL DE TELEVISIÓN TEVEANDINA S.A.S. - TEVEANDINA S.A.S.\",\"descripcion\":\"CANAL REGIONAL DE TELEVISIÓN TEVEANDINA S.A.S. - TEVEANDINA S.A.S.\",\"entidad\":\"FONDO UNICO DE TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9843067\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"},{\"id_proceso\":\"CO1.REQ.9993489\",\"nombre_proceso\":\"CORPORACIÓN COLOMBIA CREA TALENTO - COCREA\",\"descripcion\":\"CORPORACIÓN COLOMBIA CREA TALENTO - COCREA\",\"entidad\":\"FONDO UNICO DE TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES\",\"url_secop\":\"https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9844553\",\"fuente\":\"Datos/bandeja_revision_2026.json\",\"tipo_registro\":\"SECOP\"}]")
FUENTE = "Datos/tc2_corpus_2026.json"
BLOB_FUENTE = "4b1ac0dd22137984b7e81864a70e04075af2af39"
ESQUEMA = ["id_proceso","nombre_proceso","descripcion","entidad","url_secop","fuente","tipo_registro"]
assert len(CORPUS)==200 and all(set(r)==set(ESQUEMA) for r in CORPUS)
assert len({r["id_proceso"] for r in CORPUS})==200
HASH_SNAPSHOT = sha(canon(CORPUS))
print("Snapshot",len(CORPUS),"procesos, SHA-256 canónico",HASH_SNAPSHOT)


In [ ]:
df=pd.DataFrame(CORPUS)
assert df.id_proceso.is_unique
display(df[["id_proceso","entidad","nombre_proceso"]].head(5))
conteo=df.groupby("entidad",dropna=False).size()
assert conteo.sum()==len(df)
display(conteo.sort_values(ascending=False).head(10))
DUPLICADOS=5
FALTANTES=1
# La copia se crea antes de alterar entidades; coincide con la presentación.
copia=pd.concat([df.copy(),df.head(DUPLICADOS)],ignore_index=True)
if FALTANTES:
    copia.loc[:FALTANTES-1,"entidad"]=""
assert len(copia)==200+DUPLICADOS and copia.id_proceso.nunique()==200
print({"filas":len(copia),"unicos":copia.id_proceso.nunique(),"repeticiones":int(copia.id_proceso.duplicated().sum()),"entidad_vacia":int(copia.entidad.eq("").sum())})
assert sha(canon(CORPUS))==HASH_SNAPSHOT
def tokens(s):
    norm=unicodedata.normalize("NFD",str(s)).lower()
    return re.findall(r"[a-z0-9]+", "".join(c for c in norm if not "\u0300"<=c<="\u036f"))
termino="salud"
listas=[tokens(r["descripcion"]) for r in CORPUS]
print("Término",termino,"ocurrencias",sum(t.count(termino) for t in listas),"procesos",sum(termino in t for t in listas))

N=80  # modifica a 40 o 120; mismo orden que presentación
assert 1<=N<=len(df)
muestra=df.iloc[:N].copy()
agrupada=muestra.groupby("entidad",dropna=False).size()
assert int(agrupada.sum())==N and muestra.id_proceso.is_unique
print("Selección",N,"entidades",len(agrupada)); display(agrupada.sort_values(ascending=False).head(7))

# Separa los errores de identidad y de agregación del producto válido.
def diagnosticar_tabla(tabla,n_esperado,omitir_grupo=False):
    grupos=tabla.groupby("entidad",dropna=False).size()
    if omitir_grupo: grupos=grupos.drop(index=tabla.iloc[0]["entidad"])
    unicos=int(tabla.id_proceso.nunique()); suma=int(grupos.sum())
    valido=len(tabla)==unicos==suma==n_esperado
    return {"filas":len(tabla),"ids":unicos,"suma":suma,"esperado":n_esperado,"valido":valido,
            "recomendacion":"Conserva el resultado" if valido else "Revisa IDs y grupos; no cambies N esperado"}
pruebas_tabla=[diagnosticar_tabla(muestra,N),diagnosticar_tabla(pd.concat([muestra,muestra.head(1)],ignore_index=True),N),diagnosticar_tabla(muestra,N,True)]
assert [r["valido"] for r in pruebas_tabla]==[True,False,False]
display(pd.DataFrame(pruebas_tabla))


## A4 · HTTP, errores y consulta optativa
Una petición válida no garantiza cobertura. La función clasifica respuestas de ejemplo: 200 requiere validar JSON y esquema; 429/503 pueden requerir reintento acotado; 401/403 exigen revisar acceso; timeout no equivale a vacío. Si se omite una página fallida, el proceso no puede afirmar completitud. Un servidor real puede tener reglas adicionales. En el núcleo no usamos red.

La ruta opcional consulta cinco registros contractuales de la API pública. Se mantiene separada del corpus de procesos y no pretende demostrar paginación del universo. Si falla, se informa el error y no se fabrica una respuesta. No solicita credenciales. Actívala solo si quieres contrastar el servicio; el resto del cuaderno no depende de esta consulta.


In [ ]:
def estado_http(status):
    if status==200: return "validar cuerpo y esquema"
    if status in (429,503,"timeout"): return "reintento acotado según política"
    if status in (401,403): return "revisar acceso"
    return "registrar incidencia"
for status in (200,429,503,403,"timeout",404):
    print(status,estado_http(status))
assert estado_http("timeout")!="validar cuerpo y esquema"
CONSULTA_VIVA=False
if CONSULTA_VIVA:
    import requests
    url="https://www.datos.gov.co/resource/rpmr-utcd.json"
    params={"$limit":5,"$select":"nombre_de_la_entidad,departamento_entidad,valor_contrato"}
    try:
        r=requests.get(url,params=params,timeout=15)
        r.raise_for_status()
        registros=r.json()
        assert isinstance(registros,list) and len(registros)<=5
        print({"modo":"consulta viva","grano":"registro contractual","fuente":url,"filas":len(registros)})
    except (requests.RequestException,ValueError,AssertionError) as exc:
        print("Consulta viva no comprobada:",type(exc).__name__)
else:
    print("Consulta viva no ejecutada; snapshot disponible.")

# HTTP 200 no basta: comprobar cuerpo y campos en un ejemplo sin red.
def evaluar_respuesta_docente(status,cuerpo):
    if status!=200: return {"confirmar":False,"accion":estado_http(status)}
    try: filas=json.loads(cuerpo)
    except (ValueError,TypeError): return {"confirmar":False,"accion":"aislar cuerpo: JSON inválido"}
    if not isinstance(filas,list) or not all(isinstance(r,dict) and isinstance(r.get("id_proceso"),str) and r["id_proceso"].strip() for r in filas):
        return {"confirmar":False,"accion":"corregir esquema"}
    return {"confirmar":False,"accion":"cuerpo analizable: todavía validar cobertura, contenido y orden"}
for estado,cuerpo in [(200,'[{"id_proceso":"P01"}]'),(200,"<html>"),(200,'[{"otro":1}]'),(429,""),(403,"")]:
    print(estado,evaluar_respuesta_docente(estado,cuerpo))


## A5 · Paginación, orden y cobertura
Offset omite posiciones; limit fija máximo de filas por página. Para un universo congelado de N registros, el salto igual a limit y techo(N/limit) peticiones cubren todas las posiciones. Si el salto es menor, hay solapamiento; si es mayor, huecos. Una lista mutable puede desplazar registros aun con parámetros correctos. Para una fuente viva, el orden total y un mecanismo de versión o reconciliación deben establecerse según el servicio.

**Predice:** cuatro páginas de 50 con salto 40 devuelven 200 filas, pero ¿cuántas identidades? Ejecuta el control. Compara ausencias, extras y repeticiones, no solo conteo. Luego usa salto 50, cambia el tamaño a 37 y calcula automáticamente el número de peticiones. No avances si los invariantes fallan.

**Cómo se lee:** ventanas, IDs únicos y faltantes. **Qué nos dice:** cobertura exacta frente a este universo congelado. **Qué no permite concluir:** cobertura de un portal que cambia. **Error común:** usar HTTP 200 o número bruto de filas como prueba de adquisición completa.

En este laboratorio el orden total es la posición en el snapshot versionado; no reordenamos los registros antes de seleccionar un prefijo. En una consulta viva usa un orden total según el contrato de esa fuente, pero no confundas el orden con una instantánea.


In [ ]:
def adquirir(base,limite,salto,paginas):
    assert limite>0 and salto>0 and paginas>0
    esperado={r["id_proceso"] for r in base}
    assert len(esperado)==len(base)
    filas=[]; traza=[]
    for p in range(paginas):
        offset=p*salto; lote=base[offset:offset+limite]
        filas.extend(lote)
        traza.append({"pagina":p+1,"offset":offset,"filas":len(lote)})
    c=Counter(r["id_proceso"] for r in filas)
    control={"filas":len(filas),"unicos":len(c),"duplicados":sum(v-1 for v in c.values()),
             "ausentes":sorted(esperado-set(c)),"extras":sorted(set(c)-esperado)}
    return filas,control,traza
base=list(CORPUS)  # orden estable del snapshot versionado, igual al explorador
_,mal,_=adquirir(base,50,40,4)
assert mal["filas"]==200 and mal["unicos"]==170 and mal["duplicados"]==len(mal["ausentes"])==30
print("Salto incorrecto", {k:(len(v) if isinstance(v,list) else v) for k,v in mal.items()})
LIMIT=37
raw,control,traza=adquirir(base,LIMIT,LIMIT,math.ceil(len(base)/LIMIT))
assert control["unicos"]==200 and control["duplicados"]==0 and not control["ausentes"] and not control["extras"]
display(pd.DataFrame(traza))
print("Cobertura correcta",control)

# Contraejemplo de fuente mutable (no SECOP en vivo).
original=[f"P{i:02d}" for i in range(1,9)]
p1=original[:4]
p2=["P00",*original][4:8]
assert len(set(p1+p2))==7 and set(original)-set(p1+p2)=={"P08"}
print("Inserción entre páginas:",p1,p2,"ausente P08; repetido P04")


## A6 · Fragmentación, reducción y medición
Leer por bloques mantiene un estado parcial. Para conteos por entidad, acumula un Counter; el tamaño de bloque cambia la cantidad de llamadas, no el resultado si el universo y la operación son iguales. Para media, combina suma y conteo de valores válidos. Una media de medias da peso a los grupos, no a las observaciones. La mediana exacta no se obtiene así.

El benchmark genera un CSV sintético de dos columnas. Para la misma N, mide lectura completa y fragmentada en tres configuraciones. El archivo y el DataFrame ocupan cantidades diferentes. memory_usage(deep=True) estima tamaño lógico de columnas e índice, no pico RSS. perf_counter mide tiempo transcurrido, no tiempo de CPU. No concluyas que una estrategia es universalmente más rápida por una sola ejecución; la caché y el entorno pueden cambiar tiempos. No se ejecutan cargas grandes por defecto.

**Predice:** ¿qué ocurre al cambiar de 1000 a 5000 filas por bloque? ¿Debe cambiar la suma? Registra resultado y máxima memoria lógica de un bloque. Las pruebas de igualdad deben pasar antes de comparar costo.


In [ ]:
def contar_bloques(filas,chunk):
    assert chunk>0
    out=Counter(); tamanos=[]
    for i in range(0,len(filas),chunk):
        lote=filas[i:i+chunk]; tamanos.append(len(lote))
        out.update(r["entidad"] for r in lote)
    return out,tamanos
directo=Counter(r["entidad"] for r in raw)
for b in (7,25,50):
    parcial,tamanos=contar_bloques(raw,b)
    assert parcial==directo and sum(tamanos)==len(raw)
    print("bloque",b,"particiones",len(tamanos),"paridad OK")
a=[10,10,None]; b=[100,100,100,100]
estados=[(sum(x for x in z if x is not None),sum(x is not None for x in z)) for z in (a,b)]
media=sum(x[0] for x in estados)/sum(x[1] for x in estados)
assert media==70
print("Estados",estados,"media global",media,"media de medias",55)


### A6 · Interpretar un fallo y repararlo
El simulador de la diapositiva 13 muestra el estado tras cada bloque. Aquí se prueba el mismo mecanismo con los datos del snapshot. Primero predice qué pasará si se reinicia el acumulador, se duplica el primer bloque o se omite el último. Luego ejecuta el escenario. Los fallos deliberados no detienen la sesión: un control devuelve el diagnóstico y una acción correctiva. El producto principal nunca usa el resultado defectuoso. No alteres el valor esperado para ocultar el fallo; repara el mecanismo y verifica otra vez. Si tienes un único bloque, reiniciar el estado antes de procesarlo no necesariamente produce un error observable; una prueba debe incluir múltiples bloques.


In [ ]:
def diagnosticar_reduccion(filas, tam, fallo="ninguno"):
    if tam<=0 or not filas:
        raise ValueError("Usa filas no vacías y un bloque positivo")
    if fallo not in ("ninguno","omitir último bloque","reiniciar acumulador","duplicar primer bloque"):
        raise ValueError("Escenario desconocido")
    bloques=[filas[i:i+tam] for i in range(0,len(filas),tam)]
    estado=Counter()
    for i,lote in enumerate(bloques):
        if fallo=="omitir último bloque" and i==len(bloques)-1:
            continue
        if fallo=="reiniciar acumulador":
            estado.clear()
        estado.update(x["entidad"] for x in lote)
        if fallo=="duplicar primer bloque" and i==0:
            estado.update(x["entidad"] for x in lote)
    esperado=Counter(x["entidad"] for x in filas)
    correcto=estado==esperado
    acciones={"omitir último bloque":"Procesa también la última ventana parcial.",
              "reiniciar acumulador":"Inicializa el estado una sola vez, antes del bucle.",
              "duplicar primer bloque":"Procesa cada ventana exactamente una vez.",
              "ninguno":"Conserva el estado y verifica cobertura."}
    return {"escenario":fallo,"esperado":len(filas),"observado":sum(estado.values()),
            "paridad":correcto,"recomendacion":acciones[fallo] if not correcto else "Control superado; verifica también identidad."}

diagnosticos=[diagnosticar_reduccion(CORPUS,25,f) for f in
              ("ninguno","omitir último bloque","reiniciar acumulador","duplicar primer bloque")]
display(pd.DataFrame(diagnosticos))
assert [x["paridad"] for x in diagnosticos]==[True,False,False,False]
assert diagnosticar_reduccion(CORPUS,37)["paridad"]


In [ ]:
def medir_csv(n,chunk_sizes):
    # Misma operación y mismo alcance temporal para todas las estrategias.
    assert 1<=n<=1_000_000 and all(0<c<=n for c in chunk_sizes)
    ciclos,resto=divmod(n,101)
    referencia=(n,ciclos*5050+resto*(resto-1)//2)
    resultados=[]
    with tempfile.TemporaryDirectory(prefix="s01_medicion_") as tmp:
        p=Path(tmp)/"sintetico.csv"
        with p.open("w",newline="",encoding="utf-8") as f:
            w=csv.writer(f);w.writerow(["id","valor"])
            w.writerows((i,i%101) for i in range(n))
        for c in [None,*chunk_sizes]:
            inicio=time.perf_counter()
            lector=[pd.read_csv(p)] if c is None else pd.read_csv(p,chunksize=c)
            cnt=total=mem=partes=0
            for tabla in lector:
                cnt+=len(tabla);total+=int(tabla.valor.sum());partes+=1
                mem=max(mem,int(tabla.memory_usage(deep=True).sum()))
            segundos=time.perf_counter()-inicio
            assert (cnt,total)==referencia
            assert partes==(1 if c is None else (n+c-1)//c)
            resultados.append({"N":n,"chunk":"completo" if c is None else c,
                               "filas":cnt,"suma":total,"bloques":partes,"bytes_logicos":mem,
                               "bytes_csv":p.stat().st_size,"segundos_incluye_instrumentacion":segundos})
            del tabla,lector
    return resultados
N_BENCH=10_000
mediciones=medir_csv(N_BENCH,[1000,5000])
display(pd.DataFrame(mediciones))
assert len({r["suma"] for r in mediciones})==1
print("Igualdad con referencia matemática. Tiempo incluye lectura, agregación e instrumentación; no pico RSS.")


### A6.2 · Escala controlada: mismo problema, diferente N
La primera prueba enseña equivalencia en 10.000 filas. La segunda amplía a 100.000 sin cambiar la regla de generación. La extensión de un millón es voluntaria y manual; no provoca agotamiento de memoria. Predice qué cambia: filas, bytes lógicos, tiempo. La suma esperada se calcula independientemente mediante ciclos de 101. Se conserva un registro de métricas para el producto. La memoria de columnas e índice no equivale al pico de RAM del proceso. Las rutas se ejecutan una sola vez para limitar tiempo; no concluyas superioridad universal. Para comparar tiempos rigurosamente alterna el orden y repite como en el apéndice. El CSV sintético nunca se mezcla con SECOP real.


In [ ]:
AMPLIAR_A_MILLON=False
escalas=[100_000]+([1_000_000] if AMPLIAR_A_MILLON else [])
metricas_s01=list(mediciones)
for n_escala in escalas:
    metricas_s01.extend(medir_csv(n_escala,[1000,5000]))
tabla_metricas=pd.DataFrame(metricas_s01)
display(tabla_metricas)
assert all(g.suma.nunique()==1 and g.filas.nunique()==1 for _,g in tabla_metricas.groupby("N"))


## A7 · Producto RAW, derivado y manifiesto
RAW preserva los registros adquiridos sin alterar campos; JSONL guarda un objeto por línea. Un derivado es el conteo por entidad. El manifiesto conserva fuente, versión de blob, hash canónico del snapshot, grano, esquema, parámetros, traza, controles y SHA-256 de archivos. La carpeta nueva se crea con tempfile.mkdtemp y no se elimina al cerrar la celda; dura en el runtime. En Colab puedes descargarla desde el explorador de archivos. No se escribe en el repositorio.

Una huella verifica bytes, no veracidad ni autoría. Por eso se comprueba también contenido por ID, ausencia de duplicados y equivalencia de conteos al releer. Una modificación de un campo conserva identidad pero cambia bytes y contenido. Una inversión de orden puede cambiar bytes sin alterar contenido por ID. No se guarda información de credenciales.

**Producto mínimo:** carpeta con snapshot.json, raw.jsonl, conteos.json y manifest.json; relectura satisfactoria. La fuente completa se conserva separada de la adquisición seleccionada. El manifiesto indica si el universo elegido es un prefijo del snapshot, no lo llama cobertura nacional.


In [ ]:
def producir(base,limite,chunk,universo="snapshot completo"):
    filas,ctrl,tr=adquirir(base,limite,limite,math.ceil(len(base)/limite))
    assert ctrl["unicos"]==len(base) and not ctrl["duplicados"] and not ctrl["ausentes"] and not ctrl["extras"]
    agg,partes=contar_bloques(filas,chunk)
    assert agg==Counter(r["entidad"] for r in base)
    destino=Path(tempfile.mkdtemp(prefix="s01_producto_"))
    (destino/"snapshot.json").write_bytes(canon(CORPUS))
    (destino/"raw.jsonl").write_bytes(b"".join(canon(r)+b"\n" for r in filas))
    (destino/"conteos.json").write_bytes(canon(dict(agg)))
    man={"schema":"s01-producto-v1","fuente":FUENTE,"blob_fuente":BLOB_FUENTE,"hash_snapshot":HASH_SNAPSHOT,
         "grano":"proceso","orden":"posición en snapshot versionado","universo":universo,"total_snapshot":len(CORPUS),"esquema":ESQUEMA,
         "limit":limite,"chunk":chunk,"traza":tr,"bloques":partes,"control":ctrl,
         "archivos":{p.name:sha(p.read_bytes()) for p in destino.iterdir()}}
    (destino/"manifest.json").write_bytes(canon(man))
    leido=json.loads((destino/"manifest.json").read_bytes())
    assert all(sha((destino/k).read_bytes())==v for k,v in leido["archivos"].items())
    releido=[json.loads(l) for l in (destino/"raw.jsonl").read_bytes().splitlines()]
    assert releido==filas and len({r["id_proceso"] for r in releido})==len(base)
    assert json.loads((destino/"conteos.json").read_bytes())==dict(Counter(r["entidad"] for r in releido))
    assert sha((destino/"snapshot.json").read_bytes())==HASH_SNAPSHOT
    return destino,man
destino,man=producir(base,37,25)
print("Paquete comprobado:",destino)
print({k:man[k] for k in ("grano","universo","control","archivos")})


In [ ]:
import shutil
def verificar_paquete_s01(carpeta,huella_esperada=None):
    carpeta=Path(carpeta)
    def exigir(ok,msg):
        if not ok: raise ValueError(msg)
    def leer(nombre): return json.loads((carpeta/nombre).read_bytes())
    m=leer("manifest.json")
    exigir(m.get("schema")=="s01-producto-v1" and m.get("grano")=="proceso","contrato de producto")
    exigir(m.get("orden")=="posición en snapshot versionado","contrato de orden")
    exigir(m.get("esquema")==ESQUEMA,"esquema declarado distinto del contrato")
    exigir(m.get("fuente")==FUENTE and m.get("blob_fuente")==BLOB_FUENTE,"procedencia declarada")
    exigir(set(m["archivos"])=={"snapshot.json","raw.jsonl","conteos.json"},"lista de archivos")
    for nombre,huella in m["archivos"].items():
        p=carpeta/nombre
        exigir(p.is_file() and sha_archivo(p)==huella,"archivo o huella: "+nombre)
    exigir(sha_archivo(carpeta/"snapshot.json")==m["hash_snapshot"],"hash de snapshot")
    if huella_esperada is not None:
        exigir(m["hash_snapshot"]==huella_esperada,"fuente externa no coincide")
    snapshot=leer("snapshot.json")
    exigir(isinstance(snapshot,list) and len(snapshot)==m["total_snapshot"] and bool(snapshot),"universo fuente")
    exigir(all(isinstance(r,dict) and set(r)==set(m["esquema"]) and all(isinstance(r[k],str) for k in ESQUEMA) and bool(r["id_proceso"].strip()) and bool(r["entidad"].strip()) for r in snapshot),"esquema snapshot")
    exigir(len({r["id_proceso"] for r in snapshot})==len(snapshot),"identidad snapshot")
    filas=[json.loads(l) for l in (carpeta/"raw.jsonl").read_bytes().splitlines()]
    n0=len(filas);N0=len(snapshot)
    exigir(1<=n0<=N0,"cantidad RAW")
    etiqueta=m["universo"]
    exigir(etiqueta==f"prefijo {n0} de {N0}" or (n0==N0 and etiqueta=="snapshot completo"),"etiqueta universo")
    exigir(filas==snapshot[:n0],"orden o contenido RAW")
    ctrl=m["control"]
    exigir(ctrl["filas"]==ctrl["unicos"]==n0 and ctrl["duplicados"]==0 and not ctrl["ausentes"] and not ctrl["extras"],"control de cobertura")
    l,c=m["limit"],m["chunk"]
    exigir(type(l) is int and type(c) is int and l>0 and c>0,"parámetros")
    tr=[{"pagina":i+1,"offset":a,"filas":min(l,n0-a)} for i,a in enumerate(range(0,n0,l))]
    exigir(m["traza"]==tr and m["bloques"]==[min(c,n0-a) for a in range(0,n0,c)],"traza o bloques")
    agregado=pd.DataFrame(filas).groupby("entidad",dropna=False).size().to_dict()
    exigir(leer("conteos.json")==agregado,"derivado no coincide")
    return {"estado":"VERIFICADO","filas":n0,"entidades":len(agregado),"paginas":len(tr),"bloques":len(m["bloques"])}
print("Verificación independiente",verificar_paquete_s01(destino,HASH_SNAPSHOT))


In [ ]:
# Prueba negativa breve: el producto original no cambia.
import shutil
with tempfile.TemporaryDirectory(prefix="s01_rechazo_") as tmp:
    copia_paquete=Path(tmp)/"copia"
    shutil.copytree(destino,copia_paquete)
    f=copia_paquete/"conteos.json"
    f.write_bytes(f.read_bytes()+b" ")
    try:
        verificar_paquete_s01(copia_paquete,HASH_SNAPSHOT)
    except ValueError as exc:
        print("Rechazo esperado:",exc)
    else:
        raise AssertionError("El verificador aceptó una copia alterada")
assert verificar_paquete_s01(destino,HASH_SNAPSHOT)["estado"]=="VERIFICADO"


## A8 · Transferencia y defensa
Construye cuatro configuraciones: base; solo cambia límite; vuelve a base y cambia solo bloque; finalmente cambia N manteniendo ambos parámetros de base. Así separamos decisiones de ejecución de cambios en el universo. El simulador de la diapositiva 17 utiliza la misma regla. Los primeros tres casos deben conservar RAW y conteos; el cuarto modifica el universo.
El código de A8.1 conserva un ejemplo de prueba automática. El ejercicio de construcción y las cuatro comparaciones se realizan en A8.2–A8.3. La recuperación HTTP es una ampliación docente, no una dependencia. La variante autónoma requiere escribir una hipótesis y completar una agregación. No se espera memorizar tiempos.
**Salida para S02:** fuente, grano, cantidad observada, esquema, identidad, reglas de adquisición y restricciones. La tasa de actualización de SECOP y el SLA siguen desconocidos; no se inventan. La arquitectura futura deriva de requisitos, no de marcas.


### Puente de práctica: completa el razonamiento antes de la variante autónoma

Con N=83, limit=20, chunk=12, escribe sin ejecutar: número de páginas, última página, número de bloques y último bloque. Después, en A8.1 cambia los tres parámetros a una combinación propia. No se exige modificar la implementación del servidor. Si necesitas ayuda, pide una sola pista: usa el techo de N dividido por el tamaño; el último grupo es N menos el tamaño de los anteriores. No hay respuestas automáticas aquí; la función del reto contrastará tus propias predicciones. La evidencia formativa es el paquete verificado y una justificación de por qué su RAW no cambia si se modifica solo la ejecución.


In [ ]:
# Activa tras escribir tu hipótesis. Los ceros no son una respuesta ni se evalúan en modo guiado.
ACTIVAR_RETO_AUTONOMO=False
N_AUTONOMO=137
LIMIT_AUTONOMO=29
CHUNK_AUTONOMO=17
HIPOTESIS={"paginas":None,"ultima_pagina":None,"bloques":None,"ultimo_bloque":None}
def reto_s01(n0,l0,c0,hipotesis):
    if not (type(n0) is type(l0) is type(c0) is int and 41<=n0<=199 and 7<=l0<=45 and 6<=c0<=40):
        raise ValueError("Parámetros fuera del reto")
    if set(hipotesis)!={"paginas","ultima_pagina","bloques","ultimo_bloque"} or any(type(x) is not int or x<=0 for x in hipotesis.values()):
        raise ValueError("Completa las cuatro predicciones antes de ejecutar")
    universo=CORPUS[:n0]; etiqueta=f"prefijo {n0} de {len(CORPUS)}"
    p1,m1=producir(universo,l0,c0,etiqueta)
    l2=l0+1 if l0<45 else l0-1;c2=c0+1 if c0<40 else c0-1
    p2,m2=producir(universo,l2,c2,etiqueta)
    v1=verificar_paquete_s01(p1,HASH_SNAPSHOT);v2=verificar_paquete_s01(p2,HASH_SNAPSHOT)
    observado={"paginas":v1["paginas"],"ultima_pagina":m1["traza"][-1]["filas"],
               "bloques":v1["bloques"],"ultimo_bloque":m1["bloques"][-1]}
    resultado=[{"propiedad":k,"hipotesis":hipotesis[k],"observado":v,"coincide":hipotesis[k]==v} for k,v in observado.items()]
    invariancia=all(m1["archivos"][k]==m2["archivos"][k] for k in ("raw.jsonl","conteos.json"))
    assert invariancia and v1["filas"]==v2["filas"]==n0
    return p1,p2,resultado,invariancia
if ACTIVAR_RETO_AUTONOMO:
    p1,p2,retro,invariante=reto_s01(N_AUTONOMO,LIMIT_AUTONOMO,CHUNK_AUTONOMO,HIPOTESIS)
    display(pd.DataFrame(retro))
    print("Paquetes verificados",p1,p2,"RAW y conteos invariantes",invariante)
    print("Escribe la explicación de grano, límite y diferencias en tu bitácora.")
else:
    print("Reto autónomo disponible. Completa HIPOTESIS y activa la celda; no se registra acierto automático.")



### A8.2 · Construcción con ayuda decreciente
Antes de utilizar una función completa, observa este modelo: si los bloques son [2,3] y [4], el acumulador empieza en cero, recibe 5 y finalmente 9. Para agrupar por entidad, el estado es un diccionario de conteos. El algoritmo debe recorrer los registros de cada bloque, obtener la clave de entidad y aumentar exactamente una unidad. No conserves la lista de bloques en el estado. 
Completa la línea marcada dentro de la función siguiente. La celda permanece ejecutable en modo guiado; para acreditar construcción debes activar el reto, cambiar los parámetros y escribir la línea faltante. No reemplaces la comparación con pandas ni edites los resultados esperados. La comparación independiente verifica que no se pierdan ni se inventen filas. Si la prueba falla, examina un bloque pequeño y el estado antes y después.

Para adquisición, completa además la expresión del offset en paginar_estudiante. El control comprueba tanto las páginas individuales como el orden completo. Usa N=137 y límite 29 como caso de ejemplo; después verifica tu combinación.


In [ ]:
def conteo_estudiante(filas, tam):
    acumulado={}
    for inicio in range(0,len(filas),tam):
        for registro in filas[inicio:inicio+tam]:
            clave=registro["entidad"]
            # TODO ESTUDIANTE: actualiza acumulado[clave] sin perder el valor anterior.
            pass
    return acumulado

def validar_construccion(filas,tam,funcion):
    esperado=pd.DataFrame(filas).groupby("entidad",dropna=False).size().to_dict()
    obtenido=funcion(filas,tam)
    if obtenido!=esperado:
        raise ValueError("La reducción no coincide con el conteo independiente")
    return {"entidades":len(obtenido),"filas":sum(obtenido.values())}

def paginar_estudiante(filas,limite):
    paginas=[]
    for i in range(math.ceil(len(filas)/limite)):
        offset=None  # TODO ESTUDIANTE: calcula el desplazamiento de la página i.
        if offset is None:
            return []
        paginas.append(filas[offset:offset+limite])
    return paginas

def validar_paginacion_estudiante(filas,limite,funcion):
    paginas=funcion(filas,limite)
    esperado=[min(limite,len(filas)-i) for i in range(0,len(filas),limite)]
    if [len(p) for p in paginas]!=esperado or [r for p in paginas for r in p]!=filas:
        raise ValueError("Paginación incorrecta: revisa offset, última página y orden")
    return {"paginas":len(paginas),"filas":len(filas)}

if ACTIVAR_RETO_AUTONOMO:
    evidencia_construccion=validar_construccion(CORPUS[:N_AUTONOMO],CHUNK_AUTONOMO,conteo_estudiante)
    evidencia_paginacion=validar_paginacion_estudiante(CORPUS[:N_AUTONOMO],LIMIT_AUTONOMO,paginar_estudiante)
    print("Construcción comprobada",evidencia_construccion,evidencia_paginacion)
else:
    print("Construcción pendiente: activa el reto y completa el TODO.")


### A8.3 · Aislar decisiones y transferir
Registra cuatro hipótesis: número de páginas, última página, número de bloques y último bloque en el caso base. El simulador puede exportar un plan JSON para contrastar parámetros; no contiene el producto Python. Las cuatro configuraciones siguientes cambian una variable a la vez. Los hashes de RAW y conteos de las primeras tres deben ser iguales. El cuarto cambia N: la cobertura sigue siendo exacta dentro del nuevo prefijo, pero el RAW es distinto. No confundas esta evidencia con representatividad de la contratación nacional.

La presentación permite descargar s01-plan.json. Si quieres contrastarlo con Python, copia ese archivo al entorno de Colab, indica la ruta en PLAN_PRESENTACION y vuelve a ejecutar A8.3. Se comparan las cuatro configuraciones, identidades ordenadas y conteos; el archivo JSON es una planificación, no una firma ni una entrega autónoma.


In [ ]:
def comparar_decisiones(n,l,c):
    l2=l+1 if l<45 else l-1
    c2=c+1 if c<40 else c-1
    n2=n+1 if n<199 else n-1
    configs=[("base",n,l,c),("solo limite",n,l2,c),("solo bloque",n,l,c2),("solo universo",n2,l,c)]
    salidas=[]
    for caso,N,L,C in configs:
        p,m=producir(CORPUS[:N],L,C,f"prefijo {N} de {len(CORPUS)}")
        v=verificar_paquete_s01(p,HASH_SNAPSHOT)
        salidas.append({"caso":caso,"N":N,"L":L,"C":C,"paginas":v["paginas"],"ultima_pagina":m["traza"][-1]["filas"],"bloques":v["bloques"],"ultimo_bloque":m["bloques"][-1],"hash_raw":m["archivos"]["raw.jsonl"],"hash_conteos":m["archivos"]["conteos.json"],"ruta":str(p)})
    assert len({r["hash_raw"] for r in salidas[:3]})==1
    assert len({r["hash_conteos"] for r in salidas[:3]})==1
    assert salidas[3]["hash_raw"]!=salidas[0]["hash_raw"]
    return salidas
if ACTIVAR_RETO_AUTONOMO:
    decisiones_s01=comparar_decisiones(N_AUTONOMO,LIMIT_AUTONOMO,CHUNK_AUTONOMO)
    display(pd.DataFrame(decisiones_s01).drop(columns=["ruta","hash_raw","hash_conteos"]))
else:
    decisiones_s01=[]
    print("Comparación independiente pendiente de activar el reto.")

PLAN_PRESENTACION=""  # opcional: ruta local al JSON exportado en la diapositiva 17
def contrastar_plan_s01(plan):
    if not isinstance(plan,dict) or plan.get("schema")!="s01-plan-v1" or not isinstance(plan.get("cases"),list) or len(plan["cases"])!=4:
        raise ValueError("Plan de presentación no válido")
    casos=plan["cases"]
    if not all(isinstance(r,dict) and all(type(r.get(k)) is int for k in ("N","L","C")) for r in casos):
        raise ValueError("Parámetros del plan no válidos")
    n,l,c=(casos[0][k] for k in ("N","L","C"))
    if not (41<=n<=199 and 7<=l<=45 and 6<=c<=40):
        raise ValueError("Parámetros fuera de la ruta autónoma")
    nl=l+1 if l<45 else l-1; nc=c+1 if c<40 else c-1; nn=n+1 if n<199 else n-1
    configs=[(n,l,c),(n,nl,c),(n,l,nc),(nn,l,c)]
    for r,(N,L,C) in zip(casos,configs):
        esperado=CORPUS[:N]
        if (r["N"],r["L"],r["C"])!=(N,L,C): raise ValueError("Configuración no coincide")
        if r.get("ids")!=[x["id_proceso"] for x in esperado]: raise ValueError("Identidad u orden distintos")
        pares=r.get("counts")
        if not isinstance(pares,list) or not all(isinstance(p,list) and len(p)==2 and isinstance(p[0],str) and type(p[1]) is int for p in pares):
            raise ValueError("Conteos no válidos")
        if len(dict(pares))!=len(pares) or dict(pares)!=dict(Counter(x["entidad"] for x in esperado)):
            raise ValueError("Conteos distintos")
        if (r.get("pages"),r.get("lastPage"),r.get("blocks"),r.get("lastBlock"))!=(math.ceil(N/L),N%L or L,math.ceil(N/C),N%C or C):
            raise ValueError("Páginas o bloques distintos")
    return {"estado":"PLAN_COMPATIBLE","casos":4,"universo_base":n}
if PLAN_PRESENTACION:
    p=Path(PLAN_PRESENTACION)
    if p.stat().st_size>2_000_000: raise ValueError("Plan demasiado grande")
    print(contrastar_plan_s01(json.loads(p.read_text(encoding="utf-8"))))
else:
    print("Plan externo no importado: contraste opcional.")


### Entrega verificable (sin depender de la memoria del kernel)
La celda de exportación crea un ZIP únicamente a partir de un paquete que ya pasó el verificador. Antes de comprimir verifica fuente, esquema, orden, identidades, conteos y huellas. La carpeta original permanece intacta. En el explorador de archivos de Colab puedes localizar el ZIP por la ruta impresa; no se envía a un servicio externo. A8.1 genera dos paquetes de referencia. A8.3 genera cuatro casos aislados y A8.4 reúne estos últimos en S01-entrega.zip. Si aún no resolviste el reto, la entrega guiada sirve para practicar el mecanismo pero no acredita autonomía. El manifiesto sigue siendo la evidencia del producto; la bitácora explica hipótesis y límites. Cada ZIP interno contiene cuatro archivos de datos y metadatos; el ZIP integral añade métricas, bitácora, decisiones y validación, sin credenciales ni código ejecutable. La verificación del ZIP comprueba nombres y bytes contra los archivos autorizados, no certifica autenticidad de una fuente externa.


In [ ]:
import zipfile
def empaquetar_s01(carpeta,huella_esperada):
    carpeta=Path(carpeta)
    verificar_paquete_s01(carpeta,huella_esperada)
    salida=carpeta.parent/(carpeta.name+".zip")
    nombres=("snapshot.json","raw.jsonl","conteos.json","manifest.json")
    with zipfile.ZipFile(salida,"w",compression=zipfile.ZIP_DEFLATED) as z:
        for nombre in nombres: z.write(carpeta/nombre,arcname=nombre)
    with zipfile.ZipFile(salida) as z:
        assert set(z.namelist())==set(nombres) and z.testzip() is None
        assert all(sha(z.read(nombre))==sha_archivo(carpeta/nombre) for nombre in nombres)
    return salida
if ACTIVAR_RETO_AUTONOMO:
    aciertos=sum(r["coincide"] for r in retro)
    print("Paquetes válidos del reto:",empaquetar_s01(p1,HASH_SNAPSHOT),empaquetar_s01(p2,HASH_SNAPSHOT))
    print(f"Predicciones comprobadas: {aciertos}/4. La defensa conceptual se revisa aparte.")
    if aciertos!=4: print("Revisa las hipótesis incorrectas; la validez técnica del paquete no acredita dominio de las cuatro predicciones.")
else:
    print("Producto guiado (no acredita autonomía):",empaquetar_s01(destino,HASH_SNAPSHOT))


### A8.4 · Evidencia integral
El ZIP de datos original permanece compatible con su verificador. Un ZIP de entrega separado reúne ese paquete, métricas, validación, decisiones y bitácora. No registra autonomía por el mero hecho de ejecutarse. En modo guiado declara explícitamente que falta construcción y defensa. Antes de entregar escribe tu explicación: hipótesis, efecto de la variable modificada, error detectado y límite de interpretación. El docente evalúa esa explicación, no solo el indicador automático. 


In [ ]:
BITACORA_ESTUDIANTE=""  # escribe tu interpretación antes de entregar
ruta_evidencia=Path(tempfile.mkdtemp(prefix="s01_evidencia_"))/"S01-entrega.zip"
autonomia_tecnica=bool(ACTIVAR_RETO_AUTONOMO and decisiones_s01)
paquetes=[empaquetar_s01(Path(r["ruta"]),HASH_SNAPSHOT) for r in decisiones_s01] if autonomia_tecnica else [empaquetar_s01(destino,HASH_SNAPSHOT)]
validacion={"modo":"autonomo_tecnico" if autonomia_tecnica else "guiado","paquetes":len(paquetes),"defensa_pendiente":not bool(BITACORA_ESTUDIANTE.strip()),"aprendizaje_certificado":False}
if autonomia_tecnica:
    validacion.update({"construccion":evidencia_construccion,"paginacion":evidencia_paginacion,"predicciones":retro,"hipotesis":HIPOTESIS})
with zipfile.ZipFile(ruta_evidencia,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for i,p in enumerate(paquetes):
        z.write(p,arcname=f"datos/caso_{i+1}.zip")
    z.writestr("metricas.csv",tabla_metricas.to_csv(index=False))
    z.writestr("bitacora.md",BITACORA_ESTUDIANTE or "PENDIENTE: hipótesis, interpretación y límite.")
    z.writestr("validacion.json",canon(validacion))
    z.writestr("decisiones.json",canon([{k:v for k,v in r.items() if k!="ruta"} for r in decisiones_s01]))
with zipfile.ZipFile(ruta_evidencia) as z:
    assert z.testzip() is None and {"metricas.csv","bitacora.md","validacion.json","decisiones.json"}<=set(z.namelist())
print("Entrega:",ruta_evidencia,validacion)


### Ticket de salida · defensa y cierre (formativo, no altera la nota oficial)

1. Una extracción devuelve 200 filas, pero 170 IDs distintos de un universo conocido de 200 y ningún extra. ¿Qué dos controles fallaron? Justifica cómo lo sabes.
2. Si el mismo universo cambia de páginas de 50 a páginas de 37, ¿qué propiedad esperas que permanezca y qué evidencia de los archivos usarías para comprobarla?
3. Explica una afirmación que puedes hacer con el conteo de procesos por nombre de entidad y otra que no puedes hacer. Identifica el dato que falta para esta última.

Escribe las respuestas en BITACORA_ESTUDIANTE (A8.4), vuelve a ejecutar la exportación y entrega S01-entrega.zip con el resultado del verificador. No necesitas activar la consulta viva. Para el docente: comprobar cada respuesta por mecanismo y evidencia, no por uso de una palabra clave. Una respuesta incompleta orienta la siguiente práctica; no equivale a una calificación oficial. Registrar de manera agregada cuántas personas completaron el paquete, cuántas eligieron parámetros propios sin recibir la solución y cuántas justificaron correctamente un límite. Si una persona no terminó por un bloqueo técnico, registrar ese motivo por separado de una dificultad conceptual. No inventar resultados de estudiantes.


# Apéndice · profundización y pruebas de ingeniería
Estas celdas complementan el recorrido de aula, no se requieren para producir el primer paquete. Se conserva el desarrollo detallado A0–A8, los experimentos de formatos, cola variable, recuperación HTTP, pruebas negativas y benchmark repetido. Ejecutar en orden desde el núcleo si quieres reproducir todos los ejemplos. El servidor local no necesita credenciales ni conexión externa; si el entorno impide loopback, registra la limitación de esta demostración. Los casos de fallo son inyectados y no describen el rendimiento de SECOP. No se usa el apéndice como lista de actividades obligatorias de 150 minutos.


### A0.1 · Leer el cuaderno como un sistema de trabajo, no como una lista de comandos

Un notebook combina celdas Markdown, celdas de código, un kernel que mantiene el estado de ejecución y salidas que documentan lo observado. El archivo ipynb es una representación del cuaderno, no el proceso Python en ejecución. Abrirlo en Colab no implica que todas las variables ya existan ni que las salidas de una sesión anterior correspondan a la ejecución actual. El kernel se inicia en un ambiente con ciertas bibliotecas, versiones y permisos; ejecutar una celda crea objetos que pueden quedar disponibles para las siguientes. Si modificas una variable y no vuelves a ejecutar las celdas que dependen de ella, puedes observar una mezcla de estados. Para certificar reproducibilidad, comienza con un kernel limpio y ejecuta las celdas en orden. Para experimentar después, cambia un parámetro explícito y ejecuta desde la celda donde se define hasta las que lo utilizan. No borres archivos de ejecuciones previas para resolver un problema de estado.

La primera celda importa bibliotecas de la biblioteca estándar y pandas. import sys permite consultar la versión del intérprete; json convierte objetos Python a una representación JSON y viceversa; csv implementa reglas de lectura y escritura tabular; hashlib proporciona funciones de huella; math ofrece operaciones como ceil; tempfile permite crear directorios temporales con nombres exclusivos; time proporciona relojes para medir duración; re permite patrones de texto; unicodedata permite consultar y normalizar caracteres. Counter, de collections, es un diccionario especializado en conteos. Path representa rutas de archivos de manera explícita y facilita leer o escribir bytes. pandas es la biblioteca tabular que usaremos para inspección y comparación; display es la función de presentación del entorno Jupyter, no una transformación de datos. Si display no muestra una tabla por alguna limitación de interfaz, el objeto puede seguir existiendo; usa print o to_string para inspeccionarlo. Si falta pandas, la ejecución debe reportar ese bloqueo; no se sustituye silenciosamente por resultados preescritos.

Observa las dos funciones cortas de preparación. sha recibe una secuencia de bytes y devuelve hashlib.sha256(b).hexdigest(). El objeto hashlib mantiene un estado de cálculo que representa la secuencia recibida; hexdigest expresa la huella como texto hexadecimal. No se puede pasar directamente cualquier objeto Python porque el algoritmo trabaja con bytes. canon es la operación previa que transforma un objeto serializable a JSON y codifica su texto en UTF-8. ensure_ascii=False preserva caracteres Unicode en la representación textual; sort_keys=True ordena las claves de objetos; separators define delimitadores sin espacios adicionales; allow_nan=False evita producir valores flotantes no representables en JSON estricto. La lista conserva el orden de sus elementos. El objetivo de estas reglas es que dos objetos equivalentes bajo la representación acordada generen los mismos bytes si también conservan orden. No afirmamos que canon sea una implementación general de un estándar internacional de canonicalización; el conjunto de datos de esta sesión contiene campos de texto y la regla es suficiente para el contrato docente.

Una prueba sencilla permite separar contenido y bytes. Construye dos diccionarios con las mismas claves y valores, insertadas en orden distinto. La igualdad de diccionarios en Python puede ser verdadera aunque json.dumps sin ordenar claves produzca cadenas distintas. Con canon esperamos bytes idénticos. Si en cambio creas dos listas con los mismos diccionarios en orden inverso, las listas no son iguales por posición y canon conserva esa diferencia. Este comportamiento será importante cuando se compare una adquisición por páginas con otra: el cambio de límites no debe alterar el orden del snapshot. No utilices el hash como única prueba de identidad, porque un objeto puede tener la misma identidad de negocio con contenido modificado. La huella demuestra una propiedad de representación, no la validez de una decisión.

Los errores del entorno se distinguen de errores del algoritmo. Un ImportError significa que no se pudo importar una biblioteca; un NameError puede ocurrir si una celda previa no se ejecutó; un FileNotFoundError indica que la ruta solicitada no está disponible. Un AssertionError indica que una propiedad que el autor declaró obligatoria no se cumplió. Un error de interpretación puede existir aunque todas las celdas ejecuten sin excepción. Por ejemplo, una tabla correctamente agrupada por nombre de entidad puede interpretarse incorrectamente como registro de gasto. Cuando aparezca una excepción, lee el último mensaje y busca la primera línea de tu código que interviene en ella. Comprueba si faltó una dependencia o si se alteró el universo. Evita introducir una variable vacía o desactivar un assert para llegar al final. El objetivo del error controlado es identificar una condición incumplida.

Para trabajar con archivos, distingue el repositorio del runtime. Un archivo generado en la carpeta temporal del cuaderno existe en ese entorno y no modifica GitHub. Colab puede eliminar sus archivos al cerrar o reiniciar el runtime. Si necesitas conservar evidencia, descarga el paquete autorizado desde el explorador o utiliza almacenamiento permitido por el curso. No introduzcas credenciales en las celdas de demostración. La ruta principal no requiere autenticación a SECOP, MongoDB, Spark ni otros servicios; el snapshot está incorporado. Si se activa la consulta viva, se trata de una actividad independiente y limitada. La API docente de A4.1 escucha solo en loopback y termina cuando concluye el contexto. No requiere exponer puertos a internet.

La bitácora mínima de trabajo registra fecha de ejecución, versión de Python y pandas, referencia del snapshot, parámetros principales y estado de cada control. No es necesario incluir nombre del equipo personal ni información sensible. Si el ambiente cambia, anota el cambio y compara propiedades lógicas antes de comparar rendimiento. Una versión diferente de pandas puede representar índices o inferir tipos de otra manera; eso puede cambiar bytes lógicos sin cambiar suma. Una diferencia de tiempo no es por sí sola un error. Al final de la sesión, otra persona debe poder abrir tu paquete, conocer la operación y comprobar los resultados sin confiar en que el kernel siga abierto. La primera prueba de autonomía consiste precisamente en reconstruir el estado de ejecución sin depender de resultados invisibles de una sesión anterior.

**Control inicial:** ejecuta la siguiente celda y observa tres condiciones: igualdad canónica de diccionarios, diferencia de orden de listas y disponibilidad de directorio temporal escribible. Si alguna falla, conserva el mensaje; no continúes afirmando que el entorno está listo. Esta comprobación es breve y no instala ni descarga nada. 


In [ ]:
d1={"entidad":"Bogotá","id":"P01"}
d2={"id":"P01","entidad":"Bogotá"}
assert d1==d2 and canon(d1)==canon(d2) and sha(canon(d1))==sha(canon(d2))
assert canon([d1,{"id":"P02"}])!=canon([{"id":"P02"},d1])
with tempfile.TemporaryDirectory(prefix="s01_preflight_") as tmp:
    p=Path(tmp)/"prueba.json"
    p.write_bytes(canon(d1))
    assert json.loads(p.read_bytes())==d1
print("Preparación: serialización, orden y escritura/relectura correctos.")


## A1 · Diagnóstico, escala y presupuesto
Volumen mide cantidad; velocidad, ritmo de llegada; variedad, formas de representación; veracidad, calidad; valor, la capacidad de responder una pregunta. Estas dimensiones se relacionan: una fuente grande pero estática puede procesarse por lotes y una fuente pequeña de alta frecuencia puede acumular cola.

**Predice:** si llegan 120 registros/minuto y podemos procesar 80 durante cinco minutos, ¿cuántos quedan pendientes? Si cada uno ocupa 500 bytes, ¿cuánto ocupa la cola bruta? La cola no puede ser negativa. Para memoria, separa bytes de archivo comprimido y bytes mínimos de una matriz numérica. El cálculo no incluye índices, intérprete ni copias.

**Cómo se lee:** compara llegadas, completados y pendientes; cambia una tasa. **Qué nos dice:** la capacidad de servicio limita la salida. **Qué no permite concluir:** rendimiento de una máquina real. **Error común:** confundir registros por segundo con duración de una petición.


In [ ]:
llegada, capacidad, minutos = 120,80,5
recibidos=llegada*minutos
procesados=min(llegada,capacidad)*minutos
pendientes=recibidos-procesados
assert (recibidos,procesados,pendientes)==(600,400,200)
print({"recibidos":recibidos,"procesados":procesados,"pendientes":pendientes,"bytes_cola":pendientes*500})
for n,d in [(100_000,8),(1_000_000,128),(10_000_000,128)]:
    print(n,d,"mínimo GiB",n*d*8/2**30)


### A1.1 · De la aritmética de una cola a una traza que se puede auditar

La celda de escala original calcula un caso de tasa constante. llegada, capacidad y minutos son enteros que representan registros por minuto y duración en minutos. Al multiplicar llegada por minutos se obtiene cantidad recibida. min(llegada,capacidad) elige la menor tasa porque el ejemplo parte de una cola vacía y la tasa es constante. Si llega menos que la capacidad, el servicio no puede producir registros que no existen. La resta produce pendientes y multiplicarlos por quinientos bytes aproxima el tamaño bruto del contenido pendiente. No se incluyen estructuras del sistema de colas, metadatos ni compresión. El bucle for n,d recorre pares de cantidades y dimensiones: en cada iteración calcula el límite inferior de una matriz densa de float64. Dividir por 2**30 expresa GiB. La operación no mide RAM: no se crea la matriz y por ello no se incurre en su costo.

Ahora construiremos una traza de llegada variable, que hace explícita la cola acumulada. Cada intervalo tiene una cantidad de llegadas y una capacidad máxima de servicio. El estado cola inicial indica registros pendientes de intervalos anteriores. Para cada intervalo, disponibles es cola más entradas; salidas es el mínimo entre disponibles y capacidad; cola final es la diferencia. El resultado se almacena en una lista de diccionarios para convertirlo luego a DataFrame. Este modelo tiene intervalos discretos y no simula tiempos individuales ni prioridades; sirve para conservar cantidades. No permite calcular latencia de cada evento sin rastrear su tiempo de llegada y salida. Es una distinción importante: el saldo final de una cola puede ser cero aunque algunos registros hayan esperado varios intervalos. Para analizar latencia se requeriría otra estructura de estado, por ejemplo una cola FIFO con marcas de tiempo.

Observa el control de entrada de la función: se exige igual cantidad de intervalos para llegadas y capacidades. zip empareja elementos de ambas listas por posición, pero por sí solo se detiene cuando termina la lista más corta. Si no comprobáramos longitud, podríamos perder intervalos sin excepción. Se exige tipo entero y valores no negativos para evitar tasas imposibles en este modelo. Una capacidad cero es válida: el sistema no puede procesar y acumula entradas. Una llegada cero también es válida y permite reducir la cola si hay capacidad. El contador de intervalo comienza en uno para presentación; la posición de las listas Python comienza en cero. Esta diferencia de numeración también aparece al trabajar con páginas y offsets. La función conserva tanto cola inicial como final para poder comprobar cada transición de manera independiente.

La ecuación de conservación es cola_inicial + llegadas = salidas + cola_final. Debe cumplirse en cada intervalo y en el total de la traza. Una comprobación adicional exige salidas menores o iguales a capacidad y cola final no negativa. Estas propiedades son más informativas que comparar una sola salida final. Un algoritmo defectuoso que descartara pendientes podría terminar con cola cero aunque haya perdido registros; la ecuación de conservación detectaría el error. Una segunda implementación que calcule los mismos saldos puede fortalecer la prueba, pero en un modelo sencillo basta construir casos con solución manual y verificar invariantes. El caso 120,40,200 con capacidad constante ochenta produce colas 40,0,120; las salidas son 80,80,80. La suma de entradas es 360, salidas 240 y cola 120. En cambio, si el tercer intervalo recibe cuarenta, la cola final queda cero y la capacidad no utilizada en ese intervalo no se transfiere al siguiente.

Este modelo ayuda a identificar el error de calcular capacidad a partir de promedios. Si en tres intervalos llegan 0,0,300 y la capacidad es cien en cada uno, el promedio de llegadas es cien y el promedio de capacidad también. Sin embargo, dos intervalos no tienen trabajo y el último deja doscientos pendientes. Si llegan cien en cada intervalo, el balance total de llegadas es igual, pero no se forma cola. El orden de llegadas afecta el estado y el nivel de servicio. Aumentar el total de capacidad no garantiza resolver una restricción de latencia si la capacidad adicional aparece en momentos en que no hay trabajo. Para una arquitectura real convendría medir distribución de tasas y demoras, no solo totales diarios. Las trazas por intervalos permiten identificar picos y decidir cuándo conviene desacoplar etapas.

La misma ecuación puede aplicarse a una cadena de procesamiento. Supón que adquisición produce archivos y OCR los consume. Si adquisición completa sesenta archivos por minuto y OCR transforma treinta, el buffer intermedio aumenta treinta por minuto mientras se mantengan esas condiciones. Si hay una etapa posterior de indexación con capacidad cien, aumentar esa última capacidad no corrige el atraso de OCR. El cuello de botella depende de las capacidades efectivas y de las relaciones de dependencia. Sin embargo, no todos los archivos requieren el mismo tiempo: PDF de una página y PDF de cien páginas son unidades de trabajo distintas. Para dimensionar mejor se podrían utilizar páginas por minuto o segundos de CPU por documento, según la etapa. En esta sesión la unidad registros por intervalo es suficiente para demostrar conservación, pero una métrica de capacidad debe elegirse de acuerdo con el trabajo real.

**Cómo explorar el resultado:** cambia la lista LLEGADAS y la lista CAPACIDADES conservando su longitud. Predice primero la cola máxima y la cola final. El DataFrame muestra una fila por intervalo, que ahora es el grano de la traza, no un registro SECOP. Si agrupas por intervalo, no estás contando procesos reales. Construye un escenario con capacidad temporalmente cero y otro con un pico de llegada en el último intervalo. Para comparar dos políticas, conserva las mismas llegadas y cambia solo capacidades. Registra salidas, máximo de cola y final; no digas que una política redujo latencia sin medir tiempos individuales. Esta actividad desarrolla el paso de diagnóstico de S01, no reemplaza una simulación de eventos más detallada.


In [ ]:
def simular_cola(llegadas, capacidades):
    if len(llegadas)!=len(capacidades) or not llegadas:
        raise ValueError("Se requieren intervalos emparejados")
    if any(type(x) is not int or x<0 for x in [*llegadas,*capacidades]):
        raise ValueError("Cantidades enteras no negativas")
    cola=0; traza=[]
    for i,(l,c) in enumerate(zip(llegadas,capacidades),1):
        antes=cola
        salida=min(antes+l,c)
        cola=antes+l-salida
        assert antes+l==salida+cola and 0<=salida<=c and cola>=0
        traza.append(dict(intervalo=i,cola_inicial=antes,llegadas=l,capacidad=c,salidas=salida,cola_final=cola))
    assert sum(llegadas)==sum(r["salidas"] for r in traza)+cola
    return traza
LLEGADAS=[120,40,200]
CAPACIDADES=[80,80,80]
traza_cola=simular_cola(LLEGADAS,CAPACIDADES)
display(pd.DataFrame(traza_cola))
assert [r["cola_final"] for r in simular_cola([120,40,200],[80,80,80])]==[40,0,120]
assert simular_cola([0,0,300],[100,100,100])[-1]["cola_final"]==200
assert simular_cola([100,100,100],[100,100,100])[-1]["cola_final"]==0
print("Cola máxima",max(r["cola_final"] for r in traza_cola),"cola final",traza_cola[-1]["cola_final"])


## A2 · Grano y representación
Una fila de procesos no equivale a una fila de contratos. Cada contrato referencia un proceso; un proceso puede tener varios contratos. El siguiente ejemplo es ficticio y no se mezcla con SECOP. Un left join conserva procesos sin coincidencia, mientras un inner join los descarta. Si la dimensión repite identidades, la unión puede multiplicar filas.

**Predice:** P1 tiene dos contratos, P2 uno y P3 ninguno. ¿Cuántos procesos distintos hay tras la unión izquierda? ¿Cuántas filas resultan? La salida contiene cuatro filas y tres procesos, no cuatro procesos. Comprueba la identidad contractual no nula antes de sumar valores.

En CSV, un registro aparece en una fila y las columnas tienen delimitadores; JSON permite anidar listas. Serializar no es validar. Un archivo puede leerse correctamente pero representar el grano equivocado. En el producto RAW de esta sesión conservaremos procesos, no asignaremos dinero a ellos.

La siguiente extensión de formatos serializa dos observaciones ficticias con comas, comillas y caracteres acentuados. Compara bytes y tipos recuperados. CSV no impone tipos: un número leído como texto requiere un contrato. JSON preserva tipos básicos, XML utiliza texto y etiquetas; ninguno garantiza significado.


In [ ]:
procesos=pd.DataFrame({"id_proceso":["P1","P2","P3"]})
contratos=pd.DataFrame({"id_contrato":["C1","C2","C3"],"id_proceso":["P1","P1","P2"]})
izq=procesos.merge(contratos,on="id_proceso",how="left",validate="one_to_many")
int_=procesos.merge(contratos,on="id_proceso",how="inner",validate="one_to_many")
assert len(izq)==4 and izq.id_proceso.nunique()==3 and izq.id_contrato.count()==3
assert len(int_)==3 and int_.id_proceso.nunique()==2
display(izq)
print("Inner:",len(int_),"filas,",int_.id_proceso.nunique(),"procesos")


In [ ]:
import io, xml.etree.ElementTree as ET
ejemplo=[{"id":"P1","entidad":"Bogotá, D.C.","cantidad":2},
         {"id":"P2","entidad":'Entidad "B"',"cantidad":0}]
buf=io.StringIO()
escritor=csv.DictWriter(buf,fieldnames=list(ejemplo[0]))
escritor.writeheader(); escritor.writerows(ejemplo)
texto_csv=buf.getvalue()
leido_csv=list(csv.DictReader(io.StringIO(texto_csv)))
assert leido_csv[0]["cantidad"]=="2"
texto_json=json.dumps(ejemplo,ensure_ascii=False)
assert json.loads(texto_json)==ejemplo
raiz=ET.Element("procesos")
for registro in ejemplo:
    nodo=ET.SubElement(raiz,"proceso")
    for k,v in registro.items(): ET.SubElement(nodo,k).text=str(v)
texto_xml=ET.tostring(raiz,encoding="utf-8")
assert [x.findtext("id") for x in ET.fromstring(texto_xml)]==["P1","P2"]
for nombre,b in [("CSV",texto_csv.encode()),("JSON",texto_json.encode()),("XML",texto_xml)]:
    print(nombre,len(b),"bytes")
print("CSV recupera cantidad como texto; JSON como número.")


### A2.1 · Diagnosticar pérdidas de representación y cardinalidad antes de transformar

La celda de unión crea dos DataFrames a partir de diccionarios de columnas. Cada lista proporciona valores en el mismo orden, de modo que una posición corresponde a una fila. procesos tiene tres IDs; contratos tiene tres IDs de contrato y una referencia a proceso. merge combina las tablas por id_proceso. El argumento how="left" solicita conservar todas las filas de la izquierda; how="inner" conserva solo coincidencias. validate="one_to_many" exige unicidad de la clave del lado izquierdo, permitiendo multiplicidad de la derecha. No exige que id_contrato sea único porque no es la clave utilizada en el merge. Por ello, si se necesita un conteo contractual correcto, se requiere un control separado de identidad contractual. Este detalle es fundamental: validar una relación por una clave no valida automáticamente otras claves ni campos.

El resultado de la unión izquierda contiene una fila para P3 sin contrato. pandas representa el valor ausente en la columna derecha según sus tipos internos; count sobre id_contrato excluye ausentes y por eso retorna tres. len del DataFrame retorna cuatro porque incluye P3; nunique de id_proceso retorna tres. La aserción conjunta verifica estas tres propiedades. Si se cambia la unión a interna, P3 desaparece y la cantidad de procesos distintos pasa a dos. Observa que el cambio no demuestra que P3 sea inválido, solo que no tiene correspondencia en la tabla de contratos de ejemplo. La cardinalidad del resultado debe interpretarse a partir del objetivo: enriquecer todos los procesos o seleccionar aquellos con observación contractual disponible.

La siguiente celda serializa datos ficticios mediante csv.DictWriter. StringIO actúa como un archivo de texto en memoria, útil para enseñar escritura sin producir un archivo físico. fieldnames fija nombres y orden de columnas. writeheader escribe encabezados; writerows recibe diccionarios y escribe sus campos siguiendo el dialecto del CSV. Cuando un nombre contiene comas o comillas, la biblioteca escapa esos caracteres de forma correcta. getvalue recupera el texto que se produjo. DictReader interpreta el encabezado y genera diccionarios; en este caso los valores se leen como cadenas, incluida cantidad. La aserción que verifica "2" en lugar de 2 enseña que el formato no conserva tipos numéricos de la misma manera que JSON. Si la conversión a número es necesaria, debe aplicarse una regla explícita y revisar errores.

json.dumps transforma la lista de objetos a texto JSON. json.loads vuelve a interpretar ese texto y puede reconstruir cantidad como entero, porque la sintaxis distingue número de cadena. Esto no convierte cualquier JSON válido en dato válido: un usuario puede haber escrito "2" entre comillas cuando el contrato esperaba un número, o puede faltar un campo requerido. La comprobación de tipos sigue siendo necesaria. En XML, ElementTree crea un árbol con un nodo raíz y subelementos de proceso y campo. El texto de los elementos se expresa como cadenas; convertir cantidad a str permite guardarla, pero al volver a leerla hace falta conocer el tipo esperado. ElementTree gestiona escape de caracteres especiales. No se debe concatenar XML manualmente con contenido que puede contener ampersand o signos de menor. Este ejemplo solo trabaja con XML generado localmente, no es una guía de seguridad para documentos externos.

Añadiremos una prueba de conversión problemática que tiene tres atributos: un identificador con ceros iniciales, un número entero y un campo opcional. Un lector automático puede interpretar el identificador como número y perder ceros. En pandas, read_csv permite especificar dtype para controlar esa inferencia; keep_default_na=False evita interpretar automáticamente determinadas cadenas como faltantes en la demostración. En un CSV donde falta un campo opcional, DictWriter escribe normalmente una celda vacía; si otra observación tenía explícitamente una cadena vacía, ambas pueden quedar indistinguibles al releer. En JSON se puede conservar la diferencia mediante presencia de clave. No se afirma que sea imposible representar la distinción en CSV: se puede añadir una columna indicadora o un convenio de codificación, pero eso requiere un contrato. El formato por sí solo no resuelve semántica.

La pérdida de identidad también puede ocurrir antes de una unión. Si una tabla utiliza "0007" y otra 7, una conversión automática puede hacer que coincidan aunque los IDs tengan significado distinto. Por eso el tipo de clave debe acordarse en la fuente. Un nombre legible como entidad tampoco constituye necesariamente una clave institucional única. En el corpus de S01 los IDs de proceso son cadenas alfanuméricas; se conservan exactamente. Al agrupar por entidad se utiliza el nombre registrado, no una tabla maestra institucional. Una unión futura por nombre requeriría revisar equivalencias, ambigüedades y posibles colisiones. En este cuaderno no se normalizan identificadores ni nombres de entidad para hacerlos coincidir artificialmente.

**Prueba de transferencia:** añade a la tabla ficticia una copia exacta de C1 y calcula dos cosas: cuántas filas se obtienen en la unión y cuántos IDs contractuales distintos existen. La relación uno a muchos por proceso puede seguir siendo válida, pero la identidad contractual deja de ser única. El nuevo bloque demuestra este contraejemplo y una prueba explícita que lo detecta. Después cambia la tabla para que un contrato referencie P99, inexistente en procesos; utiliza una unión con indicador para identificar huérfanos, o compara conjuntos de claves foráneas. No conviertas un huérfano en proceso inventado. El ejercicio establece qué validación faltaría antes de sumar montos en una fuente contractual real. Todos los valores de este caso son ficticios y no se añaden al corpus SECOP.


In [ ]:
import io
casos_formato=[{"id":"0007","cantidad":2,"nota":""},{"id":"0008","cantidad":0}]
buf=io.StringIO()
w=csv.DictWriter(buf,fieldnames=["id","cantidad","nota"]);w.writeheader();w.writerows(casos_formato)
csv_prueba=buf.getvalue()
inferido=pd.read_csv(io.StringIO(csv_prueba))
explicito=pd.read_csv(io.StringIO(csv_prueba),dtype={"id":"string"},keep_default_na=False)
assert str(inferido.loc[0,"id"])=="7" and explicito.loc[0,"id"]=="0007"
assert explicito.loc[0,"nota"]==explicito.loc[1,"nota"]==""
json_prueba=json.loads(json.dumps(casos_formato))
assert "nota" in json_prueba[0] and "nota" not in json_prueba[1]
print("ID inferido",inferido.loc[0,"id"],"ID explícito",explicito.loc[0,"id"])
duplicada=pd.concat([contratos,contratos.iloc[[0]]],ignore_index=True)
union_dup=procesos.merge(duplicada,on="id_proceso",how="left",validate="one_to_many")
assert len(union_dup)==5 and union_dup.id_contrato.nunique()==3
assert not duplicada.id_contrato.is_unique
print("Unión válida por clave de proceso, identidad contractual repetida:",len(union_dup),"filas")
huerfana=pd.concat([contratos,pd.DataFrame({"id_contrato":["C99"],"id_proceso":["P99"]})],ignore_index=True)
assert set(huerfana.id_proceso)-set(procesos.id_proceso)=={"P99"}
print("Referencia sin padre:",set(huerfana.id_proceso)-set(procesos.id_proceso))


### A3.1 · Examinar un corpus real y construir controles que expliquen el resultado

La celda de datos contiene una cadena JSON extensa con los doscientos objetos del snapshot. json.loads convierte esa cadena en una lista de diccionarios Python. Cada diccionario tiene siete claves; la lista conserva orden de origen. ESQUEMA especifica los campos esperados, y una comparación de conjuntos de claves comprueba que cada objeto contiene exactamente esos nombres. El control de longitud y de conjunto de IDs demuestra que la versión incorporada tiene doscientos procesos distintos. Estas condiciones son invariantes del snapshot docente, no reglas generales de SECOP. Si un nuevo archivo tuviera 201 observaciones válidas, la prueba tendría que comparar con la nueva referencia autorizada; no se debe eliminar una fila solo para conservar una constante antigua. La referencia de blob y la huella de canon(CORPUS) vinculan la práctica a una versión definida.

pd.DataFrame(CORPUS) construye una tabla con columnas. Los atributos de diccionario pasan a columnas y las observaciones a filas. La vista de cinco filas solo ayuda a inspeccionar. Para un control de esquema hay que revisar nombres y tipos efectivos y no inferirlos de una captura. El dtype object puede representar objetos Python de distinto tipo; que una columna sea object no acredita que cada registro sea cadena. La comparación de tipos elemento a elemento en el contrato de A4 es más específica. La propiedad is_unique verifica que la serie de IDs no repita valores. No debe ser el único control: una serie con un único faltante y el resto distinto podría seguir sin duplicación, pero no tendría identidad válida para todas las filas. En el corpus se verificará además que cada campo obligatorio sea una cadena no vacía.

La variable conteo se crea con groupby y size. groupby forma grupos de acuerdo con el valor de entidad; dropna=False hace explícita la intención de conservar un grupo de claves nulas si aparecieran. size devuelve el tamaño de cada grupo. El resultado es una serie cuyo índice son nombres de entidad y cuyos valores son cantidades. sort_values ordena esa serie por conteo; head selecciona las primeras diez. El top diez no necesariamente suma doscientos; la reconciliación debe calcularse sobre toda la serie antes de recortar. Al comparar dos prefijos, una entidad puede subir o bajar de posición porque cambió el universo, no porque haya cambiado su actividad real. La regla de ordenamiento de empates debe declararse si se espera un ranking idéntico entre herramientas. Para comprobar paridad de conteos, se comparan mapas por clave, no el orden visual de barras.

La siguiente transformación crea una copia mediante concat. Se concatenan el DataFrame y cinco primeras filas, y ignore_index=True genera un índice de posiciones consecutivas nuevo. El índice de pandas no sustituye la clave de negocio id_proceso: tras concat puede ser único aunque se repitan IDs. Al escribir una cadena vacía en entidad de la fila cero, se produce un conflicto en una copia, no en CORPUS. duplicated sobre id_proceso devuelve una máscara booleana que marca apariciones adicionales según su configuración. sum de esa máscara cuenta repeticiones adicionales, no todas las filas que participan en conflictos. Si se necesita localizar todos los miembros de un grupo duplicado, se puede usar keep=False y después revisar contenido. La diferencia entre contar incidencias y localizar grupos afectados es importante para una política de cuarentena.

En el nuevo ejercicio se construyen cinco modificaciones controladas sobre copias independientes: campo ausente, campo vacío, ID vacío, tipo incorrecto y registro duplicado. Cada caso conserva una etiqueta de origen para poder contar razones. Un validador devuelve una lista de motivos, no solamente True o False. Una observación puede incumplir varias reglas y por tanto el número de incidencias puede superar el de registros rechazados. Para distinguir repetición exacta de conflicto, el validador compara bytes canónicos de registros con igual identidad. Si el mismo ID aparece dos veces y contenido cambia, se registra conflicto; si coincide, repetición exacta. Las reglas de calidad son locales a este contrato docente. No se intenta detectar que una entidad está mal escrita mediante una lista desconocida ni validar que una descripción sea cierta. El resultado permite explicar qué condición provocó el rechazo, no prejuzgar la contratación.

La función de tokenización del cuaderno se alinea con el control del navegador: minúsculas, NFD, eliminación del rango de marcas combinantes U+0300 a U+036F y patrón de letras ASCII o dígitos. Esta regla se limita deliberadamente al caso docente. Python y JavaScript no tienen que producir los mismos resultados para todos los alfabetos y transformaciones Unicode arbitrarias; por eso se comprueba el conjunto real y ejemplos definidos. No se usa casefold de Python en una ruta y toLowerCase de JavaScript en la otra, pues podrían diferir en caracteres particulares. Un patrón que acepta \w en Python puede conservar letras Unicode y guiones bajos que el patrón JavaScript original no acepta. Alinear la especificación de tokens ayuda a que los conteos de la presentación y el cuaderno tengan el mismo significado sobre el corpus. La descripción original sigue intacta.

El filtro de término se aplica a la lista de tokens de cada descripción. Para frecuencia total se suma t.count(termino) por documento; para frecuencia documental se suma el booleano termino in t, que en Python representa uno o cero. En un documento con tres repeticiones, el primer cálculo aporta tres y el segundo uno. El nuevo ejercicio calcula coincidencia de frase por ventanas contiguas, además de OR para cualquiera de los términos. La función de frase devuelve False si la consulta está vacía, para evitar que una ventana de longitud cero coincida con todos los documentos sin intención explícita. La función OR usa la intersección de conjuntos; cuando la consulta contiene varios tokens, basta uno. La coincidencia de frase implica coincidencia OR para una consulta no vacía, lo que da una propiedad comprobable. Ninguna de las dos calcula relevancia semántica ni puntuación BM25.

**Actividad de análisis:** selecciona un prefijo y una consulta, calcula la cantidad de procesos con coincidencia por entidad y su proporción dentro de cada entidad. El denominador se obtiene de todos los procesos de la selección; el numerador, de la máscara de coincidencias. Reindexar numeradores con el índice de denominadores permite conservar entidades sin coincidencias como cero. La suma de denominadores debe ser N; cada numerador debe quedar entre cero y su denominador. Compara luego ranking por cantidad con ranking por proporción y explica qué tarea de revisión favorece cada criterio. No inventes valores de gasto ni uses nombres de entidad como claves normalizadas de un registro maestro. El código preserva los IDs de las coincidencias para que el estudiante inspeccione ejemplos. Si una consulta produce cero documentos, se informa cero coincidencias dentro del universo, no ausencia de ese tema en todo SECOP.

Al finalizar A3 registra en la bitácora el universo utilizado, el esquema, el número de IDs distintos, las reglas de tokenización y las incidencias de las pruebas sintéticas. El snapshot no se debe modificar. La próxima etapa utiliza todos los doscientos registros para enseñar adquisición paginada; si en A3 elegiste 80, eso no cambia automáticamente el universo de A5. Cada ejercicio declara su selección y sus controles. Este diseño evita depender de variables que cambian invisiblemente al ejecutar celdas fuera de orden. Para transferir un prefijo propio a una extracción, usa los parámetros explícitos de A8 y conserva su etiqueta de universo.


In [ ]:
def problemas_s01(r):
    razones=[]
    faltan=set(ESQUEMA)-set(r); sobran=set(r)-set(ESQUEMA)
    if faltan: razones.append("campos ausentes: "+",".join(sorted(faltan)))
    if sobran: razones.append("campos extra: "+",".join(sorted(sobran)))
    for k in set(ESQUEMA)&set(r):
        if not isinstance(r[k],str): razones.append("tipo: "+k)
        elif k in ("id_proceso","entidad") and not r[k].strip(): razones.append("vacío: "+k)
    return razones
assert all(not problemas_s01(r) for r in CORPUS)
def evaluar_copias(filas):
    vistos={}; salida=[]
    for r in filas:
        motivos=problemas_s01(r)
        ident=r.get("id_proceso")
        if isinstance(ident,str) and ident.strip():
            if ident in vistos:
                motivos.append("repetición exacta" if canon(r)==vistos[ident] else "conflicto de identidad")
            else: vistos[ident]=canon(r)
        salida.append({"id":ident,"motivos":motivos})
    return salida
original=dict(CORPUS[0]); ausente=dict(CORPUS[1]); ausente.pop("entidad")
vacio={**CORPUS[2],"entidad":" "}
sinid={**CORPUS[3],"id_proceso":""}
tipo={**CORPUS[4],"descripcion":7}
conflicto={**original,"entidad":original["entidad"]+" [copia de prueba]"}
casos_calidad=evaluar_copias([original,dict(original),conflicto,ausente,vacio,sinid,tipo])
assert not casos_calidad[0]["motivos"] and all(x["motivos"] for x in casos_calidad[1:])
assert sha(canon(CORPUS))==HASH_SNAPSHOT
display(pd.DataFrame(casos_calidad))
def frase(doc,q):
    return bool(q) and any(doc[i:i+len(q)]==q for i in range(len(doc)-len(q)+1))
def cualquiera(doc,q): return bool(set(doc)&set(q))
assert frase(tokens("Salud pública"),tokens("salud pública"))
assert not frase(tokens("salud y educación pública"),tokens("salud pública"))
assert not frase(tokens("salud"),[])
N_TEXTO=80
CONSULTA_TEXTO="salud pública"
assert 1<=N_TEXTO<=len(CORPUS)
seleccion_texto=pd.DataFrame(CORPUS[:N_TEXTO])
q=tokens(CONSULTA_TEXTO)
docs=seleccion_texto.descripcion.map(tokens)
mascara_frase=docs.map(lambda t:frase(t,q))
mascara_or=docs.map(lambda t:cualquiera(t,q))
assert not (mascara_frase & ~mascara_or).any()
den=seleccion_texto.groupby("entidad",dropna=False).size()
num=seleccion_texto[mascara_or].groupby("entidad",dropna=False).size().reindex(den.index,fill_value=0)
assert den.sum()==N_TEXTO and num.sum()==mascara_or.sum() and (num<=den).all()
tabla_texto=pd.DataFrame({"procesos":den,"coincidencias":num,"proporcion":num/den})
display(tabla_texto.sort_values(["coincidencias","proporcion"],ascending=False).head(8))
print("Frase:",int(mascara_frase.sum()),"OR:",int(mascara_or.sum()),"IDs de ejemplo:",seleccion_texto.loc[mascara_or,"id_proceso"].head(3).tolist())



### A4.1 · De una clasificación de estados a una adquisición HTTP real

En esta práctica Python ejecuta un servidor HTTP local, en la misma máquina del cuaderno, y un cliente independiente que le solicita páginas. El servidor sirve los 200 registros del snapshot SECOP ya cargado. No se finge una consulta al portal: los datos de origen son reales y congelados, el servicio y los fallos son docentes. No requiere puerto público ni credenciales. El contexto api_docente abre un puerto aleatorio de loopback (127.0.0.1), ejecuta un hilo para atender peticiones y lo cierra incluso si el cliente falla.

**Contrato del servicio:** GET /procesos?offset=0&limit=50 devuelve JSON con una lista de objetos, siete campos de texto y orden fijo. Para este ejercicio se conoce el universo completo. En un servicio real se consultaría la documentación para determinar si existen tamaño total, orden estable, paginación por cursor o revisión del snapshot. No se puede asumir que toda API devuelva exactamente el tamaño pedido.

**Lee el código en cuatro partes.** Primero, serializar convierte un objeto en bytes UTF-8 con orden canónico de claves. sha_archivo calcula una huella incremental, leyendo bloques de 65.536 bytes. Segundo, api_docente recibe una lista y construye un Handler. do_GET extrae y valida offset/limit de la URL, corta la lista y produce el cuerpo de la respuesta. El contador intentos registra cuántas veces se pidió cada offset; para el escenario transitorio, las dos primeras peticiones de la segunda página fallan con 429 y 503, y la tercera se atiende correctamente. En los demás escenarios se conserva un error persistente, un dato inválido o una página incorrecta.

Tercero, adquirir_http representa al cliente. pedir construye la URL con urlencode y utiliza urlopen con tiempo límite de tres segundos y un máximo de tres intentos. No incrementa offset al recibir 429 o 503: reintenta la misma consulta. Un 403 detiene la extracción. Retry-After se interpreta como segundos únicamente en este servicio docente; si excede el presupuesto de dos segundos, se detiene en vez de ignorarlo. En producción pueden existir fechas HTTP, retroceso exponencial, jitter, cuotas y políticas diferentes. La espera inyectada durante las pruebas elimina pausas artificiales; no se debe llevar ese comportamiento sin análisis a una API externa.

Cuarto, la página exitosa se conserva en su archivo exacto de respuesta, y luego se interpreta JSON. La comprobación requiere lista del tamaño esperado, campos correctos, identidad no vacía, entidad no vacía, IDs no repetidos ni inesperados y contenido equivalente al snapshot. Una página que no supera el contrato no se incorpora parcialmente a RAW. La cuarentena registra la primera fila inválida y su motivo. Los otros tipos de error conservan la respuesta o la traza según lo que llegó; no se inventa una fila para un cuerpo que no se pudo interpretar. El estudiante puede ampliar la política para recopilar varias incidencias, pero no debe presentar una página rechazada como exitosa.

**Persistencia:** raw.jsonl.part se escribe fila a fila, no con una lista creciente de resultados. El conjunto vistos y el diccionario esperado sí crecen con el número de identidades; esto no demuestra una memoria estrictamente constante para universos enormes. El ejercicio usa 200 procesos y mantiene una referencia de contenido deliberadamente para verificar exactitud. En una arquitectura de gran escala, los índices de identidad o validaciones externas requieren diseño propio. Cada página validada se agrega al acumulador de conteos por entidad. Se relee el archivo incrementalmente, se verifica orden y contenido contra la referencia y se comprueban conteos. Solo entonces se renombra RAW, se guardan derivados y traza y se escribe manifest.json como marcador de finalización. Un fallo deja incidencia.json y, según el punto, archivos parciales. No hay garantía transaccional de todo el directorio ante apagado o fallo de disco; la regla de consumo es aceptar únicamente un manifiesto completo y sus huellas verificadas.

**Predicción:** con cuatro páginas de cincuenta, ¿cuántas solicitudes ocurrirán si la segunda página falla dos veces y se recupera? ¿Cuántas filas estarán aceptadas si la segunda respuesta es JSON inválido? Escribe tu hipótesis antes de ejecutar. Cambia después el límite a 37 y sitúa un fallo en el segundo offset (37). Explica por qué la cantidad aceptada no es igual al número de solicitudes multiplicado por el límite.


In [ ]:
from contextlib import contextmanager
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from threading import Thread
from urllib.parse import urlsplit, parse_qs, urlencode
from urllib.request import urlopen
from urllib.error import HTTPError, URLError
import json, hashlib, tempfile, os, time
from pathlib import Path
from collections import Counter

CAMPOS = ('id_proceso','nombre_proceso','descripcion','entidad','url_secop','fuente','tipo_registro')
def serializar(obj):
    return json.dumps(obj,ensure_ascii=False,sort_keys=True,separators=(',',':'),allow_nan=False).encode('utf-8')
def sha_archivo(ruta):
    h=hashlib.sha256()
    with ruta.open('rb') as f:
        for b in iter(lambda:f.read(65536),b''): h.update(b)
    return h.hexdigest()

@contextmanager
def api_docente(base, escenario='normal', fallo_offset=50):
    """Servidor HTTP real en loopback; fuente congelada y fallos controlados."""
    if escenario not in {'normal','transitorio','prohibido','json_invalido','esquema','repetido','alterado','corto'}:
        raise ValueError('Escenario desconocido')
    intentos=Counter()
    class Handler(BaseHTTPRequestHandler):
        def log_message(self,*args): pass
        def do_GET(self):
            u=urlsplit(self.path)
            try:
                q=parse_qs(u.query,strict_parsing=True)
                offset=int(q['offset'][0]); limit=int(q['limit'][0])
                if u.path!='/procesos' or offset<0 or not 1<=limit<=100: raise ValueError
            except (KeyError,ValueError):
                self.send_error(400); return
            intentos[offset]+=1
            filas=[dict(x) for x in base[offset:offset+limit]]
            status=200; cuerpo=serializar(filas)
            if offset==fallo_offset:
                if escenario=='transitorio' and intentos[offset]<=2:
                    status=429 if intentos[offset]==1 else 503; cuerpo=b'{}'
                elif escenario=='prohibido': status=403; cuerpo=b'{}'
                elif escenario=='json_invalido': cuerpo=b'no-json'
                elif escenario=='esquema' and filas:
                    filas[0].pop('entidad'); cuerpo=serializar(filas)
                elif escenario=='repetido' and filas:
                    filas[0]=dict(base[0]); cuerpo=serializar(filas)
                elif escenario=='alterado' and filas:
                    filas[0]['entidad']+=' [cambio]'; cuerpo=serializar(filas)
                elif escenario=='corto' and filas:
                    cuerpo=serializar(filas[:-1])
            self.send_response(status)
            self.send_header('Content-Type','application/json; charset=utf-8')
            self.send_header('Content-Length',str(len(cuerpo)))
            if status==429: self.send_header('Retry-After','0')
            self.end_headers(); self.wfile.write(cuerpo)
    server=ThreadingHTTPServer(('127.0.0.1',0),Handler)
    hilo=Thread(target=server.serve_forever,daemon=True); hilo.start()
    try:
        yield f'http://127.0.0.1:{server.server_port}/procesos',intentos
    finally:
        server.shutdown(); server.server_close(); hilo.join(timeout=2)

class AdquisicionIncompleta(RuntimeError): pass

def adquirir_http(url, base, limit, destino, max_intentos=3, esperar=lambda s:time.sleep(s)):
    """Solo para la API docente congelada: contrato, reintentos, archivo por páginas y verificación."""
    if not 1<=limit<=100 or not 1<=max_intentos<=5 or not base:
        raise ValueError('Parámetros fuera del laboratorio')
    destino=Path(destino)
    if destino.exists(): raise FileExistsError(destino)
    destino.mkdir(parents=True)
    (destino/'paginas').mkdir()
    esperado={r['id_proceso']:serializar(r) for r in base}
    if len(esperado)!=len(base): raise ValueError('Fuente sin identidad única')
    vistos=set(); conteos=Counter(); traza=[]; cuarentena=[]
    raw_part=destino/'raw.jsonl.part'
    def pedir(offset):
        for intento in range(1,max_intentos+1):
            consulta=url+'?'+urlencode({'offset':offset,'limit':limit})
            try:
                with urlopen(consulta,timeout=3) as r:
                    cuerpo=r.read(2_000_001)
                    if len(cuerpo)>2_000_000: raise AdquisicionIncompleta('Página supera límite de bytes')
                    traza.append({'offset':offset,'intento':intento,'http':r.status,'bytes':len(cuerpo)})
                    return cuerpo
            except HTTPError as e:
                status=e.code; retry=e.headers.get('Retry-After','0'); e.close()
                traza.append({'offset':offset,'intento':intento,'http':status})
                if status not in (429,503) or intento==max_intentos:
                    raise AdquisicionIncompleta(f'HTTP {status} en offset {offset}') from e
                # Esta API docente usa segundos; no se ignora una espera superior al presupuesto.
                if not retry.isdigit() or int(retry)>2:
                    raise AdquisicionIncompleta('Retry-After fuera del presupuesto docente')
                esperar(int(retry))
            except (URLError,TimeoutError) as e:
                traza.append({'offset':offset,'intento':intento,'http':None,'error':type(e).__name__})
                if intento==max_intentos: raise AdquisicionIncompleta('Transporte agotado') from e
                esperar(0)
    try:
        with raw_part.open('wb') as raw:
            for offset in range(0,len(base),limit):
                cuerpo=pedir(offset)
                (destino/'paginas'/f'{offset:06d}.json').write_bytes(cuerpo)
                try: pagina=json.loads(cuerpo)
                except (ValueError,UnicodeError) as e: raise AdquisicionIncompleta('JSON inválido') from e
                if not isinstance(pagina,list) or len(pagina)!=min(limit,len(base)-offset):
                    raise AdquisicionIncompleta('Tamaño de página no coincide con universo congelado')
                # Validar toda la página antes de agregarla; conservar bytes recibidos incluso si falla.
                locales=set()
                for r in pagina:
                    ident=r.get('id_proceso') if isinstance(r,dict) else None
                    razon=None
                    if not isinstance(r,dict) or set(r)!=set(CAMPOS) or any(not isinstance(r[k],str) for k in CAMPOS): razon='esquema'
                    elif not r['id_proceso'].strip() or not r['entidad'].strip(): razon='campo obligatorio vacío'
                    elif ident in vistos or ident in locales: razon='identidad repetida'
                    elif ident not in esperado: razon='identidad inesperada'
                    elif serializar(r)!=esperado[ident]: razon='contenido distinto del snapshot'
                    if razon:
                        cuarentena.append({'offset':offset,'id':ident,'razon':razon,'registro':r})
                        raise AdquisicionIncompleta(razon)
                    locales.add(ident)
                for r in pagina:
                    raw.write(serializar(r)+b'\n'); vistos.add(r['id_proceso']); conteos[r['entidad']]+=1
        if vistos!=set(esperado): raise AdquisicionIncompleta('Cobertura incompleta')
        # Relectura incremental: verificar orden, contenido y conteos sin cargar el archivo entero.
        verificados=0; cuenta_leida=Counter()
        with raw_part.open('rb') as f:
            for i,linea in enumerate(f):
                r=json.loads(linea)
                if i>=len(base) or serializar(r)!=serializar(base[i]): raise AdquisicionIncompleta('Relectura: orden/contenido')
                verificados+=1; cuenta_leida[r['entidad']]+=1
        if verificados!=len(base) or cuenta_leida!=conteos: raise AdquisicionIncompleta('Relectura: conteo')
        os.replace(raw_part,destino/'raw.jsonl')
        (destino/'conteos.json').write_bytes(serializar(dict(conteos)))
        (destino/'traza.json').write_bytes(serializar(traza))
        archivos={str(p.relative_to(destino)):sha_archivo(p) for p in sorted(destino.rglob('*')) if p.is_file()}
        manifest={'estado':'COMPLETO','grano':'proceso','origen':'servidor HTTP docente; snapshot congelado',
                  'hash_universo':hashlib.sha256(serializar(base)).hexdigest(),'limit':limit,'filas':verificados,
                  'unicos':len(vistos),'paginas':len(range(0,len(base),limit)),'solicitudes':len(traza),'archivos':archivos}
        (destino/'manifest.json.part').write_bytes(serializar(manifest))
        os.replace(destino/'manifest.json.part',destino/'manifest.json')
        assert all(sha_archivo(destino/k)==v for k,v in manifest['archivos'].items())
        return manifest
    except (AdquisicionIncompleta,OSError,ValueError) as e:
        (destino/'incidencia.json').write_bytes(serializar({'estado':'INCOMPLETO','error':str(e),'filas_aceptadas':len(vistos),'traza':traza,'cuarentena':cuarentena}))
        raise

def probar_http(base, carpeta):
    resultados=[]
    for esc in ('normal','transitorio','prohibido','json_invalido','esquema','repetido','alterado','corto'):
        ruta=Path(carpeta)/esc
        with api_docente(base,esc,50) as (url,intentos):
            try:
                man=adquirir_http(url,base,50,ruta,esperar=lambda s:None)
                assert esc in ('normal','transitorio') and man['filas']==len(base)
                assert man['solicitudes']==(4 if esc=='normal' else 6)
                resultados.append({'caso':esc,'estado':man['estado'],'filas':man['filas'],'solicitudes':man['solicitudes']})
            except AdquisicionIncompleta:
                assert esc not in ('normal','transitorio')
                inc=json.loads((ruta/'incidencia.json').read_bytes())
                assert not (ruta/'manifest.json').exists() and inc['filas_aceptadas']==50
                resultados.append({'caso':esc,'estado':inc['estado'],'filas':inc['filas_aceptadas'],'solicitudes':sum(intentos.values()),'error':inc['error']})
            assert not any(k>50 for k in intentos) if esc not in ('normal','transitorio') else True
    return resultados


In [ ]:
# Ejecutar sobre el snapshot SECOP ya incorporado en A3.
carpeta_http=Path(tempfile.mkdtemp(prefix="s01_http_"))
resultados_http=probar_http(CORPUS,carpeta_http)
display(pd.DataFrame(resultados_http))
assert sum(r["estado"]=="COMPLETO" for r in resultados_http)==2
assert sum(r["estado"]=="INCOMPLETO" for r in resultados_http)==6
print("Evidencias HTTP:",carpeta_http)
print("Trazas:",json.loads((carpeta_http/"transitorio"/"traza.json").read_bytes()))


### A4.2 · Interpretar la matriz de pruebas
En el escenario normal se esperan cuatro solicitudes HTTP y 200 registros confirmados. En transitorio se esperan seis solicitudes (la página problemática tiene tres intentos) y el mismo conjunto de 200 registros. Para los seis fallos permanentes o de calidad se espera que se acepten cincuenta registros y no se produzca manifest.json. No son cifras de latencia del portal. La tabla se construye con la respuesta efectiva del servidor local, no con una lista de salidas de ejemplo.

Examina los dos paquetes completos y compara el hash de raw.jsonl: debe coincidir, aunque el hash de traza.json no coincida. Esa diferencia es precisamente la evidencia de que el proceso siguió otro recorrido sin cambiar el producto. En el paquete prohibido abre incidencia.json y observa que no se solicitó un offset superior a 50. En esquema, repetido y alterado abre la cuarentena y revisa el motivo. En JSON inválido abre el archivo de la página para confirmar que el cuerpo se preservó como evidencia y no se incorporó a RAW.

**Pregunta de ingeniería:** ¿qué ocurre si el proceso se interrumpe después de escribir RAW pero antes de crear el manifiesto? El directorio no se considera completo. Esta práctica no implementa reanudación entre procesos: solo reintenta solicitudes y conserva evidencia. Una política de checkpoint duradero debe definir cuándo se confirma el progreso y cómo se detectan resultados parciales. Esa es una extensión posterior, no una garantía demostrada aquí.

**Comprobación autónoma:** en la siguiente celda cambia LIMIT_HTTP, MAX_INTENTOS_HTTP y ESCENARIO_HTTP. Usa transitorio con dos intentos: la extracción debe detenerse. Repite con tres: debe completar. Para probar otro caso, anticipa filas aceptadas y solicitudes. No edites la función para que la aserción pase; si la salida no coincide con la hipótesis, explica el error del razonamiento o del código. Los paquetes quedan en carpetas distintas y no modifican el repositorio.


In [ ]:
LIMIT_HTTP=37
MAX_INTENTOS_HTTP=3
ESCENARIO_HTTP="transitorio"
assert 1<=LIMIT_HTTP<=100 and 1<=MAX_INTENTOS_HTTP<=5
ruta_variante=Path(tempfile.mkdtemp(prefix="s01_variante_"))/"ejecucion"
with api_docente(CORPUS,ESCENARIO_HTTP,LIMIT_HTTP) as (url,intentos):
    try:
        evidencia_http=adquirir_http(url,CORPUS,LIMIT_HTTP,ruta_variante,MAX_INTENTOS_HTTP,esperar=lambda s:None)
        print("COMPLETO",evidencia_http["filas"],"filas,",evidencia_http["solicitudes"],"solicitudes")
        assert ESCENARIO_HTTP=="normal" or (ESCENARIO_HTTP=="transitorio" and MAX_INTENTOS_HTTP>=3)
    except AdquisicionIncompleta:
        incidencia=json.loads((ruta_variante/"incidencia.json").read_bytes())
        assert not (ruta_variante/"manifest.json").exists()
        assert incidencia["filas_aceptadas"]==LIMIT_HTTP
        assert ESCENARIO_HTTP!="normal"
        print("INCOMPLETO",incidencia["error"],"filas aceptadas",incidencia["filas_aceptadas"])
print("Paquete variante:",ruta_variante)


### A4.3 · Depurar el cliente HTTP a partir de la evidencia, no a partir de conjeturas

El servidor docente se inicia con un administrador de contexto. El decorador contextmanager convierte una función generadora en un objeto que puede usarse con with. La parte anterior a yield prepara el servidor y el hilo, y la parte finally solicita cierre incluso cuando el cuerpo del with produce una excepción. ThreadingHTTPServer escucha en una dirección y puerto; la dirección 127.0.0.1 restringe el acceso a la máquina local y el puerto cero permite que el sistema operativo elija uno disponible. Thread ejecuta serve_forever en segundo plano mientras el cliente puede emitir solicitudes en el mismo kernel. daemon=True evita que el hilo mantenga indefinidamente el proceso al terminar, pero no reemplaza el cierre explícito. shutdown pide detener el bucle de servicio, server_close libera el socket y join espera de forma limitada al hilo. La función no crea una API pública ni un despliegue de producción. Si el entorno impide escuchar en loopback, el laboratorio debe registrar ese bloqueo; no simular la ejecución ni sustituir el resultado por un número fijo.

Dentro de la clase Handler, do_GET se ejecuta cuando llega una petición GET. urlsplit separa ruta y consulta; parse_qs interpreta los parámetros como un diccionario cuyas entradas pueden contener listas de valores. El código toma el primer valor de offset y limit, los convierte a int y exige que offset no sea negativo y limit esté entre uno y cien. Una solicitud con ruta distinta o parámetro faltante produce HTTP 400 en el servicio docente. En producción también convendría decidir cómo tratar parámetros duplicados, filtros desconocidos y tamaños extremos. La clase cierra sobre la lista base, es decir, usa la referencia que recibió la función externa; para evitar modificar los objetos originales crea dict(x) de cada registro de la ventana. Como los valores del corpus son cadenas, una copia superficial basta en este caso; si hubiera listas o diccionarios anidados, modificar un elemento interior podría afectar la referencia y necesitaríamos otra política de copia.

La respuesta HTTP se construye a partir del estado, cuerpo y encabezados. Content-Type indica que esperamos JSON codificado en UTF-8; Content-Length informa cantidad de bytes, no caracteres ni registros. La función serializar ordena claves y produce bytes; el servidor escribe esos bytes en wfile. En los escenarios de fallo de calidad se altera la copia de la ventana y se vuelve a serializar. En json_invalido se emite un cuerpo que no se interpreta como JSON. En prohibido se responde 403. En transitorio se usa un contador por offset para decidir si los dos primeros intentos de la segunda página fallan. Este contador es una instrumentación del servidor y no es la referencia de cobertura del cliente. Permite comparar cuántas veces se solicitó un offset con la traza del consumidor. En un sistema concurrente real un contador compartido requeriría considerar sincronización; las pruebas de esta sesión usan un cliente secuencial.

El cliente utiliza urlencode para representar offset y limit. urlopen abre una respuesta como contexto y la cierra al terminar; read recibe bytes. Solicitar 2.000.001 bytes permite detectar que la página excede el presupuesto de dos millones sin leer todo el contenido de una respuesta arbitrariamente grande. Si se supera el límite, el proceso se detiene. Este mecanismo no mide tráfico total ni implementa una descarga parcial reanudable. HTTPError contiene un código y encabezados que pueden utilizarse para decidir reintento; el código cierra el objeto de error. URLError y TimeoutError representan otras condiciones de transporte. Las categorías no son equivalentes: un problema de autorización se detiene de inmediato; los estados 429 y 503 se reintentan si aún hay presupuesto. Un timeout no acredita que el servidor no procesó la solicitud, de modo que un diseño de operaciones de escritura exigiría considerar idempotencia. Aquí se consulta un GET de una fuente congelada y se verifica identidad antes de escribir el producto.

La función pedir se define dentro de adquirir_http para compartir URL, límites y traza de esa ejecución. Al recorrer range(1,max_intentos+1) numeramos intentos desde uno. Si el estado es transitorio y aún quedan intentos, la función no retorna una página ni incrementa offset: vuelve al comienzo del bucle con los mismos parámetros. Cuando obtiene un cuerpo, retorna; el código exterior pasa entonces a preservar respuesta, parsear y validar. El tratamiento de Retry-After acepta únicamente segundos enteros pequeños en este laboratorio. Una fecha HTTP o espera superior al presupuesto no se ignora: se registra bloqueo. La función esperar se recibe como argumento para que una prueba automatizada pueda sustituir espera por una función sin demora. Esa inyección hace reproducible la prueba de lógica, pero no permite usar sus tiempos como rendimiento de una API externa. En producción se debe respetar la política de espera real.

La validación de página contiene varias fronteras. Primero, la respuesta debe ser interpretable como una lista del tamaño esperado. Después, cada registro debe ser un diccionario con los campos y tipos acordados. Solo si esa estructura existe se consultan los campos obligatorios, evitando intentar acceder a claves inexistentes. locales representa identidades observadas dentro de esa página; vistos representa páginas previamente aceptadas. Si una identidad aparece en cualquiera de ellos, se rechaza como repetición. esperado es un diccionario de identidad a bytes canónicos del snapshot. Su finalidad es comparar contenido de cada observación contra una referencia conocida. No sería posible construirlo exactamente de la misma manera si el consumidor no dispusiera del universo completo. El costo de este mapa crece con el corpus; el laboratorio no afirma que sea la estrategia óptima para una extracción de millones de documentos. Enseña cómo expresar una prueba fuerte cuando la referencia está disponible.

Si una fila falla, se agrega a cuarentena con offset, identidad, razón y registro. Luego se genera una excepción AdquisicionIncompleta. Esta clase permite distinguir una extracción que no se puede confirmar de otros problemas del lenguaje. Al lanzar la excepción dentro de la validación, se evita escribir el resto de una página cuya integridad no está establecida. Las respuestas exitosas anteriores permanecen en el directorio; la página problemática queda conservada si llegó un cuerpo interpretable o no. El cliente no recopila todas las posibles incidencias de una página porque se detiene en la primera; si se quisiera hacer una auditoría exhaustiva sin aceptar la página, se podría acumular razones por cada fila y luego decidir el estado de la ventana. Esa extensión debe mantener separados el inventario de incidencias y el estado del producto.

Después de escribir todas las páginas válidas, la función relee el archivo JSONL de forma incremental. enumerate conserva posición y permite contrastar cada registro con el objeto de origen en el mismo orden. La comparación de conteos verifica el derivado mediante otro recorrido. Se calcula huella de cada archivo leyendo bloques y se genera un manifiesto de finalización. Si falla una escritura o lectura, se guarda una incidencia cuando es posible y se vuelve a lanzar el error. No se garantiza que el archivo de incidencia exista ante un apagado o disco totalmente lleno. El consumidor debe tratar ausencia de manifiesto como producto no confirmado. Si hay manifiesto, también debe comprobar sus huellas y estado; la mera existencia de un nombre de archivo no es una prueba suficiente. El ejercicio A7.1 ampliará la verificación independiente del paquete.

El nuevo control compara las dos adquisiciones correctas y lee dos evidencias de rechazo. La traza de transitorio debe contener exactamente los intentos 429,503,200 para offset 50; las demás páginas deben solicitarse una vez. El hash de RAW coincide entre normal y transitorio, porque el producto lógico y su serialización se conservan. La huella de traza difiere, porque hubo otro recorrido operativo. En esquema se verifica que no se haya confirmado el directorio y que la cuarentena identifique la pérdida de campo. En corto se comprueba que el cuerpo recibido contenga cuarenta y nueve observaciones, mientras las filas aceptadas permanezcan en cincuenta de la primera página. No se calcula una tasa de error de SECOP: todos los fallos son inyectados en un servidor docente. La prueba busca demostrar que los controles reaccionan de forma diferente a estados válidos e inválidos.

**Reto de diagnóstico:** elige el escenario transitorio con presupuesto uno, dos y tres. Antes de ejecutar predice qué estado y qué secuencia de códigos aparecerán. Cambia limit a 100 y analiza las dos páginas esperadas. Una extracción normal debe requerir dos peticiones y confirmar 200 filas; la transitoria con tres intentos requiere cuatro peticiones porque la segunda página se pide tres veces. Con dos intentos, se conservan cien registros aceptados y la extracción queda incompleta. No añadas nuevas llamadas al portal real para este reto. En una variante de página corta, examina el archivo de respuesta y explica por qué no se permite continuar al siguiente offset. Tu defensa debe mencionar qué parte de la evidencia procede de HTTP y qué parte procede del contrato de identidad.


In [ ]:
normal_http=carpeta_http/"normal"
transitorio_http=carpeta_http/"transitorio"
assert sha_archivo(normal_http/"raw.jsonl")==sha_archivo(transitorio_http/"raw.jsonl")
assert sha_archivo(normal_http/"traza.json")!=sha_archivo(transitorio_http/"traza.json")
traza_trans=json.loads((transitorio_http/"traza.json").read_bytes())
assert [r["http"] for r in traza_trans if r["offset"]==50]==[429,503,200]
inc_esquema=json.loads((carpeta_http/"esquema"/"incidencia.json").read_bytes())
assert inc_esquema["estado"]=="INCOMPLETO" and inc_esquema["cuarentena"][0]["razon"]=="esquema"
assert not (carpeta_http/"esquema"/"manifest.json").exists()
inc_corto=json.loads((carpeta_http/"corto"/"incidencia.json").read_bytes())
assert inc_corto["filas_aceptadas"]==50
assert len(json.loads((carpeta_http/"corto"/"paginas"/"000050.json").read_bytes()))==49
print("QA HTTP: RAW idéntico, trazas distintas; esquema y página corta rechazados.")


### A5.1 · Diseñar pruebas de frontera para la adquisición paginada

La función adquirir recibe base, limite, salto y paginas. base es una lista de observaciones en el orden congelado; limite y salto pueden ser diferentes para construir ejemplos correctos e incorrectos; paginas fija cuántas iteraciones realiza la función. Esta función no utiliza HTTP: crea ventanas de la lista y enseña la aritmética. La aserción inicial evita límites, saltos o número de páginas no positivos. El conjunto esperado se crea a partir de id_proceso en base y se comprueba que su tamaño sea igual al número de filas. Si una referencia contiene duplicados, no es una lista de identidades únicas y el experimento de cobertura no tiene la interpretación prevista. En ese caso se debe revisar grano o referencia, no hacer que la prueba ignore las repeticiones.

El bucle for p in range(paginas) comienza en p=0. offset=p*salto calcula posición de inicio; base[offset:offset+limite] devuelve una lista con las observaciones entre esas posiciones. El final se excluye y Python recorta la ventana automáticamente si sobrepasa la longitud de la lista. Una posición fuera del universo puede producir lista vacía sin excepción. En una API real esta conducta depende de su contrato; aquí la regla está determinada por la semántica de Python. filas.extend(lote) agrega cada registro de la ventana a la lista de salida, a diferencia de append(lote), que agregaría la lista como un elemento anidado y cambiaría el grano. La traza registra página con numeración humana, offset y cantidad real. La cantidad real puede ser menor que el límite; no se reemplaza por el tamaño solicitado.

Counter construye un mapa de apariciones de cada id_proceso. len(c) cuenta claves distintas; sum(v-1 for v in c.values()) calcula repeticiones adicionales, porque toda clave presente tiene al menos una aparición. Si un ID aparece tres veces aporta dos repeticiones adicionales. Las expresiones set(c) y set(esperado) representan conjuntos de identidades. La resta esperado-set(c) identifica ausentes; set(c)-esperado identifica extras. sorted convierte el conjunto de incidencias a una lista ordenada para que la salida sea estable y fácil de inspeccionar. La prueba normal exige que ambas listas estén vacías. Una cantidad igual de IDs únicos a N no basta si aparecen extras; podrían reemplazar identidades esperadas. Las pruebas negativas que construiremos verifican ambas posibilidades.

Un error típico consiste en utilizar siempre una cantidad fija de páginas que funcionaba para una configuración previa. Con N=200 y L=50 se necesitan cuatro páginas, pero al cambiar L a 37 se necesitan seis. Si se conservan cuatro, se obtienen 148 filas correctas y faltan 52. El código de ruta válida utiliza math.ceil(len(base)/LIMIT), que redondea hacia arriba. Para evitar operaciones de punto flotante innecesarias en tamaños enormes, también puede utilizarse (len(base)+LIMIT-1)//LIMIT. El número de peticiones no debe confundirse con número de filas, y el número de páginas calculado presupone un universo conocido. Si se cambia base a un prefijo, el cálculo debe usar su longitud, no el tamaño original del snapshot. Este es un ejemplo de parametrización correcta: las constantes de la prueba describen el ejemplo, no el algoritmo.

Las pruebas de frontera examinan valores que suelen descubrir errores de bucle. Un universo de un registro con limit 1 debe producir una sola página; limit mayor que el universo debe producir una única página parcial; limit igual al tamaño exacto debe producir una completa; limit justo por debajo de N produce una página grande y una pequeña. Una selección de 103 con limit 50 produce tres ventanas y una última de tres. La combinación de estos casos permite detectar errores de inclusión del último elemento, terminación antes de leer la última ventana o cálculo de offsets que omite el cero. En la siguiente celda se usan cuatro prefijos del corpus y seis tamaños de página; las propiedades esperadas se calculan independientemente mediante aritmética entera y comparación de lista de origen, no a partir de los resultados de adquirir. Se conserva la referencia de objetos reales del snapshot.

Para una prueba de orden, no basta comparar IDs como conjunto. Si la salida contiene todos los objetos pero en orden inverso, el conteo por entidad sigue siendo correcto, aunque el manifiesto que promete orden de snapshot no se cumpla. Por ello las pruebas positivas comparan lista de salida con lista de entrada. Python compara diccionarios por contenido, de modo que el orden interno de claves no afecta esa igualdad. La huella canónica verifica adicionalmente los bytes acordados. Si se construye una nueva versión del producto donde el orden no importa, se puede cambiar la especificación, pero no se elimina arbitrariamente el control de orden de la versión actual. La prueba de cobertura debe reflejar el contrato, no una preferencia de implementación.

Una prueba negativa sobre objetos recibidos permite simular un caso que no surge de una simple rebanada de la misma lista: sustituir una identidad por otra inesperada. Se toma una copia de la lista y se reemplaza un objeto por otro que conserva campos pero tiene ID ficticio distinto. El resultado tiene la misma longitud y cantidad de únicos. La igualdad de conjuntos falla con un ausente y un extra. Esta prueba muestra por qué una reconciliación necesita la intersección con el universo esperado, no solo cardinalidad. En otra variante se conserva el ID y se cambia el nombre de entidad; los controles de conjunto no detectan la diferencia, pero la comparación de contenido por ID sí. Estas modificaciones son exclusivamente pruebas; se generan sobre copias y no entran al paquete de datos real.

La matriz de pruebas no debe depender del nombre de la herramienta. El mismo razonamiento se puede trasladar a una consulta por lotes, a una lectura particionada o a un pipeline que recibe archivos. Cada unidad debe contar con una identidad y el consumidor debe saber qué universo está validando. En una fuente viva, puede que no haya una lista completa de IDs; entonces se registran otros controles, pero no se afirma que se probó una propiedad que no es observable. El laboratorio proporciona una referencia completa porque pretende enseñar exactitud de manera reproducible. La fuente pública de origen y la API docente son elementos distintos de esa evidencia. En una evaluación posterior se puede probar la adquisición viva con un recurso y contrato específicos.

**Actividad autónoma:** elige una combinación que no esté en la matriz, por ejemplo N=137 y limit=29. Calcula páginas y tamaño final sin ejecutar. Luego cambia salto a 30, conserva cantidad de páginas calculada y predice ausencias. Registra tanto resultado incorrecto como correcto y explica qué prueba falla. Si aparece un lote vacío, comprueba si se debe a una petición adicional fuera del universo o a una pérdida de respuesta; en esta función de lista solo puede ser la primera causa. Finalmente compara los conteos por entidad de los resultados válidos con un recorrido directo del prefijo. Tu evidencia debe incluir parámetros, universo y controles; no basta imprimir OK.


In [ ]:
def reconciliar_s01(esperados,recibidos):
    e={r["id_proceso"]:canon(r) for r in esperados}
    assert len(e)==len(esperados)
    c=Counter(r["id_proceso"] for r in recibidos)
    return {"filas":len(recibidos),"unicos":len(c),"repetidos":len(recibidos)-len(c),
            "ausentes":sorted(set(e)-set(c)),"extras":sorted(set(c)-set(e)),
            "contenido_distinto":sorted({r["id_proceso"] for r in recibidos if r["id_proceso"] in e and canon(r)!=e[r["id_proceso"]]})}
matriz_paginas=[]
for n0 in (1,40,103,200):
    universo=CORPUS[:n0]
    for l0 in (1,7,20,37,50,200):
        p0=(n0+l0-1)//l0
        salida,ctrl,tr=adquirir(universo,l0,l0,p0)
        assert salida==universo and len(tr)==p0 and tr[-1]["filas"]==n0-(p0-1)*l0
        assert not ctrl["ausentes"] and not ctrl["extras"] and not ctrl["duplicados"]
        matriz_paginas.append(dict(N=n0,limit=l0,paginas=p0,ultima=tr[-1]["filas"],unicos=ctrl["unicos"]))
display(pd.DataFrame(matriz_paginas))
copia_extra=list(CORPUS)
copia_extra[-1]={**copia_extra[-1],"id_proceso":"ID_FICTICIO_PRUEBA"}
ctrl_extra=reconciliar_s01(CORPUS,copia_extra)
assert ctrl_extra["filas"]==ctrl_extra["unicos"]==200 and len(ctrl_extra["ausentes"])==len(ctrl_extra["extras"])==1
copia_cambio=list(CORPUS)
copia_cambio[0]={**copia_cambio[0],"entidad":copia_cambio[0]["entidad"]+" [prueba]"}
ctrl_cambio=reconciliar_s01(CORPUS,copia_cambio)
assert not ctrl_cambio["ausentes"] and not ctrl_cambio["extras"] and len(ctrl_cambio["contenido_distinto"])==1
assert sha(canon(CORPUS))==HASH_SNAPSHOT
print("24 casos de frontera y dos pruebas negativas comprobadas.")



### A6.1 · Separar la estructura de datos del algoritmo y medir el costo que realmente interesa

contar_bloques recibe una lista de registros y un tamaño de bloque. La variable out es un Counter vacío y tamanos una lista que conserva longitud de cada bloque. range(0,len(filas),chunk) genera posiciones de inicio, no datos. Cada rebanada lote contiene referencias a los objetos de la lista; no se crean copias profundas de registros. update recibe un generador de nombres de entidad y aumenta el contador por cada aparición. Un generador produce valores de manera perezosa, pero la lista base sigue cargada; por eso esta función enseña estados combinables y no acredita que el conjunto completo haya dejado de estar en RAM. La salida out permite responder conteos por entidad; tamanos permite comprobar que se procesó la cantidad esperada. Un fallo que omita un bloque podría detectarse mediante suma de tamaños y comparación con el contador directo.

La segunda celda mide lectura de un CSV generado. TemporaryDirectory crea una carpeta y la elimina al terminar el contexto. Path(tmp)/"sintetico.csv" construye la ruta con el separador adecuado. open con newline="" permite que el módulo csv gestione saltos de línea correctamente. writerow escribe nombres de columnas; writerows consume un generador de tuplas, evitando crear previamente una lista de N tuplas. El generador produce (i,i%101) para cada i desde cero hasta N−1. La operación módulo da un patrón repetible de valores entre cero y cien. No se introduce ruido ni muestreo: cada configuración lee exactamente el mismo archivo. La escritura se realiza antes de iniciar el cronómetro de lectura; el experimento mide lectura y agregación, no el tiempo de generar datos. Si el objetivo fuera medir ETL completo, habría que incluir ambas etapas para todas las estrategias.

La ruta completa y las rutas fragmentadas usan ahora el mismo bucle de medición. perf_counter se inicia antes de crear el lector y se detiene después de recorrer todos sus bloques. La lectura completa se representa como una lista con un único DataFrame; la ruta fragmentada utiliza el iterador que devuelve read_csv. En cada caso se actualizan cantidad, suma, número de bloques y máximo de memoria lógica. Por ello ambos tiempos incluyen lectura, agregación e instrumentación de memoria. La memoria lógica no equivale al pico de memoria residente y la medición no incluye la generación del CSV. El borrado de referencias locales al terminar cada ruta permite liberar objetos cuando no existen otras referencias, pero no garantiza una reducción inmediata del RSS. El orden de las rutas puede afectar caché y otras condiciones; se conserva como una demostración breve, no como prueba de superioridad universal. El benchmark repetido del apéndice rota el orden para reducir una fuente de sesgo.

En el bucle, cnt cuenta filas, total suma valores, partes registra cuántos bloques se recorrieron y mem conserva el máximo lógico observado, no la suma de memoria de todos los bloques. Esta diferencia importa: si se leen diez bloques de mil filas secuencialmente, sumar su memoria describe bytes procesados, no necesariamente bytes activos al mismo tiempo. El lector puede mantener buffers adicionales y el intérprete conserva otras estructuras. La prueba exige que el conteo y la suma coincidan con una fórmula independiente de pandas y que la cantidad de bloques coincida con la aritmética esperada. No basta que dos caminos arrojen el mismo resultado si ambos comparten el mismo error. La comparación mantiene fijo el CSV y cambia únicamente el tamaño de lectura. Una medición de memoria residente requeriría otro método y controlar las demás variables del proceso.

Una referencia matemática permite comprobar la suma sin depender exclusivamente de pandas. Cada ciclo completo de 101 valores desde cero hasta cien tiene suma 5050. Para N, divmod(N,101) devuelve número de ciclos completos y residuo. La suma esperada es ciclos por 5050 más la suma de enteros desde cero hasta residuo−1, calculable como residuo*(residuo−1)//2. Esta fórmula es independiente de la lectura del CSV. Si dos rutas pandas coincidieran por un error común de generación o filtrado, la referencia matemática podría detectar algunas inconsistencias adicionales. Por ejemplo, si falta una fila de valor cero, la suma puede permanecer igual; la comprobación de cantidad detecta esa omisión. Si se reemplazan dos valores de forma que la suma no cambia, ninguno de esos dos controles garantiza identidad completa; para ese caso se necesitarían otras propiedades, como IDs y contenido. Toda prueba tiene un alcance.

El nuevo benchmark repite tres veces cuatro estrategias sobre el mismo archivo de 20.000 filas. Se rota el orden de las estrategias en cada ronda para no asignar siempre la primera lectura a la misma configuración. No se vacía la caché ni se controla actividad de otras aplicaciones, de modo que sigue siendo una medición didáctica, no un benchmark de hardware. Las rutas miden lectura, agregación y estimación de memoria lógica con la misma convención de duración. Para mantener comparabilidad, la ruta completa procesa un DataFrame como un único bloque y la fragmentada procesa el lector; en ambas el bucle actualiza cantidad, suma y máximo lógico. Se incluye tiempo de instrumentación porque las llamadas a memory_usage forman parte de la operación medida en ambas. La mediana de tres observaciones reduce la influencia de una corrida extrema sin pretender significancia estadística. El resultado conserva todas las muestras individuales para que se pueda revisar dispersión.

La tabla de resumen se agrupa por estrategia y registra segundos de mediana, mínimo, máximo, cantidad de ejecuciones y memoria lógica máxima. Agrupar tiempos de estrategias distintas no produciría una medida interpretable. La variable estrategia es una cadena, de modo que se puede etiquetar "completo", "500", "1000" y "5000" sin mezclar tipos. La medida bytes se mantiene en enteros. Para visualizar, se podrían convertir a KiB dividiendo por 1024, conservando la unidad en el encabezado. El cociente de memoria lógica entre ruta completa y bloque muestra diferencia de representación activa, no reducción de RAM total. Un cociente de duración se calcula solo dentro de esta prueba y no se extrapola a otras operaciones. Si la estrategia de bloque pequeño tarda más, la observación es compatible con mayor número de iteraciones y costos de instrumentación; no se atribuye automáticamente una causa sin medir sus componentes.

Para una aplicación de Big Data, una lectura por bloques puede ser necesaria aunque sea más lenta que una lectura completa que cabe en memoria. La restricción puede ser factibilidad y no velocidad. Elegir una estrategia exige distinguir objetivos: reducir memoria, reducir tiempo, aumentar tasa, facilitar recuperación o limitar costo. Un algoritmo que no cabe no es una opción válida para ese presupuesto aunque su tiempo hipotético sea bajo. Un algoritmo por bloques puede conservar memoria activa pero mantener estados de cardinalidad elevada; por eso se analiza también K. Cuando la operación requiere ordenar globalmente o combinar datos por claves, la estrategia puede necesitar almacenamiento temporal o intercambio de particiones. La práctica de esta sesión prueba una reducción simple y no pretende resolver todos los casos.

**Reto experimental:** cambia N a 30.000 y conserva estrategias. Antes de ejecutar calcula referencia matemática de suma y cantidad de bloques. Registra si la memoria lógica por bloque permanece aproximadamente estable para el mismo tamaño y si la memoria completa aumenta. Cambia después solo chunksize a 2.000 y 10.000 manteniendo N. Compara mediana y dispersión, sin asumir monotonicidad de tiempo. Explica por qué un archivo con textos largos podría producir tamaños lógicos distintos para bloques del mismo número de filas. No aumentes N deliberadamente hasta agotar memoria ni uses el resultado de esta prueba como rendimiento de Dask o Spark. La evidencia de aprendizaje es diseñar una comparación controlada, verificar equivalencia y explicar qué costo se midió.


In [ ]:
import statistics
def suma_teorica_s01(n):
    ciclos,resto=divmod(n,101)
    return ciclos*5050+resto*(resto-1)//2
assert suma_teorica_s01(100_000)==4_999_545
def benchmark_repetido_s01(n=20_000,bloques=(500,1000,5000),rondas=3):
    assert 1<=n<=200_000 and 1<=rondas<=5 and all(0<b<=n for b in bloques)
    estrategias=[None,*bloques]; mediciones=[]
    with tempfile.TemporaryDirectory(prefix="s01_bench_repetido_") as tmp:
        p=Path(tmp)/"entrada.csv"
        with p.open("w",newline="",encoding="utf-8") as f:
            w=csv.writer(f);w.writerow(["id","valor"]);w.writerows((i,i%101) for i in range(n))
        for j in range(rondas):
            k=j%len(estrategias)
            for b in estrategias[k:]+estrategias[:k]:
                inicio=time.perf_counter()
                lector=[pd.read_csv(p)] if b is None else pd.read_csv(p,chunksize=b)
                filas=total=mem=partes=0
                for tabla in lector:
                    filas+=len(tabla);total+=int(tabla.valor.sum());partes+=1
                    mem=max(mem,int(tabla.memory_usage(deep=True).sum()))
                segundos=time.perf_counter()-inicio
                assert (filas,total)==(n,suma_teorica_s01(n))
                assert partes==(1 if b is None else (n+b-1)//b)
                mediciones.append({"ronda":j+1,"estrategia":"completo" if b is None else str(b),
                                   "filas":filas,"suma":total,"partes":partes,"bytes_logicos":mem,
                                   "segundos_incluye_instrumentacion":segundos})
                del tabla,lector
    return pd.DataFrame(mediciones)
rep=benchmark_repetido_s01()
resumen_rep=rep.groupby("estrategia").agg(
    mediana=("segundos_incluye_instrumentacion","median"),
    minimo=("segundos_incluye_instrumentacion","min"),
    maximo=("segundos_incluye_instrumentacion","max"),
    ejecuciones=("ronda","size"),bytes_logicos=("bytes_logicos","max"))
assert len(rep)==12 and (resumen_rep.ejecuciones==3).all()
display(rep);display(resumen_rep)
print("Medidas de lectura + agregación + instrumentación; no pico RSS.")


### A6.3 · Representación y complejidad (ampliación)
Una misma información puede ocupar cantidades diferentes de memoria según su tipo. La comparación de int64 e int32 solo es válida si los valores caben en ambos y los resultados son equivalentes. Mediremos memoria lógica de dos columnas enteras, manteniendo fijas las filas. Después contrastaremos dos mecanismos para contar claves: recorrer la colección una vez por cada clave, o mantener un contador en una sola pasada. Ambos resultados deben coincidir. Los conteos de operaciones son teóricos bajo el modelo indicado; el tiempo es observado. No ejecutamos un algoritmo cuadrático sobre millones de registros para provocar un fallo.


In [ ]:
import statistics
tabla64=pd.DataFrame({"id":range(100_000),"valor":[i%101 for i in range(100_000)]},dtype="int64")
tabla32=tabla64.astype("int32")
assert tabla64.equals(tabla32.astype("int64"))
assert int(tabla64.valor.sum())==int(tabla32.valor.sum())
display(pd.DataFrame([{"tipo":str(t.valor.dtype),"filas":len(t),"bytes_logicos":int(t.memory_usage(deep=True).sum()),"suma":int(t.valor.sum())} for t in [tabla64,tabla32]]))
del tabla64,tabla32
claves=[i%50 for i in range(10_000)]
universo=sorted(set(claves))
def contar_repetido(): return {k:claves.count(k) for k in universo}
def contar_una_pasada(): return dict(Counter(claves))
resultados_complejidad=[]
for nombre,func,operaciones in [("una pasada por clave",contar_repetido,len(claves)*len(universo)),("contador",contar_una_pasada,len(claves))]:
    tiempos=[]
    for _ in range(3):
        t0=time.perf_counter(); resultado=func(); tiempos.append(time.perf_counter()-t0)
        assert resultado==dict(Counter(claves))
    resultados_complejidad.append({"estrategia":nombre,"comparaciones_o_visitas_modelo":operaciones,"mediana_s":statistics.median(tiempos)})
display(pd.DataFrame(resultados_complejidad))
print("La comparación cuenta visitas/comparaciones del modelo; no todas cuestan igual en CPU.")


### A7.1 · Verificar el producto sin depender del código que lo generó

La función producir combina etapas que el estudiante ya conoce. adquirir construye las ventanas según límite y salto igual a límite. La aserción comprueba que las identidades coincidan con base, que no haya repeticiones, ausencias ni extras. contar_bloques calcula un agregado y verifica igualdad contra un Counter directo. tempfile.mkdtemp crea una carpeta que permanece al terminar la función, a diferencia de TemporaryDirectory que la elimina al salir del contexto. El paquete incluye snapshot.json con el universo completo, raw.jsonl con el universo seleccionado, conteos.json con el derivado y manifest.json con parámetros y controles. write_bytes escribe la secuencia producida por canon. Para JSONL se agrega un salto de línea después de cada objeto, permitiendo que un lector recorra observaciones sin interpretar una lista JSON gigante. Esta implementación de producir usa join de bytes para construir el archivo completo en memoria; el laboratorio HTTP demuestra la alternativa de escribir fila a fila. No se atribuye streaming de extremo a extremo a esta función cuando no lo implementa.

El manifiesto es un diccionario de metadatos y referencias. La propiedad archivos contiene nombre de archivo y SHA-256. Se calcula antes de escribir el propio manifiesto, evitando una dependencia circular donde el manifiesto tendría que incluir su propia huella. El producto guarda el hash del snapshot canónico para que se pueda verificar su contenido de manera independiente. La referencia de blob identifica la fuente de repositorio. El universo puede ser completo o prefijo y la longitud original se conserva separada. La traza describe ventanas y bloques. El control de cobertura registra ausencias y extras como listas, no solo un booleano. La función lee de nuevo el manifiesto y los archivos y comprueba las huellas. Después lee RAW, compara sus observaciones con filas, verifica unicidad y comprueba el agregado. Esta es una prueba de la función en el mismo proceso; para acreditar un paquete portátil conviene crear un verificador que reciba únicamente la carpeta, lea sus archivos y contraste las propiedades sin variables de memoria del productor.

El nuevo verificador no requiere que exista la variable man ni la lista raw de la ejecución. Recibe carpeta y, opcionalmente, una huella del snapshot confiable. Valida la estructura mínima del manifiesto y exige exactamente los tres archivos de datos esperados bajo el contrato v1; de ese modo no abre rutas arbitrarias indicadas por un manifiesto modificado. Calcula huellas desde bytes y comprueba coincidencia. Interpreta el snapshot y exige un universo no vacío de IDs únicos con el esquema acordado. Comprueba que hash_snapshot coincida con su archivo; si se proporciona una referencia externa, también compara con ella. Una referencia externa confiable es importante porque un atacante o una modificación accidental podrían cambiar simultáneamente el archivo y su hash en el manifiesto. No se promete autenticidad criptográfica: la prueba solo compara con las referencias suministradas y verifica consistencia interna.

Luego interpreta raw.jsonl línea por línea. Para el contrato v1, la selección se obtiene como prefijo del snapshot según la cantidad observada y la etiqueta universo. Se exige que los registros coincidan por posición y contenido con esa referencia, que todos tengan el esquema y que las identidades sean únicas. La etiqueta debe ser "snapshot completo" cuando corresponde o "prefijo N de M"; si una configuración selecciona todos los registros pero usa la etiqueta prefijo M de M, ambos describen el mismo universo y se admiten como declaraciones válidas bajo esta versión. La verificación compara filas y únicos de control, ausencia de duplicados, ausentes y extras. También reconstruye el número y tamaños esperados de páginas y bloques mediante aritmética entera. No utiliza adquirir ni contar_bloques, evitando compartir sus posibles errores de fragmentación. El derivado se obtiene con groupby de pandas, independiente de Counter usado en producir. El verificador sigue dependiendo de las mismas reglas de grano y serialización; esa dependencia es deliberada porque son parte del contrato.

La función utiliza un error explícito de validación, ValueError, en lugar de aserciones para las condiciones del archivo recibido. Esto permite comunicar qué propiedad no pasó aun si el intérprete se ejecuta con optimizaciones que deshabilitan assert. Las aserciones se reservan para las pruebas de aula que verifican comportamiento de las funciones. Una condición como exigir(len(filas)==control["filas"],"cantidad") produce un mensaje de dominio más claro que un fallo genérico. El verificador no repara los archivos y no sobrescribe el paquete. Su salida resume filas, entidades, páginas, bloques y estado de verificación. Si los datos no cumplen el contrato, no devuelve un éxito parcial. En una aplicación real podría producir un informe de todas las incidencias en vez de detenerse en la primera, pero debería mantener claro el estado del producto. Esta práctica enseña consumo responsable de una entrega de datos.

Las pruebas negativas del apéndice usan shutil.copytree para copiar el directorio a una carpeta exclusiva, de modo que el paquete original permanece intacto. En la primera copia se agrega un espacio al archivo conteos.json; JSON sigue siendo interpretable, pero el hash cambia y el verificador debe rechazarlo. En la segunda se elimina raw.jsonl; la lista de archivos exigidos impide continuar. En la tercera se invierte el orden de las líneas de RAW y se actualiza su hash en el manifiesto de prueba. Un control limitado a hashes internos podría aceptar esta modificación. El verificador independiente debe rechazarla porque el contrato conserva orden de snapshot. En la cuarta copia se cambia un conteo y también su hash interno; el recálculo del agregado debe detectar inconsistencia. Estas pruebas ilustran que la validación no termina al comprobar integridad de bytes. No se modifican los paquetes correctos ni se presentan variantes alteradas como productos autorizados.

El caso de una modificación simultánea de datos y metadatos exige distinguir consistencia y confianza. Si alguien sustituye todo el snapshot, RAW, conteos y manifiesto de forma coherente, una validación interna podría aceptar el nuevo producto como consistente. La comparación con una huella esperada de la fuente independiente detectaría esa sustitución si no coincide. Una huella copiada del mismo manifiesto no añade una referencia externa. En S01 la versión de origen está incorporada al cuaderno y puede compararse con HASH_SNAPSHOT. Para una entrega que circule fuera del entorno, se puede conservar esa referencia en un registro confiable del proyecto. Este mecanismo no constituye una firma digital ni prueba de autoría. El estudiante debe explicar qué amenaza o error permite detectar cada control y qué propiedad queda fuera. La reproducibilidad es más fuerte cuando la fuente y la verificación se conservan de forma independiente.

**Reto:** toma el paquete generado por tu variante de A8 y ejecuta el verificador utilizando solo la ruta y la huella esperada. Si cambias limit, comprueba que los offsets se reconstruyan de acuerdo con el nuevo parámetro; si cambias chunk, verifica tamaños y suma. Modifica una copia del manifiesto para que declare un universo incorrecto aunque los archivos sean idénticos. Predice qué condición debe fallar. Crea otra copia donde falta un campo obligatorio en una línea de RAW y actualiza su huella de prueba: el verificador debe detectar esquema o diferencia de contenido. Explica por qué las pruebas de corrupción intencional no sustituyen una revisión de procedencia, permisos y autenticidad. Para la defensa, muestra una ruta correcta y una negativa, el mensaje de rechazo y la propiedad que se protegió.

Se añaden dos pruebas de metadatos: una declara un esquema distinto y otra sustituye la referencia de blob. Ambas deben rechazarse aunque los archivos de datos no cambien. Así se verifica que el consumidor compara con el contrato esperado, no solo con lo que declara el manifiesto. La función de verificación ya se ejecutó en el núcleo; la celda siguiente recorre los seis casos de rechazo sobre copias.


In [ ]:
with tempfile.TemporaryDirectory(prefix="s01_pruebas_paquete_") as tmp:
    for fallo in ("bytes","ausente","orden","conteo","esquema","procedencia"):
        p=Path(tmp)/fallo
        shutil.copytree(destino,p)
        if fallo=="bytes":
            f=p/"conteos.json";f.write_bytes(f.read_bytes()+b" ")
        elif fallo=="ausente": (p/"raw.jsonl").unlink()
        elif fallo in ("esquema","procedencia"):
            m=json.loads((p/"manifest.json").read_bytes())
            if fallo=="esquema": m["esquema"]=m["esquema"][:-1]
            else: m["blob_fuente"]="referencia alterada"
            (p/"manifest.json").write_bytes(canon(m))
        else:
            m=json.loads((p/"manifest.json").read_bytes())
            if fallo=="orden":
                f=p/"raw.jsonl";f.write_bytes(b"\n".join(reversed(f.read_bytes().splitlines()))+b"\n")
            else:
                f=p/"conteos.json";d=json.loads(f.read_bytes());d[next(iter(d))]+=1;f.write_bytes(canon(d))
            m["archivos"][f.name]=sha_archivo(f)
            (p/"manifest.json").write_bytes(canon(m))
        try: verificar_paquete_s01(p,HASH_SNAPSHOT)
        except ValueError as exc: print("Rechazo esperado",fallo,str(exc))
        else: raise AssertionError("Aceptó paquete alterado: "+fallo)
assert verificar_paquete_s01(destino,HASH_SNAPSHOT)["estado"]=="VERIFICADO"




In [ ]:
# Cambia estos valores para practicar, antes de ejecutar predice qué se conserva.
N_RETO=80
LIMIT_RETO=19
CHUNK_RETO=13
assert 1<=N_RETO<=len(base)
d2,m2=producir(base,LIMIT_RETO,CHUNK_RETO)
assert m2["archivos"]["raw.jsonl"]==man["archivos"]["raw.jsonl"]
assert m2["archivos"]["conteos.json"]==man["archivos"]["conteos.json"]
d3,m3=producir(base[:N_RETO],LIMIT_RETO,CHUNK_RETO,universo=f"prefijo {N_RETO} de {len(base)}")
assert m3["control"]["unicos"]==N_RETO
print("Mismo universo, mismos bytes RAW:",d2)
print("Universo distinto, cobertura declarada:",d3,m3["universo"])
print("RELEVO",{"fuente":FUENTE,"grano":"proceso","filas":len(base),"esquema":ESQUEMA,
                "frecuencia":"no medida","SLA":"no definido"})


### A8.1 · Un reto no resuelto de antemano: formular hipótesis y generar evidencia propia

La celda de transferencia guiada cambia parámetros de página y bloque manteniendo universo completo, después produce un prefijo de tamaño elegido. Se comprueban huellas de RAW y conteos entre las dos rutas del mismo universo. La igualdad esperada se deriva de que ambas utilizan el mismo orden y contenido, no de que sus manifiestos tengan los mismos parámetros. El tercer paquete cambia población y se valida contra su propio prefijo. Este recorrido guiado es útil para comprender la mecánica, pero por sí solo no acredita autonomía: un estudiante podría ejecutar los valores ya escritos sin comprender la relación entre ellos. Para comprobar transferencia se introduce una actividad separada con parámetros elegidos por el estudiante, predicciones que se registran antes de ejecutar y un evaluador de propiedades. El objetivo no es una nueva nota oficial ni resolver una evaluación sumativa; es proporcionar retroalimentación verificable.

Elige N entre 41 y 199, un tamaño de página de 7 a 45 y un tamaño de bloque de 6 a 40. Evita utilizar exactamente los parámetros del ejemplo previo. La función de preparación valida que los valores sean enteros dentro de los intervalos y que el bloque no supere el universo. La cantidad esperada de páginas depende de N y limit, la cantidad de bloques de N y chunk. La última ventana puede ser completa si N es múltiplo de limit; no se utiliza residuo cero como tamaño de una página inexistente. Para predecirla, calcula páginas y resta (páginas−1)*limit a N. La misma fórmula vale para bloques. Estas predicciones deben realizarse antes de ejecutar producir. El código no ofrece automáticamente la solución cuando la hipótesis todavía no se ha diligenciado. La celda de demostración puede ejecutarse de manera limpia con un modo por defecto que no evalúa la actividad; la persona activa el reto cuando haya escrito sus predicciones. Esto evita incluir una respuesta precalculada que se pueda copiar como evidencia propia.

El evaluador recibe una ruta de paquete y un diccionario de hipótesis. Abre el manifiesto y llama al verificador independiente, que comprueba archivos y derivado. Luego compara las predicciones numéricas con las propiedades observadas. Una hipótesis incorrecta no convierte un paquete válido en inválido; se distinguen errores de comprensión de errores de ejecución. Si el paquete no supera el verificador, no se pueden interpretar las predicciones como contrastadas con un resultado fiable. Si el paquete es válido pero el número de páginas predicho es incorrecto, se informa qué relación revisar. No se modifica la hipótesis ni se otorga un acierto por reintento automático. El estudiante puede volver a probar con una hipótesis corregida, conservando el primer intento en su bitácora. La prueba no envía datos al profesor ni registra identidad personal.

El reto añade una condición de transferencia operativa. Se genera una segunda adquisición del mismo prefijo con un límite distinto calculado a partir del elegido y un tamaño de bloque distinto. El verificador exige que ambos paquetes sean consistentes. Se compara hash de RAW y conteos: deben ser iguales. Los parámetros y trazas de manifiesto pueden diferir. Si el nuevo límite es diferente pero la cantidad de páginas coincide, eso no invalida la comparación: distintos tamaños pueden producir la misma cantidad de ventanas por efecto del redondeo. Se compara la propiedad que realmente debe conservarse, no se exige que todas las métricas cambien. La defensa requiere explicar esta diferencia. A8.1 es una referencia de invariancia, no el experimento aislado final. A8.3 exige cuatro configuraciones: base, solo límite, solo bloque y solo universo. El cuarto cambia N en una unidad (disminuye en el límite superior); el agregado total cambia en una unidad.

El resultado de retroalimentación se presenta por propiedad: páginas, última página, bloques, último bloque, cobertura e invariancia. La cobertura se comprueba sobre N, no se expresa como cobertura de todo SECOP. Las propiedades numéricas se calculan a partir de las trazas verificadas. La comparación de hashes es una prueba de igualdad de representación bajo las reglas del producto. El estudiante debe explicar por qué una coincidencia de hashes no demuestra que el proveedor haya publicado información verdadera. La actividad incluye una pregunta abierta que se responde en la bitácora: ¿qué dato adicional sería necesario para sumar valor contractual por entidad? El evaluador no intenta calificar automáticamente esta justificación, porque una respuesta de texto requiere revisar significado. Una evaluación pedagógica rigurosa debe distinguir validaciones automáticas de valoración de razonamiento humano.

Para completar la transferencia, redacta una ficha con cinco enunciados. Primero, identifica el snapshot y el tamaño de la selección. Segundo, declara grano y campos que se conservan. Tercero, explica qué se modificó entre ejecuciones y qué propiedad se esperaba mantener. Cuarto, muestra la evidencia de verificación, incluyendo una prueba negativa que detuvo un producto alterado. Quinto, declara un límite de interpretación y una pregunta que requiere otra fuente o medida. La ficha no necesita datos personales. Puede guardarse como texto junto a los resultados de práctica sin modificar el manifiesto de los archivos ya verificados; si se incorpora formalmente a un producto, debe generarse una nueva versión con su propio control. Esta actividad permitirá a otra persona revisar si el estudiante sabe distinguir transporte, adquisición, transformación, validación y decisión.

**Criterio de retroalimentación:** la referencia A8.1 es satisfactoria cuando ambos paquetes pasan el verificador y conservan RAW y conteos. La entrega autónoma exige además ambas construcciones y los cuatro casos aislados de A8.3; las predicciones se evalúan por separado. Si alguna predicción falla, el estudiante debe explicar la diferencia y repetir con otra combinación, no solamente copiar las cifras impresas. La comprensión conceptual se revisa mediante su justificación de grano, interpretación y límite. La duración de este reto depende de la experiencia y entorno del estudiante; aún requiere ensayo docente. No se confunde la ejecución automatizada del cuaderno con una prueba de autonomía observada. La puerta de calidad de S01 permanece separada del reto formativo: solo podrá desbloquear S02 cuando se cumplan los criterios del curso reconstruido.
